# Réplica de Sidelnik et al. (2020) con MEIGA: neutrones de 500 MeV en el WCD (Sección 4.1.1)

Corridas de MEIGA (Geant4 10.7.p04, `QGSP_BERT_HP_NoThermal`, sin $S(\alpha,\beta)$, modo óptico `eFast`) que reproducen la configuración de
Sidelnik et al., *Adv. Space Res.* **65** (2020) 2216: 150 000 neutrones de 500 MeV que entran horizontalmente a media altura del tanque
(radio 48 cm, altura 133 cm, Tyvek 0,12 mm + acero 0,5 mm, PMT de 10,16 × 10,16 × 5 cm en la tapa).

| Medio | Corrida (disco ADATA, `Corridas-Sidelnik-500MeV/`) | Neutrones |
|---|---|---|
| Agua pura | `150000N-500MeV-Sidelnik-H2O-pura-sinSab-20261012-061022` | 150 000 |
| Agua + 2,5 % NaCl | `42649N-parcial-...-20261013` (interrumpida) + `107351N-...-20261013-135213` (complemento), unidas con `unir_tramos.py` | 150 000 |

**Orden de ejecución.** Ejecute las celdas en orden.
1. *Procesamiento* (sección 1): lee los archivos crudos del disco; se salta para cada medio si su resultado ya está en `resultados/`
   (cambie `REPROCESAR = True` para rehacerlo).
2. *Referencias* (sección 2): digitalización por color de las Figs. 3 y 5 de Sidelnik (las imágenes están en `datos/`).
3. *Análisis y figuras* (secciones 3–6): trabajan con `resultados/` y tardan segundos. Cada celda contiene el código del script correspondiente.
4. *Informe* (sección 7): `informe/Informe-Sidelnik-500MeV.pdf`, con el mismo contenido que la Sec. 4.1.1 de la tesis en formato de informe (fuente común: `informe/seccion_4_1_1.tex` y `informe/apendice_procesos.tex`; `scripts/construir_informe.py` ajusta la jerarquía de títulos), y las notas técnicas `informe/Informe-Sidelnik-500MeV-tecnico.pdf` (corridas, procedimientos, diagnósticos y reproducibilidad).

Los archivos `resultados/<medio>_fotones.npz` (destino de cada fotón óptico, 345 MB y 113 MB) no están en el repositorio:
se regeneran con `destino_fotones.py` desde `pasos-particulas.tsv`; sin ellos se saltan solo los diagnósticos ópticos (Sec. 5).

Este notebook se genera con `scripts/construir_notebook.py` a partir de los scripts. Todas las figuras usan `scripts/estilo.py`
(dibujadas al ancho de la página, con leyendas de al menos 9 puntos impresos) y el informe tiene el mismo contenido que la Sec. 4.1.1 de la tesis.

## 0. Configuración

In [ ]:
import os, sys, json, runpy, subprocess
from pathlib import Path
import numpy as np
from IPython.display import Image as ImagenNB, display   # (los scripts de digitalizacion importan Image de PIL)

BASE = Path.cwd()                                   # carpeta del notebook
DIR_SCRIPTS = BASE / "scripts"; DIR_RES = BASE / "resultados"; DIR_FIG = BASE / "figuras"   # (los scripts usan RES y FIG como texto)
# Carpeta con las corridas de MEIGA (cámbiela si el disco se monta en otra ruta)
CORRIDAS = Path(os.environ.get("CORRIDAS_SIDELNIK", "/ruta/a/Corridas-Sidelnik-500MeV"))   # <- escriba aquí la ruta de las corridas
os.environ["CORRIDAS_SIDELNIK"] = str(CORRIDAS)
CORRIDA = {"pura": CORRIDAS / "150000N-500MeV-Sidelnik-H2O-pura-sinSab-20261012-061022",
           "nacl25a": CORRIDAS / "42649N-parcial-500MeV-Sidelnik-H2O-2.5NaCl-sinSab-20261013",
           "nacl25b": CORRIDAS / "107351N-500MeV-Sidelnik-H2O-2.5NaCl-sinSab-20261013-135213",
           "nacl5": CORRIDAS / "150000N-500MeV-Sidelnik-H2O-5NaCl-sinSab-20261014-065724",
           "nacl10": CORRIDAS / "150000N-500MeV-Sidelnik-H2O-10NaCl-sinSab-20261015-081303"}
# medios de los analisis que leen las corridas por evento: lista de carpetas (tramos) de cada medio
FUENTES = {"pura": [CORRIDA["pura"]], "nacl25": [CORRIDA["nacl25a"], CORRIDA["nacl25b"]], "nacl5": [CORRIDA["nacl5"]], "nacl10": [CORRIDA["nacl10"]]}
TRAMOS = {"nacl25": ("nacl25a", "nacl25b")}      # corridas en dos tramos: se procesan por separado y se unen
MEDIOS = [m for m in ["pura", "nacl25", "nacl5", "nacl10"] if (DIR_RES / f"{m}.npz").exists() or (m in CORRIDA and CORRIDA[m].exists())
          or (m in TRAMOS and all(CORRIDA[t].exists() for t in TRAMOS[m]))]
REPROCESAR = False


def ejecutar(script, *args):
    """Ejecuta scripts/<script> con argumentos, como desde la terminal (mismo intérprete)."""
    argv0 = sys.argv[:]
    try:
        sys.argv = [script] + [str(a) for a in args]
        runpy.run_path(str(DIR_SCRIPTS / script), run_name="__main__")
    finally:
        sys.argv = argv0


def procesar(script, salida, *args):
    """Paso de procesamiento: se salta si la salida existe (salvo REPROCESAR) o si la corrida no está disponible."""
    if (DIR_RES / salida).exists() and not REPROCESAR:
        print(f"{salida}: ya existe, se omite")
    elif not Path(args[0]).exists():
        print(f"{salida}: no se encuentra la corrida {args[0]}")
    else:
        ejecutar(script, *args)


def mostrar(nombre):
    display(ImagenNB(filename=str(DIR_FIG / f"{nombre}.png")))


print("Medios:", MEDIOS); print("Corridas disponibles:", {m: c.exists() for m, c in CORRIDA.items()})

## 1. Procesamiento de las corridas (lee el disco; se omite si los resultados existen)

| Script | Archivo crudo | Salida |
|---|---|---|
| `procesar_corrida.py` | todos salvo `pasos-particulas.tsv` (7,5 min por 150 000 n) | `<medio>.npz` |
| `distancias_alternativas.py` | `pasos-neutrones.tsv`, `capturas-neutrones.tsv` | `<medio>_distancias.npz` |
| `fraccion_captura_neutrones.py` | `pasos-neutrones.tsv` | `fraccion_captura_<medio>.json` |
| `espectro_gamma_pasos.py` | `pasos-gamma.tsv` (bines de 1 keV) | `<medio>_gamma_pasos.npz` |
| `gammas_acumulados.py` | `pasos-gamma.tsv` (hasta 600 MeV) | `<medio>_gammas_hasta600.npz` |
| `origen_electrones.py` | `secundarias-interaccion-gamma.tsv`, `pasos-gamma.tsv` | `<medio>_origen_electrones.npz` |
| `destino_fotones.py` | `pasos-particulas.tsv` (78 GB, 14 min) | `<medio>_fotones.npz` (no va al repositorio) |

Los scripts aceptan corridas interrumpidas: usan solo los neutrones terminados (líneas completas de `Carga-total.txt`).

In [ ]:
for m, c in CORRIDA.items():
    procesar("procesar_corrida.py", f"{m}.npz", c, DIR_RES / f"{m}.npz")
    procesar("distancias_alternativas.py", f"{m}_distancias.npz", c, DIR_RES / f"{m}_distancias.npz")
    procesar("fraccion_captura_neutrones.py", f"fraccion_captura_{m}.json", c, m)
    procesar("espectro_gamma_pasos.py", f"{m}_gamma_pasos.npz", c, DIR_RES / f"{m}_gamma_pasos.npz")
    procesar("gammas_acumulados.py", f"{m}_gammas_hasta600.npz", c, m)
    procesar("origen_electrones.py", f"{m}_origen_electrones.npz", c, m)
    if not (DIR_RES / f"{m}_aceptados_por_run.npy").exists():       # el destino de los fotones solo hace falta para los diagnósticos ópticos
        procesar("destino_fotones.py", f"{m}_fotones.npz", c, DIR_RES / f"{m}_fotones.npz")
for m, (ta, tb) in TRAMOS.items():
    if not (DIR_RES / f"{m}.npz").exists() or REPROCESAR:
        if (DIR_RES / f"{ta}.npz").exists() and (DIR_RES / f"{tb}.npz").exists():
            ejecutar("unir_tramos.py", ta, tb, m)

### Esquema del detector con un evento real con captura en Cl (lee el evento de la corrida si está disponible)

`scripts/esquema_detector.py` — Esquema del WCD de la replica de Sidelnik (corte lateral y = 0, a escala) con un evento real de MEIGA (agua pura): trayectorias de los neutrones (primario de 500 MeV y secundarios), captura en H, gamma de captura y electrones del gamma. Si existe la corrida (CORRIDAS_SIDELNIK) extrae el evento y lo guarda en resultados/evento_ejemplo.json; si no, usa ese archivo. Escribe figuras/fig00_esquema_detector.{png,pdf}. Uso: esquema_detector.py [medio] [run_id]   (por defecto: nacl25, run 124, captura en 35Cl)

In [ ]:
sys.argv = ['esquema_detector.py', "nacl25", "124"]
__file__ = str(DIR_SCRIPTS / "esquema_detector.py")
# ---- código de scripts/esquema_detector.py ----
import os, sys, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyArrowPatch
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MEDIO = sys.argv[1] if len(sys.argv) > 1 else "nacl25"
RUN = int(sys.argv[2]) if len(sys.argv) > 2 else {"pura": 142, "nacl25": 124}[MEDIO]
CORR = {"pura": "150000N-500MeV-Sidelnik-H2O-pura-sinSab-20261012-061022", "nacl25": "42649N-parcial-500MeV-Sidelnik-H2O-2.5NaCl-sinSab-20261013"}
CARP = os.path.join(os.environ.get("CORRIDAS_SIDELNIK", ""), CORR[MEDIO])
CARP = os.path.join(CARP, "Datos-simulacion") if os.path.isdir(os.path.join(CARP, "Datos-simulacion")) else CARP
FJ = os.path.join(RES, f"evento_ejemplo_{MEDIO}.json")
AGUA = {"pura": "agua pura", "nacl25": "agua + 2,5 % NaCl"}[MEDIO]
if os.path.isdir(CARP):
    ev = {"run": RUN, "medio": MEDIO}
    for f, cl in [("pasos-neutrones.tsv", "n"), ("pasos-gamma.tsv", "g"), ("pasos-electrones.tsv", "e")]:
        hdr = open(os.path.join(CARP, f)).readline().lstrip("# ").rstrip("\n").split("\t")
        out = []
        for d in pd.read_csv(os.path.join(CARP, f), sep="\t", skiprows=1, header=None, names=hdr, chunksize=500000,
                             usecols=["run_id", "particle", "track_id", "parent_id", "step_number", "creator_process", "step_process",
                                      "pre_x_cm", "pre_z_cm", "post_x_cm", "post_z_cm", "pre_energy_MeV"]):
            out.append(d[d.run_id == RUN])
            if d.run_id.min() > RUN:
                break
        d = pd.concat(out).sort_values(["track_id", "step_number"])
        if cl == "e":                               # trazas de e+- que nacen sobre el umbral Cherenkov (~0,26 MeV)
            E0 = d[d.step_number == 1].set_index("track_id").pre_energy_MeV
            d = d[d.track_id.isin(E0[E0 > 0.3].index)]
        ev[cl] = [{"trk": int(t), "cre": str(g.creator_process.iloc[0]), "E0": float(g.pre_energy_MeV.iloc[0]), "padre": int(g.parent_id.iloc[0]),
                   "fin": str(g.step_process.iloc[-1]),
                   "x": [float(g.pre_x_cm.iloc[0])] + g.post_x_cm.round(2).tolist(), "z": [float(g.pre_z_cm.iloc[0])] + g.post_z_cm.round(2).tolist()}
                  for t, g in d.groupby("track_id")]
    json.dump(ev, open(FJ, "w"))
if not os.path.exists(FJ):
    raise SystemExit(f"Falta {FJ}: conecte el disco con las corridas (variable CORRIDAS_SIDELNIK) para extraer el evento {RUN} de {MEDIO}.")
ev = json.load(open(FJ))
estilo.aplicar(**{"axes.grid": False})
fig, ax = plt.subplots(figsize=(8, 8.6))
Rw, H, TY, AC = 48.0, 133.0, 0.012, 0.05; E = 1.2        # paredes exageradas x E para que se vean
z0 = AC                                                 # fondo del agua
ax.add_patch(Rectangle((-Rw - (TY + AC) * 20, z0 - AC * 20), 2 * Rw + 2 * (TY + AC) * 20, H + 2 * AC * 20 + TY * 20, fc="#8a8984", ec="none"))
ax.add_patch(Rectangle((-Rw - TY * 40, z0), 2 * Rw + 2 * TY * 40, H + TY * 40, fc="#f2f2ee", ec="none"))
ax.add_patch(Rectangle((-Rw, z0), 2 * Rw, H, fc="#d6eef8" if MEDIO == "pura" else "#d9f0e0", ec="none"))
ax.add_patch(Rectangle((-5.08, z0 + H - 5.0), 10.16, 5.0, fc="#7cc58f", ec="#2e7d32", lw=1))
ax.plot([-60, 60], [z0 - AC * 20 - 0.6] * 2, color="#c75000", lw=5, solid_capstyle="butt")
# evento: capturas clasificadas por los gammas que emiten (un gamma de ~2,224 MeV = H; cascada = Cl)
COLH, COLCL, COLE = "#2a78d6", "#d62728", "#7b1fa2"
gpad = {}
for t in ev["g"]:
    if t["cre"] == "nCapture":
        gpad.setdefault(t["padre"], []).append(t)
capt = []
for t in ev["n"]:
    if t["fin"] == "nCapture":
        gs = gpad.get(t["trk"], []); es = sorted([g["E0"] for g in gs], reverse=True)
        tipo = "H" if len(es) == 1 and abs(es[0] - 2.2244) < 0.01 else "Cl"
        capt.append({"x": t["x"][-1], "z": t["z"][-1], "tipo": tipo, "E": es, "trk": t["trk"]})
colg = {g["trk"]: (COLH if c["tipo"] == "H" else COLCL) for c in capt for g in gpad.get(c["trk"], [])}


def dibujar(a, lw=1.0):
    for t in ev["n"]:
        prim = t["trk"] == 1
        a.plot(t["x"], t["z"], "-", color="#0b0b0b" if prim else "#5b5a57", lw=(1.6 if prim else 0.8) * lw, zorder=4)
    for t in ev["g"]:
        cap = t["cre"] == "nCapture"
        a.plot(t["x"], t["z"], "--" if cap else ":", color=colg.get(t["trk"], "#eda100"), lw=(1.6 if cap else 0.8) * lw, zorder=5)
    for t in ev["e"]:
        a.plot(t["x"], t["z"], "-", color=COLE, lw=2.2 * lw, zorder=5, solid_capstyle="round")
    for c in capt:
        a.plot(c["x"], c["z"], "o", color=COLH if c["tipo"] == "H" else COLCL, ms=8, zorder=6, mec="white", mew=0.8)


dibujar(ax)
x0, zf = ev["n"][0]["x"][0], ev["n"][0]["z"][0]
ax.add_patch(FancyArrowPatch((x0 - 14, zf), (x0 - 0.5, zf), arrowstyle="-|>", mutation_scale=16, color="#0b0b0b", lw=1.8))
fs = 10.5
ax.annotate("Neutrón incidente\n500 MeV (fuente a 1 cm\nde la pared, media altura)", (x0 - 8, zf), (-73, 30), fontsize=fs, ha="center",
            arrowprops=dict(arrowstyle="-", color="#555555", lw=0.8))
ax.annotate("PMT 10,16 × 10,16 × 5 cm\n(en la tapa)", (0, z0 + H - 2.5), (25, 152), fontsize=fs, ha="left", arrowprops=dict(arrowstyle="-", color="#555555", lw=0.8))
ax.annotate("Tyvek 0,12 mm", (Rw + 0.3, 115), (58, 125), fontsize=fs, arrowprops=dict(arrowstyle="-", color="#555555", lw=0.8))
ax.annotate("Acero inoxidable 0,5 mm", (Rw + 1.6, 100), (58, 110), fontsize=fs, arrowprops=dict(arrowstyle="-", color="#555555", lw=0.8))
ax.text(10, 8, f"Volumen activo: {AGUA}\n(R = 48 cm, H = 133 cm)", ha="center", fontsize=fs, color="#1f4e79", bbox=dict(fc="white", ec="none", alpha=0.75, pad=1.5))
for k, c in enumerate(sorted(capt, key=lambda c: c["tipo"])):
    if c["tipo"] == "H":
        txt, col = "Captura n + $^1$H → d + γ\n(2224 keV)", COLH
    else:
        txt = "Captura n + $^{35}$Cl → $^{36}$Cl + cascada γ:\n" + " + ".join(f"{e * 1e3:.0f}" for e in c["E"]) + f" keV = {sum(c['E']):.3f} MeV".replace(".", ",")
        col = COLCL
    ax.annotate(txt, (c["x"], c["z"]), ((-93, 141) if c["tipo"] == "Cl" else (-44, 42)), fontsize=fs, color=col, ha="left", bbox=dict(fc="white", ec="none", alpha=0.8, pad=1.5),
                arrowprops=dict(arrowstyle="-", color=col, lw=0.8))
# ampliacion de la zona de moderacion y capturas
ins = ax.inset_axes([0.58, 0.16, 0.36, 0.36])
dibujar(ins, 1.0)
xs_ = [c["x"] for c in capt] + [x0 + 2]; zs_ = [c["z"] for c in capt]
xm, zm = (min(xs_) + max(xs_)) / 2, (min(zs_) + max(zs_)) / 2; dx = max(max(xs_) - min(xs_), max(zs_) - min(zs_)) / 2 + 5
for c in capt:
    for g in gpad.get(c["trk"], []):
        k = min(len(g["x"]) - 1, 1)
        xe, ze = np.clip(g["x"][k], xm - dx * 0.85, xm + dx * 0.85), np.clip(g["z"][k], zm - dx * 0.85, zm + dx * 0.85)
        ins.text(xe, ze, f"{g['E0'] * 1e3:.0f}", fontsize=9, color=COLH if c["tipo"] == "H" else COLCL, ha="center", va="center",
                 bbox=dict(fc="white", ec="none", alpha=0.85, pad=0.6))
ins.set_xlim(xm - dx, xm + dx); ins.set_ylim(zm - dx, zm + dx); ins.set_aspect("equal")
ins.set_facecolor("#d6eef8" if MEDIO == "pura" else "#d9f0e0"); ins.set_xticks([]); ins.set_yticks([])
ins.set_title(f"ampliación ({2 * dx:.0f} cm de lado; γ en keV)", fontsize=9)
for sp in ins.spines.values():
    sp.set_visible(True); sp.set_color("#555555")
ax.indicate_inset_zoom(ins, edgecolor="#555555", lw=0.8)
from matplotlib.lines import Line2D
h = [Line2D([], [], color="#0b0b0b", lw=1.6, label="neutrón primario"), Line2D([], [], color="#5b5a57", lw=0.8, label="neutrones secundarios (moderación)"),
     Line2D([], [], color=COLH, ls="--", lw=1.6, label="captura en H y su γ"), Line2D([], [], color=COLCL, ls="--", lw=1.6, label="captura en $^{35}$Cl y su cascada γ"), Line2D([], [], color="#eda100", ls=":", lw=1, label="otros γ"),
     Line2D([], [], color=COLE, lw=2, label="e$^\\pm$ sobre el umbral Cherenkov (recorren mm)")]
ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=9.5, frameon=False)
ax.set_xlim(-95, 95); ax.set_ylim(-6, 165); ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"WCD de la réplica de Sidelnik en MEIGA (corte y = 0, a escala; paredes exageradas)\nevento real de {AGUA}: run {ev['run']}, proyección x–z", fontsize=10.5)
fig.tight_layout()
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig00_esquema_detector{'' if MEDIO == 'nacl25' else '_' + MEDIO}.{e}"))
print("n trazas", len(ev["n"]), "g", len(ev["g"]), "e", len(ev["e"]))

In [ ]:
mostrar("fig00_esquema_detector")

In [ ]:
ejecutar("esquema_detector.py", "pura", "142"); mostrar("fig00_esquema_detector_pura")

## 2. Digitalización de las figuras de referencia

### Fig. 3 de Sidelnik (espectro gamma)

`scripts/digitalizar_fig3.py` — Digitalizacion automatica (por color de pixel) de la Fig. 3 de Sidelnik et al. (2020), espectro gamma. Marco: columna 220 = 0 MeV, columna 1000 = 20 MeV; fila 552 = 1e0, fila 54 = 1e6 (escala log). Por columna (25,6 keV) guarda, para cada medio, la fila mas alta (envolvente superior = picos) y la mediana de los pixeles de su color (continuo). La curva roja (agua pura) queda tapada en parte por las otras: su mediana es fiable en el continuo; las otras series son nubes de marcadores. Salida: datos/fig3_sidelnik_digitalizada.csv

In [ ]:
sys.argv = ['digitalizar_fig3.py']
__file__ = str(DIR_SCRIPTS / "digitalizar_fig3.py")
# ---- código de scripts/digitalizar_fig3.py ----
import os
import numpy as np, pandas as pd
from PIL import Image
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
im = np.asarray(Image.open(os.path.join(R, "datos", "fig3_sidelnik_espectro_gamma.png")).convert("RGB")).astype(int)
r, g, b = im[..., 0], im[..., 1], im[..., 2]
COLS = {"pura": (r > 190) & (g < 70) & (b < 70),
        "nacl25": (r < 60) & (g > 120) & (g < 190) & (b > 80) & (b < 150),
        "nacl5": (r > 110) & (r < 190) & (g < 60) & (b > 170),
        "nacl10": (r > 200) & (g > 130) & (g < 190) & (b < 60)}
X0, X1, Y0, Y1 = 220, 1000, 552, 54
filas = []
for c in range(X0 + 1, X1):
    E = (c - X0) / (X1 - X0) * 20
    d = {"E_MeV": E}
    for m, mk in COLS.items():
        rows = np.flatnonzero(mk[Y1 + 1:Y0, c]) + Y1 + 1
        if c > 850:  # recuadro de la leyenda (filas 54-190, E > 16,5 MeV)
            rows = rows[rows > 190]
        for nom, val in (("max", rows.min() if len(rows) else np.nan), ("mediana", np.median(rows) if len(rows) else np.nan)):
            d[f"{m}_{nom}"] = 10 ** (6 * (Y0 - val) / (Y0 - Y1)) if np.isfinite(val) else np.nan
    filas.append(d)
t = pd.DataFrame(filas); t.to_csv(os.path.join(R, "datos", "fig3_sidelnik_digitalizada.csv"), index=False)
print(t.describe().T[["count", "min", "max"]])

### Fig. 5 de Sidelnik (electrones producidos por gammas)

`scripts/digitalizar_fig5.py` — Digitalizacion por color de pixel de la Fig. 5 de Sidelnik et al. (2020): espectro de electrones producidos por gammas en el volumen activo. Marco: columna 384 = 0 MeV, columna 1167 = 10 MeV; fila 634 = 1e1 y 98,6 pixeles por decada (marcas en 634, 535 y 338). Se excluye la leyenda (filas < 275, columnas > 850). Por columna (12,8 keV) guarda la fila mas alta y la mediana de cada color. Salida: datos/fig5_sidelnik_digitalizada.csv

In [ ]:
sys.argv = ['digitalizar_fig5.py']
__file__ = str(DIR_SCRIPTS / "digitalizar_fig5.py")
# ---- código de scripts/digitalizar_fig5.py ----
import os
import numpy as np, pandas as pd
from PIL import Image
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
im = np.asarray(Image.open(os.path.join(R, "datos", "fig5_sidelnik_espectro_electrones.png")).convert("RGB")).astype(int)
r, g, b = im[..., 0], im[..., 1], im[..., 2]
COLS = {"pura": (r > 190) & (g < 80) & (b < 80),
        "nacl25": (r < 80) & (g > 120) & (g < 190) & (b > 80) & (b < 150),
        "nacl5": (r > 100) & (r < 190) & (g < 70) & (b > 170),
        "nacl10": (r > 200) & (g > 130) & (g < 200) & (b < 80)}
X0, X1, Y0, DEC = 384, 1167, 634, 98.6
filas = []
for c in range(X0 + 2, X1 - 1):
    d = {"E_MeV": (c - X0) / (X1 - X0) * 10}
    for m, mk in COLS.items():
        rows = np.flatnonzero(mk[130:Y0, c]) + 130
        if c > 850:
            rows = rows[rows > 275]
        for nom, f in (("max", np.min), ("mediana", np.median)):
            d[f"{m}_{nom}"] = 10 ** (1 + (Y0 - f(rows)) / DEC) if len(rows) else np.nan
    filas.append(d)
t = pd.DataFrame(filas); t.to_csv(os.path.join(R, "datos", "fig5_sidelnik_digitalizada.csv"), index=False)
print(t.notna().sum().to_dict())
for E in (0.5, 1, 1.5, 1.9, 2.1, 3, 4, 5, 5.8, 6.1, 7, 8, 9):
    i = (t.E_MeV - E).abs().idxmin(); print(E, {k: round(t.loc[i, k]) if np.isfinite(t.loc[i, k]) else None for k in ("pura_mediana", "nacl25_mediana")})

## 3. Captura, distancia y resumen (Tabla 2 y Fig. 2 de Sidelnik)

In [ ]:
# Interacciones del neutron primario (lee las corridas; se omite si el resultado existe)
for m, cs in FUENTES.items():
    if (DIR_RES / f"inelasticas_primario_{m}.json").exists() and not REPROCESAR:
        print(f"inelasticas_primario_{m}.json: ya existe, se omite")
    elif all(c.exists() for c in cs):
        ejecutar("inelasticas_primario.py", m, *cs)

### Interacciones inelásticas del neutrón primario

`scripts/figura_inelasticas_primario.py` — Figura de las interacciones inelasticas del neutron primario (resultados de inelasticas_primario.py): (a) profundidad de la inelastica a lo largo del haz, (b) multiplicidad media de productos, (c) espectro de energia de los productos (agua pura), (d) nucleos residuales mas frecuentes. Escribe figuras/fig20_inelasticas_primario.{png,pdf}

In [ ]:
sys.argv = ['figura_inelasticas_primario.py']
__file__ = str(DIR_SCRIPTS / "figura_inelasticas_primario.py")
# ---- código de scripts/figura_inelasticas_primario.py ----
import os, sys, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MED = [m for m in ["pura", "nacl25", "nacl5", "nacl10"] if os.path.exists(os.path.join(RES, f"inelasticas_primario_{m}.json"))]
ETQ = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
COL = {"pura": "#e41a1c", "nacl25": "#1a8a1a", "nacl5": "#7b1fa2", "nacl10": "#f5a623"}
J = {m: json.load(open(os.path.join(RES, f"inelasticas_primario_{m}.json"))) for m in MED}
Z = {m: np.load(os.path.join(RES, f"inelasticas_primario_{m}.npz")) for m in MED}
estilo.aplicar()
fig, axs = plt.subplots(2, 2, figsize=(8, 10.4))
a = axs[0, 0]; b = np.arange(0, 100.5, 2)
for m in MED:
    pr = Z[m]["prof"]; agua = np.char.startswith(Z[m]["mat"], "Water") | np.char.startswith(Z[m]["mat"], "SaltyWater")
    a.hist(pr[agua], b, histtype="step", lw=1.5, color=COL[m], weights=np.full(agua.sum(), 1 / J[m]["N"]), label=ETQ[m])
x = np.linspace(0, 96, 200); lam = J["pura"]["prof_media_cm"]
a.axvline(0.062, color="#555555", ls=":", lw=1); a.set_xlabel("Profundidad desde la cara de entrada [cm]"); a.set_ylabel("Inelásticas por neutrón / 2 cm")
a.set_title("(a) Dónde: profundidad de la inelástica\ndel primario en el medio", fontsize=10.5)
a = axs[0, 1]; T = ["neutron", "proton", "deuteron", "triton", "He3", "alpha", "pi+", "pi-", "pi0", "gamma", "núcleo residual"]
NOM = ["n", "p", "d", "t", "$^3$He", "α", "π$^+$", "π$^-$", "π$^0$", "γ", "res."]
xx = np.arange(len(T)); w = 0.8 / len(MED)
for i, m in enumerate(MED):
    v = [J[m]["multiplicidad_media"][t] for t in T]
    a.bar(xx + (i - (len(MED) - 1) / 2) * w, v, w, color=COL[m], label=ETQ[m])
a.set_yscale("log"); a.set_ylim(0.01, 5); a.set_xticks(xx); a.set_xticklabels(NOM, fontsize=9.5)
a.set_ylabel("Partículas por inelástica"); a.set_title("(b) Resultado: multiplicidad\nde los productos", fontsize=10.5)
a = axs[1, 0]; EB = Z["pura"]["EB"]; c = np.sqrt(EB[1:] * EB[:-1]); dl = np.log(EB[1:] / EB[:-1]); n = J["pura"]["inelasticas"]
for t, et, col in [("neutron", "n", "#0b0b0b"), ("proton", "p", "#2a78d6"), ("alpha", "α", "#eda100"), ("deuteron", "d", "#8a8984"),
                   ("pi-", "π$^-$", "#7b1fa2"), ("pi0", "π$^0$", "#c51b8a"), ("gamma", "γ", "#1baf7a")]:
    h = Z["pura"][f"E|{t}"] / n / dl; s = h > 0
    a.step(c[s], h[s], where="mid", color=col, lw=1.3, label=et)
a.axvline(500, color="#e41a1c", ls=":", lw=1)
a.set_xscale("log"); a.set_yscale("log"); a.set_xlim(0.05, 600); a.set_ylim(1e-4, 5)
a.set_xlabel("Energía cinética [MeV]"); a.set_ylabel("dN/d(ln E) por inelástica")
a.set_title("(c) Energía de los productos\n(agua pura; punteada: 500 MeV)", fontsize=10.5); a.set_ylim(1e-4, 60); a.legend(fontsize=8.5, ncol=4, loc="upper left", columnspacing=0.8, handlelength=1.2)
a = axs[1, 1]; rs = list(J["pura"]["nucleos_residuales"])[:12]; yy = np.arange(len(rs))
for i, m in enumerate(MED):
    a.barh(yy + (i - (len(MED) - 1) / 2) * w, [100 * J[m]["nucleos_residuales"].get(k, 0) for k in rs], w, color=COL[m], label=ETQ[m])
import re
a.set_yticks(yy); a.set_yticklabels([re.sub(r"([A-Za-z]+)(\d+)", r"$^{\2}$\1", k) for k in rs], fontsize=9.5); a.invert_yaxis()
a.set_xlabel("% de las inelásticas"); a.set_title("(d) Núcleo residual\n(blanco $^{16}$O en el 97 %)", fontsize=10.5)
estilo.leyenda_medios(fig, MED, COL, ETQ)
fig.tight_layout(rect=(0, 0.035, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig20_inelasticas_primario.{e}"))
print("ok")

In [ ]:
mostrar("fig20_inelasticas_primario")

### ¿Desaparece el primario? Relación con las capturas y con la definición de Sidelnik

`scripts/primario_y_captura.py` — Relacion entre el destino del neutron primario y las capturas (de resultados/<medio>.npz): eventos con captura, fraccion de ellos en que el primario tuvo una inelastica, primario capturado como la misma traza y capturas por evento con captura. Escribe resultados/primario_y_captura_<medio>.json

In [ ]:
sys.argv = ['primario_y_captura.py']
__file__ = str(DIR_SCRIPTS / "primario_y_captura.py")
# ---- código de scripts/primario_y_captura.py ----
import os, json
import numpy as np
RES = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..", "resultados")
for m in ["pura", "nacl25", "nacl5", "nacl10"]:
    f = os.path.join(RES, f"{m}.npz")
    if not os.path.exists(f):
        continue
    z = np.load(f); N = int(z["N"]); inel = np.unique(z["prim_inel_run"]); mat = z["cap_material"]
    ag = np.char.startswith(mat, "Water") | np.char.startswith(mat, "SaltyWater")
    cr = np.unique(z["cap_run_id"][ag]); capP = np.unique(z["cap_run_id"][(z["cap_track_id"] == 1) & ag])
    out = {"eventos_con_captura": len(cr) / N, "frac_eventos_captura_con_inel_primario": float(np.isin(cr, inel).mean()),
           "primario_capturado": len(capP) / N, "capturas_por_evento_con_captura": float(ag.sum() / len(cr))}
    json.dump(out, open(os.path.join(RES, f"primario_y_captura_{m}.json"), "w"), indent=1); print(m, out)

In [ ]:
# Dispersiones elasticas: reconstruccion de vertices (lee pasos-neutrones y pasos-particulas, ~15 min por corrida; se omite si existe)
for m, cs in FUENTES.items():
    if (DIR_RES / f"elasticas_{m}.json").exists() and not REPROCESAR:
        print(f"elasticas_{m}.json: ya existe, se omite")
    elif all(c.exists() for c in cs):
        ejecutar("elasticas_vertices.py", m, *cs)

### Dispersiones elásticas: blanco, energía, ángulo, ξ y leyes de conservación

`scripts/figura_elasticas.py` — Figura y tabla de la reconstruccion de vertices elasticos (resultados de elasticas_vertices.py): (a) blanco reconstruido segun la energia del neutron; (b) masa del blanco reconstruida solo con el neutron; (c) fraccion de energia perdida por colision (H y O) frente al limite cinematico; (d) angulo de dispersion en el laboratorio; (e) decremento logaritmico medio xi segun la energia frente a la teoria; (f) balance de energia e impulso por vertice. Escribe figuras/fig21_elasticas.{png,pdf} y resultados/elasticas_resumen.json

In [ ]:
sys.argv = ['figura_elasticas.py']
__file__ = str(DIR_SCRIPTS / "figura_elasticas.py")
# ---- código de scripts/figura_elasticas.py ----
import os, sys, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
from matplotlib.lines import Line2D
K_ = "#0b0b0b"
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MED = [m for m in ["pura", "nacl25", "nacl5", "nacl10"] if os.path.exists(os.path.join(RES, f"elasticas_{m}.npz"))]
ETQ = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
COL = {"pura": "#e41a1c", "nacl25": "#1a8a1a", "nacl5": "#7b1fa2", "nacl10": "#f5a623"}
def figura():
    Z = {m: np.load(os.path.join(RES, f"elasticas_{m}.npz")) for m in MED}
    AO = 15.86                                   # masa del 16O en unidades de masa del neutron
    aO = ((AO - 1) / (AO + 1)) ** 2; XI_O = 1 + aO * np.log(aO) / (1 - aO); PERD_O = 1 - aO
    estilo.aplicar()
    fig, axs = plt.subplots(3, 2, figsize=(8.6, 10.0))
    eb = np.logspace(-6, np.log10(600), 33); ec = np.sqrt(eb[1:] * eb[:-1]); resumen = {}
    a = axs[0, 0]
    for m in MED:
        E, b = Z[m]["Ein"], Z[m]["blanco_cin"]; ok = np.isin(b, ["H", "O", "Na", "Cl"])
        n = np.histogram(E[ok], eb)[0].astype(float)
        for bl, ls in [("H", "-"), ("O", "--"), ("Na+Cl", ":")]:
            sel = ok & (np.isin(b, ["Na", "Cl"]) if bl == "Na+Cl" else (b == bl))
            if m == "pura" and bl == "Na+Cl":
                continue
            f_ = np.histogram(E[sel], eb)[0] / np.maximum(n, 1); f_[n < 200] = np.nan
            a.plot(ec, 100 * f_, ls, color=COL[m], lw=1.5, label=f"{ETQ[m]}: {bl}")
    a.set_xscale("log"); a.set_xlabel("Energía del neutrón [MeV]"); a.set_ylabel("% de las colisiones elásticas")
    a.set_title("(a) Con quién: blanco según la energía\n(reconstruido por cinemática, E > 1 eV)", fontsize=10.5); a.set_ylim(0, 135)
    a.legend(handles=[Line2D([], [], color=K_, ls=l_, label=t_) for l_, t_ in [("-", "H"), ("--", "O"), (":", "Na + Cl")]], ncol=3, loc="upper center", handlelength=1.8, columnspacing=1.0)
    a = axs[0, 1]; mb = np.logspace(2.5, 4.7, 300)
    for m in MED:
        Mk = Z[m]["M_rec_cin"]; Mk = Mk[np.isfinite(Mk) & (Z[m]["Ein"] > 1e-3)]
        a.hist(Mk, mb, histtype="step", color=COL[m], lw=1.3, label=ETQ[m])
    for M_, t, yy in [(938.27, "p", 0.92), (14895.1, "$^{16}$O", 0.92), (21409, "$^{23}$Na", 0.82), (32573, "$^{35}$Cl", 0.92)]:
        a.axvline(M_, color="#555555", ls=":", lw=0.9); a.text(M_ * 1.03, yy, t, transform=a.get_xaxis_transform(), fontsize=9)
    a.set_xscale("log"); a.set_yscale("log"); a.set_xlabel("Masa del blanco reconstruida [MeV]"); a.set_ylabel("Colisiones")
    a.set_title("(b) Masa del blanco deducida\ndel neutrón (E > 1 keV)", fontsize=10.5)
    a = axs[1, 0]; fb = np.linspace(0, 1, 101); z = Z["pura"]; sel = (z["Ein"] > 1e-3) & (z["Ein"] < 1)
    for bl, col, et in [("H", "#2a78d6", "H"), ("O", "#d62728", "$^{16}$O")]:
        s = sel & (z["blanco_cin"] == bl); fr = (z["Ein"][s] - z["Eout"][s]) / z["Ein"][s]
        a.hist(fr, fb, histtype="step", color=col, lw=1.5, density=True, label=f"blanco {et}")
    a.plot([0, 1], [1, 1], ":", color="#2a78d6", lw=1, label="teoría H")
    a.plot([0, PERD_O, PERD_O], [1 / PERD_O, 1 / PERD_O, 0], ":", color="#d62728", lw=1, label="teoría $^{16}$O")
    a.set_yscale("log"); a.set_ylim(0.05, 3000); a.set_xlabel("Fracción de energía perdida ΔE/E"); a.set_ylabel("Densidad")
    a.set_title("(c) Energía: pérdida por colisión\n(agua pura, 1 keV–1 MeV)", fontsize=10.5); a.legend(fontsize=8, ncol=2, loc="upper right", columnspacing=0.6, handlelength=1.2)
    a = axs[1, 1]; cb = np.linspace(-1, 1, 81)
    for (lo, hi, ls, et) in [(1e-3, 1, "-", "1 keV–1 MeV"), (20, 600, "--", "> 20 MeV")]:
        s0 = (z["Ein"] > lo) & (z["Ein"] < hi)
        for bl, col in [("H", "#2a78d6"), ("O", "#d62728")]:
            s = s0 & (z["blanco_cin"] == bl)
            a.hist(z["cos_lab"][s], cb, histtype="step", color=col, ls=ls, lw=1.3, density=True, label=f"{bl}, {et}".replace("1 keV–1 MeV", "< 1 MeV"))
    a.set_yscale("log"); a.set_ylim(top=a.get_ylim()[1] * 300); a.set_xlabel("cos θ del neutrón en el laboratorio"); a.set_ylabel("Densidad")
    a.set_title("(d) Ángulo de dispersión\n(agua pura)", fontsize=10.5); a.legend(fontsize=8, ncol=2, loc="upper right", columnspacing=0.6, handlelength=1.2)
    a = axs[2, 0]
    for m in MED:
        E, b, xi = Z[m]["Ein"], Z[m]["blanco_cin"], Z[m]["xi"]
        for bl, mk in [("H", "o"), ("O", "s")]:
            s = b == bl; idx = np.digitize(E[s], eb) - 1
            v = np.array([xi[s][idx == i].mean() if (idx == i).sum() > 300 else np.nan for i in range(len(ec))])
            a.plot(ec, v, mk, color=COL[m], ms=4, mfc="none" if m != "pura" else COL[m], label=f"{ETQ[m]}: {bl}")
    a.axhline(1, color="#2a78d6", ls=":", lw=1); a.axhline(XI_O, color="#d62728", ls=":", lw=1)
    a.text(30, 0.62, "ξ$_H$ = 1", fontsize=9, color="#2a78d6"); a.text(1e-3, XI_O * 0.72, f"ξ$_O$ = {XI_O:.3f}".replace(".", ","), fontsize=9, color="#d62728")
    a.set_xscale("log"); a.set_yscale("log"); a.set_ylim(5e-3, 3); a.set_xlabel("Energía del neutrón [MeV]"); a.set_ylabel("ξ = ⟨ln(E/E')⟩")
    a.set_title("(e) Decremento logarítmico medio\n(punteado: teoría, dispersión isótropa en el CM)", fontsize=10.5); a.legend(handles=[Line2D([], [], color=K_, marker=m_, ls="", label=t_) for m_, t_ in [("o", "H"), ("s", "O")]], ncol=2, loc="lower left")
    a = axs[2, 1]
    for m in MED:
        rec = Z[m]["rec"]; s = rec != "sin retroceso"
        dE = np.abs(Z[m]["dE_cons"][s]) / Z[m]["Ein"][s]; dp = Z[m]["dp_rel"][s]
        a.hist(np.log10(np.maximum(dE, 1e-12)), np.linspace(-12, 0, 61), histtype="step", color=COL[m], lw=1.3, label=f"{ETQ[m]}: |ΔE|/E")
        a.hist(np.log10(np.maximum(dp[np.isfinite(dp)], 1e-12)), np.linspace(-12, 0, 61), histtype="step", color=COL[m], ls="--", lw=1.3, label=f"{ETQ[m]}: |Δp|/p")
    a.set_yscale("log"); a.set_xlabel("log$_{10}$ del desbalance relativo"); a.set_ylabel("Vértices con retroceso")
    a.set_title("(f) Conservación de energía e impulso\nen cada vértice", fontsize=10.5); a.set_ylim(top=a.get_ylim()[1] * 30)
    a.legend(handles=[Line2D([], [], color=K_, ls="-", label="|ΔE|/E"), Line2D([], [], color=K_, ls="--", label="|Δp|/p")], ncol=2, loc="upper left")
    estilo.leyenda_medios(fig, MED, COL, ETQ, ncol=2)
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    for e in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"fig21_elasticas.{e}"))
    resumen = {"xi_O_teoria": XI_O, "perdida_max_O_teoria": PERD_O}
    json.dump(resumen, open(os.path.join(RES, "elasticas_resumen.json"), "w"), indent=1)
    print(resumen)


if MED:
    figura()
else:      # los elasticas_<medio>.npz (~400 MB) no van al repositorio: se conserva la figura y elasticas_resumen.json
    print("Faltan resultados/elasticas_<medio>.npz (se regeneran con elasticas_vertices.py); se usa la figura guardada.")

In [ ]:
mostrar("fig21_elasticas")

### Canales de reacción inelástica, valores Q y gammas de desexcitación

`scripts/canales_reaccion.py` — canal X(n,x)Y de cada inelástica, valor Q (AME2020), energía del neutrón y líneas gamma del núcleo residual; origen de los gammas de 4–7 MeV. Las tablas por medio están en el informe (Sec. ‘Ecuaciones, energías y porcentajes de todos los procesos’).

In [ ]:
for m, cs in FUENTES.items():
    if (DIR_RES / f"canales_{m}.json").exists() and not REPROCESAR:
        print(f"canales_{m}.json: ya existe, se omite")
    elif all(c.exists() for c in cs):
        ejecutar("canales_reaccion.py", m, *cs)
import pandas as pd
for m in FUENTES:
    if (DIR_RES / f"canales_{m}.json").exists():
        d = json.load(open(DIR_RES / f"canales_{m}.json"))
        t = pd.DataFrame(d["canales"])[["canal", "frac", "Q_MeV", "E_n_mediana_MeV", "gammas_por_reaccion", "lineas"]].head(20)
        t["frac"] = (100 * t.frac).round(2)
        print(m, "- inelásticas por neutrón incidente:", round(d["inelasticas_por_neutron_incidente"], 3)); display(t)
        print("gammas de 4-7 MeV por núcleo emisor:", {k: round(100 * v, 1) for k, v in d["gammas_4_7_MeV_por_residual"].items()})

### Resumen por medio y figuras 01–05

`scripts/analizar.py` — Analisis de las corridas tipo Sidelnik (MEIGA, 500 MeV horizontales) y comparacion con Sidelnik et al., Adv. Space Res. 65 (2020) 2216 (Tabla 2 y Figs. 2-6).  Lee resultados/<medio>.npz (procesar_corrida.py); escribe resultados/resumen.json y figuras/. Uso: analizar.py [medio ...]   (por defecto: los .npz que existan de pura, nacl25, nacl5, nacl10)

In [ ]:
sys.argv = ['analizar.py']
__file__ = str(DIR_SCRIPTS / "analizar.py")
# ---- código de scripts/analizar.py ----
import os, sys, json, collections
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo

R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
TODOS = ["pura", "nacl25", "nacl5", "nacl10"]
ETQ = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
COL = {"pura": "#e34948", "nacl25": "#1baf7a", "nacl5": "#7b1fa2", "nacl10": "#eda100"}   # colores de Sidelnik
SIDELNIK = {"pura": (25.18, 8.4), "nacl25": (26.73, 8.0), "nacl5": (27.65, 7.7), "nacl10": (29.26, 7.3)}
BORDE = -48.0            # cara interna del agua por donde entra el haz [cm]
ENTRADA = np.array([-48.0, 0.0, 66.562])
E_CHER = 0.264           # MeV, umbral Cherenkov e- en agua (n = 1,33)
MED = [m for m in (sys.argv[1:] or TODOS) if os.path.exists(os.path.join(RES, f"{m}.npz"))]
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "#52514e", "axes.grid": True, "grid.color": "#e4e3df",
                     "grid.linewidth": 0.6, "lines.linewidth": 1.5, "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "savefig.dpi": 160})
S, D = {}, {}


def savefig(fig, n, rect=None):
    fig.tight_layout(rect=rect) if rect else fig.tight_layout(); fig.savefig(os.path.join(FIG, n + ".png")); fig.savefig(os.path.join(FIG, n + ".pdf")); plt.close(fig)


def bi(k, n):
    p = k / n
    return [float(p), float(np.sqrt(p * (1 - p) / n))]


def me(a):
    a = np.asarray(a, float)
    return [float(a.mean()), float(a.std(ddof=1) / np.sqrt(len(a)))]


def pico(x, h, ventana=5):
    """posicion del maximo: histograma suavizado + parabola sobre los 5 bins centrales"""
    hs = np.convolve(h, np.ones(ventana) / ventana, mode="same")
    i = int(np.argmax(hs))
    j = slice(max(i - 2, 0), i + 3)
    a, b, c = np.polyfit(x[j], hs[j], 2)
    return float(-b / (2 * a)) if a < 0 else float(x[i])


AGUA = ("Water", "SaltyWater")
NUC = {(1, 1): "H", (1, 2): "H2", (8, 16): "O16", (8, 17): "O17", (8, 18): "O18", (11, 23): "Na23",
       (17, 35): "Cl35", (17, 37): "Cl37"}

for m in MED:
    z = np.load(os.path.join(RES, f"{m}.npz"), allow_pickle=False)
    N = int(z["N"]); s = S.setdefault(m, {}); s["N"] = N; d = D.setdefault(m, {})
    # ---------------- capturas ----------------
    mat = z["cap_material"]; agua = np.char.startswith(mat, "Water") | np.char.startswith(mat, "SaltyWater")
    run = z["cap_run_id"]
    s["capturas_total"] = int(len(run)); s["capturas_en_agua"] = int(agua.sum())
    s["capturas_por_material"] = dict(collections.Counter(mat).most_common())
    s["capturas_por_neutron_agua"] = float(agua.sum() / N)
    s["P_al_menos_1_captura_agua"] = bi(len(np.unique(run[agua])), N)
    prim = z["cap_track_id"] == 1
    s["P_primario_capturado"] = bi(int((prim & agua).sum()), N)
    Z, A = z["cap_target_Z"], z["cap_target_A"]
    nuc = np.array([NUC.get((int(a), int(b)), f"Z{a}A{b}") for a, b in zip(Z, A)])
    s["capturas_agua_por_nucleo"] = {k: [int(v), v / agua.sum()] for k, v in collections.Counter(nuc[agua]).most_common()}
    x, y, zz = z["cap_post_x_cm"], z["cap_post_y_cm"], z["cap_post_z_cm"]
    dist = (x - BORDE)[agua]
    d3 = np.sqrt((x - ENTRADA[0]) ** 2 + (y - ENTRADA[1]) ** 2 + (zz - ENTRADA[2]) ** 2)[agua]
    xb = np.arange(0, 100.01, 0.5); xc = 0.5 * (xb[1:] + xb[:-1])
    h = np.histogram(dist, xb)[0]
    s["distancia_max_absorcion_cm"] = pico(xc, h)
    s["distancia_media_cm"] = float(dist.mean()); s["distancia_mediana_cm"] = float(np.median(dist))
    s["distancia3D_max_absorcion_cm"] = pico(xc, np.histogram(d3, xb)[0])
    d.update(dist=dist, d3=d3, xc=xc, hdist=h)
    alt = os.path.join(RES, f"{m}_distancias.npz")      # distancias_alternativas.py
    if os.path.exists(alt):
        za = np.load(alt)
        for k in ["d_vertice", "d_inel_primario"]:
            v = za[k][np.isfinite(za[k])]
            hv = np.histogram(v, xb)[0]
            s[f"{k}_max_cm"] = pico(xc, hv); s[f"{k}_mediana_cm"] = float(np.median(v))
            d["h_" + k] = hv
    t = z["cap_global_time_ns"][agua] / 1e3   # us
    tb = np.logspace(-1, 4, 101); ht = np.histogram(t, tb)[0]; tcen = np.sqrt(tb[1:] * tb[:-1])
    sel = (tcen > 20) & (tcen < 1000) & (ht > 0)
    p = np.polyfit(tcen[sel], np.log(ht[sel] / np.diff(tb)[sel]), 1, w=np.sqrt(ht[sel]))
    s["tau_captura_us"] = float(-1 / p[0]); s["tiempo_captura_mediana_us"] = float(np.median(t))
    d.update(t=t)
    ek = z["cap_pre_energy_MeV"][agua] * 1e9   # meV
    s["E_captura_mediana_meV"] = float(np.median(ek)); s["frac_captura_epitermica_gt_0.5eV"] = float((ek > 500).mean())
    d["ecap"] = ek
    # ---------------- gammas de captura ----------------
    sc_run, sc_trk, part, E = z["sc_run"], z["sc_ptrk"], z["sc_part"], z["sc_E"]
    g = part == "gamma"
    clave = sc_run.astype(np.int64) * 10_000_000 + sc_trk
    ckey = run.astype(np.int64) * 10_000_000 + z["cap_track_id"]
    nmap = dict(zip(ckey, nuc))
    gn = np.array([nmap.get(k, "?") for k in clave[g]])
    s["gamma_captura_por_neutron"] = float(g.sum() / N)
    s["E_gamma_captura_por_neutron_MeV"] = float(E[g].sum() / N)
    u, c = np.unique(np.round(E[g], 4), return_counts=True)
    s["lineas_captura_top"] = [[float(e), int(n)] for n, e in sorted(zip(c, u), reverse=True)[:12]]
    d.update(gcapE=E[g], gcapN=gn)
    # ---------------- primario ----------------
    pp, px = z["prim1_proc"], z["prim1_x"]
    s["primario_primera_interaccion"] = dict(collections.Counter(pp))
    s["primario_inelastica_en_agua"] = bi(len(np.unique(z["prim_inel_run"])), N)
    di = z["prim_inel_x"] - BORDE
    s["profundidad_inelastica_primario_media_cm"] = float(di.mean())
    d["dinel"] = di
    # ---------------- espectros ----------------
    EB, GB = z["EB"], z["GB"]
    H = {k[2:]: z[k] for k in z.files if k.startswith("H_")}
    d.update(EB=EB, GB=GB, H=H)
    gcre = {k.split("|")[1]: v for k, v in H.items() if k.startswith("gcre|")}
    s["gammas_creados_agua_por_neutron"] = float(sum(v.sum() for v in gcre.values()) / N)
    s["gammas_creados_por_proceso"] = {k: float(v.sum() / N) for k, v in sorted(gcre.items(), key=lambda kv: -kv[1].sum())[:8]}
    ec = 0.5 * (GB[1:] + GB[:-1])
    egl = {k: v for k, v in H.items() if k.startswith("egamlin|")}
    eT = sum(egl.values())
    s["e_de_gammas_por_neutron"] = float(eT.sum() / N)
    s["e_de_gammas_frac_sobre_umbral"] = float(eT[ec > E_CHER].sum() / eT.sum())
    s["e_de_gammas_por_proceso"] = {k.split("|", 1)[1]: float(v.sum() / N) for k, v in egl.items()}
    # ---------------- Cherenkov y carga ----------------
    nch = z["cher_n_por_run"]; q = z["Q"]
    s["fotones_cherenkov_por_neutron"] = me(nch)
    s["Q_media"] = me(q); s["eficiencia_Q_ge1"] = bi(int((q >= 1).sum()), N)
    s["Q_sobre_fotones"] = float(q.sum() / nch.sum())
    s["carga_igual_json"] = bool(np.array_equal(q, z["json_Q"])) if "json_Q" in z.files else None   # corrida interrumpida: sin output.json.gz
    tpe = z["pe_t_ns"]; eve = z["pe_ev"]
    vent = {"<1 us": (0, 1e3), "1 us-1 ms": (1e3, 1e6), ">1 ms": (1e6, np.inf)}
    s["carga_por_ventana"] = {}
    for k, (a, b) in vent.items():
        w = (tpe >= a) & (tpe < b)
        qq = np.bincount(eve[w], minlength=N)
        s["carga_por_ventana"][k] = {"frac": float(w.mean()), "eficiencia_ge1": float((qq >= 1).mean())}
    d.update(q=q, nch=nch, tpe=tpe)
    print(m, "listo")

json.dump(S, open(os.path.join(RES, "resumen.json"), "w"), indent=1, ensure_ascii=False, default=float)

# ===================== figuras =====================
ref2 = os.path.join(R, "datos", "fig2_agua_pura_digitalizada.csv")
estilo.aplicar()
from matplotlib.lines import Line2D
fig, axs = plt.subplots(1, 2, figsize=(8, 5.4))
for m in MED:
    dd = D[m]
    axs[0].step(dd["xc"], dd["hdist"], where="mid", color=COL[m], label=f"{ETQ[m]} (máx. {S[m]['distancia_max_absorcion_cm']:.1f} cm)")
    axs[0].axvline(SIDELNIK[m][1], color=COL[m], ls=":", lw=1)
    if "h_d_vertice" in dd:
        axs[1].step(dd["xc"], dd["h_d_vertice"] / dd["h_d_vertice"].max(), where="mid", color=COL[m],
                    label=f"{ETQ[m]}: vértice → captura (máx. {S[m]['d_vertice_max_cm']:.1f} cm)")
        axs[1].step(dd["xc"], dd["h_d_inel_primario"] / dd["h_d_inel_primario"].max(), where="mid", color=COL[m], ls="--", lw=1,
                    label=f"{ETQ[m]}: inelástica del primario → captura (máx. {S[m]['d_inel_primario_max_cm']:.1f} cm)")
    axs[1].step(dd["xc"], dd["hdist"] / dd["hdist"].max(), where="mid", color=COL[m], ls=":", lw=1, label=f"{ETQ[m]}: cara de entrada → captura")
if os.path.exists(ref2) and "pura" in MED:
    r = np.loadtxt(ref2, delimiter=",", skiprows=1)
    axs[1].plot(r[:, 0], r[:, 1] / r[:, 1].max(), "o", ms=3, color="#52514e", label="Sidelnik Fig. 2, agua pura (digitalizada)")
axs[0].set_xlabel("Distancia desde la cara de entrada [cm]"); axs[0].set_ylabel("Capturas / 0,5 cm"); axs[0].set_ylim(0, 1.15 * max(D[m_]["hdist"].max() for m_ in MED))
axs[0].set_title("(a) Distancia a lo largo del haz\n(punteada: máximo de Sidelnik)", fontsize=10)
axs[1].set_xlabel("Distancia [cm]"); axs[1].set_ylabel("Normalizado al máximo"); axs[1].set_xlim(0, 100)
axs[1].set_title("(b) Definiciones alternativas\nfrente a la Fig. 2 de Sidelnik", fontsize=10)
fig.legend(handles=[Line2D([], [], color="#0b0b0b", lw=1.5, label="vértice del n → captura"),
                       Line2D([], [], color="#0b0b0b", ls="--", lw=1, label="inelástica → captura"),
                       Line2D([], [], color="#0b0b0b", ls=":", lw=1, label="cara de entrada → captura"),
                       Line2D([], [], color="#52514e", marker="o", ms=4, ls="", label="Sidelnik, Fig. 2")], loc="lower center", ncol=2, bbox_to_anchor=(0.5, 0.0), frameon=False, handlelength=1.6)
estilo.leyenda_medios(fig, MED, COL, ETQ, abajo=0.11)
savefig(fig, "fig01_distancia_captura", rect=(0, 0.18, 1, 1))

fig, axs = plt.subplots(1, 2, figsize=(8, 4.0))
for m in MED:
    dd = D[m]; GB = dd["GB"]; gc = 0.5 * (GB[1:] + GB[:-1])
    hg = sum(v for k, v in dd["H"].items() if k.startswith("gcrelin|"))
    axs[0].step(gc, hg / S[m]["N"], where="mid", color=COL[m], lw=0.8, label=ETQ[m])
    # Fig. 4 de Sidelnik: cuentas acumuladas de derecha a izquierda
    axs[1].step(gc, np.cumsum(hg[::-1])[::-1] / S[m]["N"], where="mid", color=COL[m], label=ETQ[m])
axs[0].set_yscale("log"); axs[0].set_xlim(0, 12); axs[0].set_xlabel("Energía del gamma creado en el agua [MeV]"); axs[0].set_ylabel("Gammas / 5 keV / neutrón")
axs[1].set_yscale("log"); axs[1].set_xlim(0, 20); axs[1].set_xlabel("Energía [MeV]"); axs[1].set_ylabel("N(≥E) por neutrón")
estilo.leyenda_medios(fig, MED, COL, ETQ)
savefig(fig, "fig02_espectro_gamma", rect=(0, 0.09, 1, 1))

fig, axs = plt.subplots(1, 2, figsize=(8, 4.0))
for m in MED:
    dd = D[m]; GB = dd["GB"]; gc = 0.5 * (GB[1:] + GB[:-1])
    he = sum(v for k, v in dd["H"].items() if k.startswith("egamlin|"))
    axs[0].step(gc, he / S[m]["N"], where="mid", color=COL[m], lw=0.8, label=ETQ[m])
    EB = dd["EB"]; ce = np.sqrt(EB[1:] * EB[:-1]); w = np.log(EB[1:] / EB[:-1])
    hl = sum(v for k, v in dd["H"].items() if k.startswith("egam|"))
    axs[1].step(ce, hl / S[m]["N"] / w, where="mid", color=COL[m], label=ETQ[m])
for e, t in [(1.994, "borde 2,223 MeV"), (8.33, "borde 8,58 MeV")]:
    axs[0].axvline(e, color="#8a8984", ls=":", lw=1); axs[0].text(e + 0.15, 0.97, t, transform=axs[0].get_xaxis_transform(), fontsize=8.5, rotation=90, va="top")
axs[0].set_yscale("log"); axs[0].set_xlim(0, 12); axs[0].set_xlabel("Energía del electrón creado por un gamma [MeV]"); axs[0].set_ylabel("e$^\\pm$ / 5 keV / neutrón")
axs[1].set_xscale("log"); axs[1].set_yscale("log"); axs[1].axvline(E_CHER, color="#8a8984", ls=":", lw=1)
axs[1].set_xlabel("Energía del electrón [MeV]"); axs[1].set_ylabel("dN/d(ln E) por neutrón")
estilo.leyenda_medios(fig, MED, COL, ETQ)
savefig(fig, "fig03_espectro_electrones", rect=(0, 0.09, 1, 1))

ref6 = os.path.join(R, "datos", "fig6_sidelnik_digitalizada_700pts.csv")
fig, axs = plt.subplots(1, 2, figsize=(8, 4.4))
qb = np.arange(0, 1000, 10)
for m in MED:
    q = D[m]["q"]
    h = np.histogram(q[q >= 1], qb)[0]
    axs[0].step(qb[:-1] + 5, h / h.sum(), where="mid", color=COL[m], label=f"{ETQ[m]}: MEIGA")
    tb = np.logspace(-1, 19, 201)
    ht = np.histogram(D[m]["tpe"], tb)[0]
    axs[1].step(np.sqrt(tb[1:] * tb[:-1]), ht / np.diff(tb) / S[m]["N"], where="mid", color=COL[m], label=ETQ[m])
if os.path.exists(ref6):
    import pandas as pd
    r = pd.read_csv(ref6)
    cols = {"pura": "Pure_H2O_counts_digitized", "nacl25": "NaCl_2p5_counts_digitized", "nacl5": "NaCl_5_counts_digitized", "nacl10": "NaCl_10_counts_digitized"}
    for m in MED:
        c = r[cols[m]].to_numpy(); xs = r.Numero_de_fotones.to_numpy()
        hb = np.interp(qb[:-1] + 5, xs, c, left=np.nan, right=np.nan)
        axs[0].plot(qb[:-1] + 5, hb / np.nansum(hb), "--", color=COL[m], lw=1, label=f"{ETQ[m]}: Sidelnik, Fig. 6")
axs[0].set_yscale("log"); axs[0].set_xlabel("Fotoelectrones en el PMT (Q ≥ 1)"); axs[0].set_ylabel("Fracción / 10 p.e.")
axs[0].set_title("(a) Carga en el PMT frente a la Fig. 6", fontsize=10.5); axs[1].set_title("(b) Tiempo de llegada de los p.e.", fontsize=10.5)
axs[0].legend(handles=[Line2D([], [], color="#0b0b0b", label="MEIGA"), Line2D([], [], color="#0b0b0b", ls="--", lw=1, label="Sidelnik, Fig. 6")], loc="upper right")
estilo.leyenda_medios(fig, MED, COL, ETQ)
axs[1].set_xscale("log"); axs[1].set_yscale("log"); axs[1].set_xlabel("Tiempo de llegada del fotoelectrón [ns]"); axs[1].set_ylabel("p.e. / ns / neutrón")
for tt, l in [(1e3, "1 µs"), (1e6, "1 ms")]:
    axs[1].axvline(tt, color="#c3c2b7", ls=":", lw=1)
savefig(fig, "fig04_carga_y_tiempos", rect=(0, 0.08, 1, 1))

fig, axs = plt.subplots(1, 2, figsize=(8, 4.3))
for m in MED:
    axs[0].hist(D[m]["t"], bins=np.logspace(-1, 4, 101), histtype="step", color=COL[m], lw=1.4,
                label=f"{ETQ[m]} (τ = {S[m]['tau_captura_us']:.0f} µs)")
    axs[1].hist(D[m]["ecap"], bins=np.logspace(-1, 9, 101), histtype="step", color=COL[m], lw=1.4, label=ETQ[m])
axs[0].set_xscale("log"); axs[0].set_yscale("log"); axs[0].set_xlabel("Tiempo de captura [µs]"); axs[0].set_ylabel("Capturas / bin")
axs[1].set_xscale("log"); axs[1].set_yscale("log"); axs[1].axvline(25.3, color="#8a8984", ls=":", lw=1)
axs[1].set_ylim(top=3e5)
axs[1].annotate("25,3 meV\n(kT a 20 °C)", xy=(25.3, 1.2e5), xytext=(2e3, 6e4), fontsize=10.5, color="#555555", va="center",
                arrowprops=dict(arrowstyle="->", color="#555555", lw=0.9))
axs[1].set_xlabel("Energía al capturarse [meV]"); axs[1].set_ylabel("Capturas / bin")
fig.legend(*axs[0].get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False)
savefig(fig, "fig05_tiempo_energia_captura", rect=(0, 0.14, 1, 1))
print(json.dumps(S, indent=1, ensure_ascii=False, default=float)[:7000])

In [ ]:
mostrar("fig01_distancia_captura")
mostrar("fig02_espectro_gamma")
mostrar("fig03_espectro_electrones")
mostrar("fig04_carga_y_tiempos")
mostrar("fig05_tiempo_energia_captura")

## 4. Espectros gamma y de electrones (Figs. 3–5 de Sidelnik)

### Espectro gamma en el formato de la Fig. 3 (agua pura)

`scripts/espectro_gamma_tipo_sidelnik.py` — Espectro de energia de los gammas creados en el medio, en el formato de la Fig. 3 de Sidelnik et al. (2020), y comparacion con la figura digitalizada (datos/fig3_sidelnik_digitalizada.csv). Lee resultados/<medio>_gamma_pasos.npz (espectro_gamma_pasos.py, bins de 1 keV) y resultados/<medio>.npz. Escribe figuras/fig11_espectro_gamma_tipo_sidelnik.{png,pdf} y resultados/espectro_gamma_tipo_sidelnik.json. Uso: espectro_gamma_tipo_sidelnik.py [medio]

In [ ]:
sys.argv = ['espectro_gamma_tipo_sidelnik.py', "pura"]
__file__ = str(DIR_SCRIPTS / "espectro_gamma_tipo_sidelnik.py")
# ---- código de scripts/espectro_gamma_tipo_sidelnik.py ----
import os, sys, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
m = sys.argv[1] if len(sys.argv) > 1 else "pura"
z = np.load(os.path.join(RES, f"{m}_gamma_pasos.npz")); p = np.load(os.path.join(RES, f"{m}.npz"))
ref = pd.read_csv(os.path.join(R, "datos", "fig3_sidelnik_digitalizada.csv"))
ETQ = {"pura": "Pure H$_2$O", "nacl25": "2.5% NaCl in H$_2$O", "nacl5": "5% NaCl in H$_2$O", "nacl10": "10% NaCl in H$_2$O"}
COLM = {"pura": "#e41a1c", "nacl25": "#1a8a1a", "nacl5": "#7b1fa2", "nacl10": "#f5a623"}   # colores de cada medio en la tesis
COLREF = {"pura": "pura", "nacl25": "nacl25", "nacl5": "nacl5", "nacl10": "nacl10"}[m]
W = 2                                            # keV: el cociente pico/continuo a 3-5 MeV coincide con el de Sidelnik
def reb(h, w):
    h = h[:len(h) // w * w].reshape(-1, w).sum(1); return h, (np.arange(len(h)) + 0.5) * w * 1e-3
h, c = reb(z["primer_paso_agua"], W)
def linea(hh, cc, E, ww=0.02):
    return float(hh[np.abs(cc - E) < ww].max())
def cont(hh, cc, E, ww=0.25):
    return float(np.median(hh[np.abs(cc - E) < ww]))
def lin_ref(E, ww=0.1):
    return float(ref[f"{COLREF}_max"][(ref.E_MeV - E).abs() < ww].max())
def cont_ref(E, ww=0.25):
    return float(ref[f"{COLREF}_mediana"][(ref.E_MeV - E).abs() < ww].median())
pk, pkr = linea(h, c, 2.2232), lin_ref(2.223)
sin_rd = z["primer_paso_agua"] - z["primer|RadioactiveDecayBase"]   # (casi todo en el agua)
h2, _ = reb(sin_rd, W)
pos = {k.split("|")[1]: int(p[k].sum()) for k in p.files if k.startswith("H_ecre|") and k.endswith("|e+")}
out = {"ancho_bin_keV": W, "gammas_creados_en_agua": int(z["primer_paso_agua"].sum()),
       "pico_2223": {"MEIGA_cuentas": pk, "Sidelnik_cuentas_aprox": pkr},
       "cociente_0511_2223": {"Sidelnik": lin_ref(0.511) / pkr, "MEIGA": linea(h, c, 0.511) / pk,
                              # los gammas de 0,511 los crea 'annihil': se quita la fraccion de positrones beta+ (estimacion)
                              "MEIGA_sin_decaimiento_radiactivo": linea(h, c, 0.511) / pk * (1 - pos.get("RadioactiveDecayBase", 0) / sum(pos.values()))},
       "cociente_6130_2223": {"Sidelnik": lin_ref(6.13) / pkr, "MEIGA": linea(h, c, 6.1294) / pk,
                              "MEIGA_sin_decaimiento_radiactivo": linea(h2, c, 6.1294) / pk},
       "pico_sobre_continuo": {str(E): {"Sidelnik": pkr / cont_ref(E), "MEIGA": pk / max(cont(h, c, E), 0.5)} for E in (1, 3, 4, 5, 10, 15)},
       "positrones_por_proceso": pos,
       "linea_0511_por_creador": {k.split("|")[1]: int(z[k][509:513].sum()) for k in z.files if k.startswith("primer|") and z[k][509:513].sum() > 0},
       "linea_6130_por_creador": {k.split("|")[1]: int(z[k][6126:6133].sum()) for k in z.files if k.startswith("primer|") and z[k][6126:6133].sum() > 0},
       "continuo_1_8_por_creador": {k.split("|")[1]: int(z[k][1000:8000].sum()) for k in z.files if k.startswith("primer|") and z[k][1000:8000].sum() > 100},
       "continuo_10_17_por_creador": {k.split("|")[1]: int(z[k][10000:17000].sum()) for k in z.files if k.startswith("primer|") and z[k][10000:17000].sum() > 20}}
SUF = "" if m == "pura" else f"_{m}"
N = len(p["Q"]); out["N"] = N
# lineas del cloro (captura en 35Cl) y del O: cociente con la de 2,223 MeV. La resolucion de la digitalizacion (25,6 keV/pixel)
# no separa 0,511/0,517 ni 6,11/6,13 MeV: se compara la linea mas alta dentro de +-30 keV en ambos casos.
LIN = {"0,511/0,517": 0.514, "1,165 (Cl)": 1.1647, "1,951 (Cl)": 1.9512, "6,11/6,13": 6.12, "7,41 (Cl)": 7.4141, "7,79 (Cl)": 7.7901, "8,58 (Cl)": 8.5785}
out["lineas_sobre_2223"] = {k: {"Sidelnik": lin_ref(E, 0.04) / pkr, "MEIGA": linea(h, c, E, 0.03) / pk} for k, E in LIN.items()}
json.dump(out, open(os.path.join(RES, f"espectro_gamma_tipo_sidelnik{SUF}.json"), "w"), indent=1)

estilo.aplicar()
fig, axs = plt.subplots(2, 2, figsize=(8, 10.3 if m == "pura" else 9.6))
a = axs[0, 0]; s = h > 0
a.plot(c[s], h[s], "-o", color=COLM[m], lw=0.5, ms=1.6, label={"pura": "MEIGA, agua pura", "nacl25": "MEIGA, agua + 2,5 % NaCl", "nacl5": "MEIGA, agua + 5 % NaCl", "nacl10": "MEIGA, agua + 10 % NaCl"}[m])
a.set_title(f"(a) MEIGA, formato de la Fig. 3\n(intervalos de {W} keV, {N} n)", fontsize=10.5)
a = axs[0, 1]; k = pkr / pk
a.plot(c[s], h[s] * k, "-", color=COLM[m], lw=0.6, alpha=0.8, label=f"MEIGA × {k:.2f}".replace(".", ","))
a.plot(ref.E_MeV, ref[f"{COLREF}_mediana"], "s", color="#0b0b0b", ms=2.5, label="Sidelnik (mediana)")
a.plot(ref.E_MeV, ref[f"{COLREF}_max"], "^", color="#6b6b6b", ms=2.5, label="Sidelnik (máximo)")
a.set_title("(b) Comparación con la Fig. 3\nde Sidelnik", fontsize=10.5)
for a in axs[0]:
    a.set_yscale("log"); a.set_ylim(0.8, 1e8); a.set_xlim(0, 20); a.set_xlabel("Energía [MeV]"); a.set_ylabel("Cuentas"); a.legend(fontsize=8.5, loc="upper right", handlelength=1.3)
# (c) composicion por proceso creador, bins de 50 keV
a = axs[1, 0]
grupos = [("nCapture", "captura n (H)" if m == "pura" else "captura n (H y Cl)", "#2a78d6"), ("neutronInelastic", "inelástica n", "#1baf7a"), ("eBrem", "bremsstrahlung", "#eda100"),
          ("annihil", "aniquilación e$^+$", "#e34948"), ("RadioactiveDecayBase", "decaimiento radiactivo", "#4a3aa7"),
          ("protonInelastic", "inelástica p", "#8a8984")]
for nom, et, col in grupos:
    hh, cc = reb(z[f"primer|{nom}"], 50); s = hh > 0
    a.step(cc[s], hh[s], where="mid", color=col, lw=1.2, label=et)
a.set_yscale("log"); a.set_xlim(0, 20); a.set_ylim(0.5, 1e7); a.set_xlabel("Energía [MeV]"); a.set_ylabel("Gammas / 50 keV")
a.set_title("(c) MEIGA: gammas según el\nproceso que los crea", fontsize=10.5)
# (d) cocientes de lineas
a = axs[1, 1]
if m == "pura":
    cs_ = [("0,511 / 2,223 MeV", "cociente_0511_2223"), ("6,13 / 2,223 MeV", "cociente_6130_2223")]
    xs = np.arange(2); wb = 0.27
    for i, (lab, key, col) in enumerate([("Sidelnik", "Sidelnik", "#0b0b0b"), ("MEIGA", "MEIGA", COLM[m]),
                                         ("MEIGA sin decaim. radiactivo", "MEIGA_sin_decaimiento_radiactivo", "#f4a3a3")]):
        v = [out[k_][key] for _, k_ in cs_]
        a.bar(xs + (i - 1) * wb, v, wb, color=col, label=lab)
        for x_, y_ in zip(xs + (i - 1) * wb, v):
            a.text(x_, y_ * 1.15, f"{y_:.3f}".replace(".", ","), ha="center", fontsize=8.5, rotation=90, va="bottom")
    a.set_yscale("log"); a.set_ylim(5e-3, 300); a.set_xticks(xs); a.set_xticklabels([l.replace(" / ", "\n/ ") for l, _ in cs_])
    a.set_ylabel("Altura / altura de 2,223 MeV"); a.legend(fontsize=8.5, loc="upper right")

else:
    L = out["lineas_sobre_2223"]; xs = np.arange(len(L)); wb = 0.38
    for i, (lab, key, col) in enumerate([("Sidelnik", "Sidelnik", "#0b0b0b"), ("MEIGA", "MEIGA", COLM[m])]):
        v = [L[k_][key] for k_ in L]
        a.bar(xs + (i - 0.5) * wb, v, wb, color=col, label=lab)
    a.set_yscale("log"); a.set_ylim(5e-3, 100); a.set_xticks(xs); a.set_xticklabels([k_.replace(" (Cl)", "") for k_ in L], fontsize=9, rotation=45, ha="right"); a.set_xlabel("Línea [MeV]")
    a.set_ylabel("Altura / altura de 2,223 MeV"); a.legend(fontsize=8.5, loc="upper right")
a.set_title("(d) Cocientes de líneas\n(no dependen del intervalo)", fontsize=10.5)
fig.legend(*axs[1, 0].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False, columnspacing=1.0, handlelength=1.4, title="(c) proceso creador:", title_fontsize=11.5)
fig.tight_layout(rect=(0, 0.085, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig11_espectro_gamma_tipo_sidelnik{SUF}.{e}"))
print(json.dumps(out, indent=1))

In [ ]:
mostrar("fig11_espectro_gamma_tipo_sidelnik")

In [ ]:
# Mismo análisis para los medios con NaCl (código idéntico al de la celda anterior)
for m in MEDIOS[1:]:
    ejecutar("espectro_gamma_tipo_sidelnik.py", m); mostrar(f"fig11_espectro_gamma_tipo_sidelnik_{m}")

### Gammas acumulados (Fig. 4 de Sidelnik)

`scripts/fig4_gammas_acumulados.py` — Fig. 4 de Sidelnik et al. (2020): gammas producidos, acumulados de derecha a izquierda (N(>=E), bines de 1 MeV). Digitaliza los marcadores de las dos imagenes del articulo (completa 0-100 MeV y zoom 1-9 MeV) por color y compara con MEIGA (resultados/<medio>_gammas_hasta600.npz, de gammas_acumulados.py), con cuentas por 150000 neutrones. Escribe datos/fig4_sidelnik_digitalizada.csv, resultados/fig4_gammas_acumulados.json y figuras/fig17_gammas_acumulados.{png,pdf}

In [ ]:
sys.argv = ['fig4_gammas_acumulados.py']
__file__ = str(DIR_SCRIPTS / "fig4_gammas_acumulados.py")
# ---- código de scripts/fig4_gammas_acumulados.py ----
import os, json
import numpy as np, pandas as pd
from PIL import Image
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG, DAT = (os.path.join(R, d) for d in ("resultados", "figuras", "datos"))
def colores(im):
    r, g, b = im[..., 0], im[..., 1], im[..., 2]
    return {"pura": (r > 190) & (g < 90) & (b < 90),
            "nacl25": (r < 160) & (g > 165) & (b > 135) & (b < 205) & (g - r > 40),
            "nacl5": (r > 100) & (r < 190) & (g < 70) & (b > 160),
            "nacl10": (r > 220) & (g > 150) & (g < 230) & (b < 185) & (g - b > 30)}
def digit(f, x0, dx, y_dec, px_dec, Es, ylim):
    im = np.asarray(Image.open(os.path.join(DAT, f)).convert("RGB")).astype(int); C = colores(im); out = {}
    for m, mk in C.items():
        v = []
        for E in Es:
            c = int(round(x0 + E * dx)); sub = mk[ylim[0]:ylim[1], c - 7:c + 8]
            rows = np.flatnonzero(sub.any(1)) + ylim[0]
            if len(rows) < 3:
                v.append(np.nan); continue
            # marcador = grupo de filas contiguas mas grande
            grupos = np.split(rows, np.flatnonzero(np.diff(rows) > 2) + 1); gmax = max(grupos, key=len)
            v.append(10 ** (y_dec[1] + (y_dec[0] - np.median(gmax)) / px_dec))
        out[m] = v
    return out
EZ = np.arange(1, 10); EC = np.arange(1, 100)
z = digit("fig4_sidelnik_acumulado_zoom.png", 380, 74.9, (511, 4), 209, EZ, (35, 505))       # fila 511 = 1e4
c = digit("fig4_sidelnik_acumulado.png", 258, 7.49, (578, 2), 96, EC, (100, 575))             # fila 578 = 1e2
for m in c:                                                     # recuadro de la leyenda (E > 60 MeV, N > 1e5): no son datos
    c[m] = [np.nan if (E > 60 and v > 1e5) else v for E, v in zip(EC, c[m])]
ref = pd.DataFrame({"E_MeV": EC})
for m in z:
    v = np.array(c[m], float); v[:9] = np.where(np.isfinite(z[m]), z[m], v[:9]); ref[m] = v
# desde 9 MeV las cuatro curvas del articulo se superponen y los marcadores se tapan: se usa el valor comun (mediana de los colores)
comun = np.nanmedian(ref[["pura", "nacl25", "nacl5", "nacl10"]].to_numpy(), axis=1)
for m in ("pura", "nacl25", "nacl5", "nacl10"):
    ref.loc[ref.E_MeV >= 9, m] = comun[ref.E_MeV >= 9]
ref.to_csv(os.path.join(DAT, "fig4_sidelnik_digitalizada.csv"), index=False)
# MEIGA
ETQ = {"pura": "Pure H$_2$O", "nacl25": "2.5% NaCl in H$_2$O", "nacl5": "5% NaCl in H$_2$O", "nacl10": "10% NaCl in H$_2$O"}
COL = {"pura": "#e41a1c", "nacl25": "#1baf7a", "nacl5": "#7b1fa2", "nacl10": "#e69f00"}
MED = [m for m in ETQ if os.path.exists(os.path.join(RES, f"{m}_gammas_hasta600.npz"))]
cum = {}
for m in MED:
    q = np.load(os.path.join(RES, f"{m}_gammas_hasta600.npz")); k = 150000 / int(q["N"])
    for tag in ("todos", "sin_piones"):
        h = q[tag] * k; cs = np.cumsum(h[::-1])[::-1]
        cum[(m, tag)] = np.array([cs[int(round(E / 0.01))] for E in EC])
out = {"puntos": {}, "NaCl_sobre_pura": {}, "cociente_Sidelnik_MEIGA_pura": {}}
for m in MED:
    out["puntos"][m] = {int(E): {"MEIGA": float(cum[(m, "todos")][i]), "MEIGA_sin_piones": float(cum[(m, "sin_piones")][i]),
                                 "Sidelnik": float(ref[m][i]) if np.isfinite(ref[m][i]) else None} for i, E in enumerate(EC) if E in (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 17, 20, 30, 50, 70, 90, 99)}
for i, E in enumerate(EC):
    if E <= 20 or E % 10 == 0:
        out["cociente_Sidelnik_MEIGA_pura"][int(E)] = float(ref.pura[i] / cum[("pura", "todos")][i])
        if "nacl25" in MED:
            out["NaCl_sobre_pura"][int(E)] = {"MEIGA": float(cum[("nacl25", "todos")][i] / cum[("pura", "todos")][i]),
                                             "Sidelnik": float(ref.nacl25[i] / ref.pura[i])}
# linea de 2,223 MeV respecto a los gammas de >= 3 MeV
for m in MED:
    out.setdefault("N2menosN3_sobre_N3", {})[m] = {"MEIGA": float((cum[(m, "todos")][1] - cum[(m, "todos")][2]) / cum[(m, "todos")][2]),
                                                  "Sidelnik": float((ref[m][1] - ref[m][2]) / ref[m][2])}
    out.setdefault("N20_sobre_N3", {})[m] = {"MEIGA": float(cum[(m, "todos")][19] / cum[(m, "todos")][2]),
                                            "MEIGA_sin_piones": float(cum[(m, "sin_piones")][19] / cum[(m, "sin_piones")][2]),
                                            "Sidelnik": float(ref[m][19] / ref[m][2])}
json.dump(out, open(os.path.join(RES, "fig4_gammas_acumulados.json"), "w"), indent=1)
estilo.aplicar()
fig, axs = plt.subplots(2, 2, figsize=(8, 10.0))
for a, lim in ((axs[0, 0], (0, 100)), (axs[0, 1], (0, 10))):
    for m in MED:
        a.plot(EC, cum[(m, "todos")], "s" if m == "pura" else "o", color=COL[m], ms=4)
        a.plot(EC, ref[m], "s" if m == "pura" else "o", mfc="none", mec=COL[m], ms=6, mew=1.2)
    a.set_yscale("log"); a.set_xlim(*lim); a.set_xlabel("Energía [MeV]"); a.set_ylabel("N(≥E) por 150 000 n")
axs[0, 0].set_ylim(1e2, 1e7); axs[0, 0].set_title("(a) Gammas acumulados de derecha\na izquierda (Fig. 4 de Sidelnik)", fontsize=10.5)
axs[0, 1].set_ylim(1e4, 2e6); axs[0, 1].set_title("(b) Detalle por debajo de 10 MeV", fontsize=10.5)
a = axs[1, 0]
for m in MED:
    a.plot(EC, cum[(m, "todos")] / cum[(m, "todos")][2], "-", color=COL[m])
    a.plot(EC, cum[(m, "sin_piones")] / cum[(m, "sin_piones")][2], ":", color=COL[m])
    a.plot(EC, ref[m] / ref[m][2], "s" if m == "pura" else "o", mfc="none", mec=COL[m], ms=5)
a.set_yscale("log"); a.set_xlim(0, 100); a.set_xlabel("Energía [MeV]"); a.set_ylabel("N(≥E) / N(≥3 MeV)")
a.set_title("(c) Forma: normalizado a N(≥3 MeV)", fontsize=10.5)
a = axs[1, 1]
if "nacl25" in MED:
    Ei = np.arange(1, 21)
    for m_ in MED[1:]:
        a.plot(Ei, cum[(m_, "todos")][:20] / cum[("pura", "todos")][:20], "o-", color=COL[m_], ms=4)
        a.plot(Ei, ref[m_][:20] / ref.pura[:20], "o--", color=COL[m_], mfc="none", ms=5, lw=1)
    a.axhline(1, color="#8a8984", lw=0.8); a.set_xlabel("Energía [MeV]"); a.set_ylabel("N(≥E) NaCl / N(≥E) pura")
    a.set_title("(d) Efecto del NaCl\n(no depende de la normalización)", fontsize=10.5)
from matplotlib.lines import Line2D
h_ = [Line2D([], [], ls="none", marker="s", color="#8a8984", label="MEIGA (lleno; línea continua)"),
      Line2D([], [], ls="none", marker="s", mfc="none", mec="#0b0b0b", label="Sidelnik (vacío; línea discontinua)"),
      Line2D([], [], ls=":", color="#8a8984", label="MEIGA sin eventos con piones")]
h_ += [Line2D([], [], ls="-", lw=6, color=COL[m], label={"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}[m]) for m in MED]
fig.legend(handles=h_, loc="lower center", ncol=2, fontsize=9, columnspacing=1.5)
fig.tight_layout(rect=(0, 0.115, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig17_gammas_acumulados.{e}"))
print(json.dumps({k: out[k] for k in ("cociente_Sidelnik_MEIGA_pura", "NaCl_sobre_pura", "N2menosN3_sobre_N3", "N20_sobre_N3")}, indent=1))

In [ ]:
mostrar("fig17_gammas_acumulados")

### Electrones producidos por gammas (Fig. 5 de Sidelnik)

`scripts/espectro_electrones_tipo_sidelnik.py` — Espectro de electrones producidos por gammas en el agua (Compton, fotoelectrico y pares; e-) en el formato de la Fig. 5 de Sidelnik et al. (2020) y comparacion con la figura digitalizada (datos/fig5_sidelnik_digitalizada.csv). Las cuentas de MEIGA se escalan a 150000 neutrones. Compara: (a) escala con bines de 20 keV, (b) forma, (c) cociente NaCl/agua pura (no depende de la normalizacion), (d) saltos en los bordes Compton. Escribe figuras/fig13_espectro_electrones_tipo_sidelnik.{png,pdf} y resultados/espectro_electrones_tipo_sidelnik.json

In [ ]:
sys.argv = ['espectro_electrones_tipo_sidelnik.py']
__file__ = str(DIR_SCRIPTS / "espectro_electrones_tipo_sidelnik.py")
# ---- código de scripts/espectro_electrones_tipo_sidelnik.py ----
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MED = [m for m in ["pura", "nacl25", "nacl5", "nacl10"] if os.path.exists(os.path.join(RES, f"{m}.npz"))]
ETQ = {"pura": "Pure H$_2$O", "nacl25": "2.5% NaCl in H$_2$O", "nacl5": "5% NaCl in H$_2$O", "nacl10": "10% NaCl in H$_2$O"}
COL = {"pura": "#e41a1c", "nacl25": "#009e73", "nacl5": "#7b1fa2", "nacl10": "#e69f00"}
ref = pd.read_csv(os.path.join(R, "datos", "fig5_sidelnik_digitalizada.csv"))
W = 4                                            # bines de 5 keV agrupados de a 4 = 20 keV
# bordes Compton E_c = E (2E/m)/(1+2E/m) de las lineas principales
LINEAS = {"2,223 (H)": 2.2232, "6,13 (O) / 6,11 (Cl)": 6.12, "7,41 (Cl)": 7.4141, "7,79 (Cl)": 7.7901, "8,58 (Cl)": 8.5785}
borde = lambda E: E * (2 * E / 0.511) / (1 + 2 * E / 0.511)
esp = {}
for m in MED:
    z = np.load(os.path.join(RES, f"{m}.npz")); N = int(z["N"]); GB = z["GB"]
    h = (z["H_egamlin|compt|e-"] + z["H_egamlin|phot|e-"] + z["H_egamlin|conv|e-"]).astype(float) * 150000 / N
    h = h[:len(h) // W * W].reshape(-1, W).sum(1); c = (np.arange(len(h)) + 0.5) * 0.005 * W
    esp[m] = (c, h, N)
def nivel(c, h, E, w=0.1):
    s = np.abs(c - E) < w; return float(np.mean(h[s]))
def nivel_ref(m, E, w=0.1):
    s = (ref.E_MeV - E).abs() < w; return float(np.nanmedian(ref[f"{m}_mediana"][s]))
EV = [0.5, 1.0, 1.5, 1.9, 2.1, 3.0, 4.0, 5.0, 5.8, 6.1, 7.0, 7.4, 7.7, 8.0, 9.0]
out = {"ancho_bin_keV": 5 * W, "niveles": {}, "cociente_Sidelnik_MEIGA": {}, "NaCl_sobre_pura": {}, "saltos_bordes": {}}
for m in MED:
    c, h, N = esp[m]
    out["niveles"][m] = {str(E): {"MEIGA": nivel(c, h, E), "Sidelnik": nivel_ref(m, E)} for E in EV}
    rr = [out["niveles"][m][str(E)]["Sidelnik"] / out["niveles"][m][str(E)]["MEIGA"] for E in EV if E <= 7.0]
    out["cociente_Sidelnik_MEIGA"][m] = {"mediana": float(np.nanmedian(rr)), "min": float(np.nanmin(rr)), "max": float(np.nanmax(rr))}
    # salto en cada borde Compton: nivel 0,10-0,04 MeV antes / 0,04-0,10 MeV despues
    out["saltos_bordes"][m] = {}
    for k, E in LINEAS.items():
        Ec = borde(E)
        a = np.mean(h[(c > Ec - 0.12) & (c < Ec - 0.03)]); b = np.mean(h[(c > Ec + 0.03) & (c < Ec + 0.12)])
        sa = ref[(ref.E_MeV > Ec - 0.12) & (ref.E_MeV < Ec - 0.03)][f"{m}_mediana"].median()
        sb = ref[(ref.E_MeV > Ec + 0.03) & (ref.E_MeV < Ec + 0.12)][f"{m}_mediana"].median()
        out["saltos_bordes"][m][k] = {"borde_MeV": Ec, "MEIGA": float(a / b) if b > 0 else None,
                                      "Sidelnik": float(sa / sb) if np.isfinite(sa) and np.isfinite(sb) and sb > 0 else None}
for m in MED[1:]:
    out["NaCl_sobre_pura"][m] = {str(E): {"MEIGA": out["niveles"][m][str(E)]["MEIGA"] / out["niveles"]["pura"][str(E)]["MEIGA"],
                                          "Sidelnik": nivel_ref(m, E) / nivel_ref("pura", E)} for E in EV}
json.dump(out, open(os.path.join(RES, "espectro_electrones_tipo_sidelnik.json"), "w"), indent=1)

estilo.aplicar()
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
fig, axs = plt.subplots(2, 2, figsize=(8, 10.8))
a = axs[0, 0]
for m in MED:
    c, h, N = esp[m]; s = h > 0
    a.plot(c[s], h[s], "-", color=COL[m], lw=0.9)
a.set_title(f"(a) MEIGA, formato de la Fig. 5\n(intervalos de {5 * W} keV, 150 000 n)")
a = axs[0, 1]
fac = []
for m in MED:
    c, h, N = esp[m]; k = out["cociente_Sidelnik_MEIGA"][m]["mediana"]; s = h > 0
    a.plot(c[s], h[s] * k, "-", color=COL[m], lw=0.8, alpha=0.8)
    a.plot(ref.E_MeV, ref[f"{m}_mediana"], "o", color=COL[m], mec="#0b0b0b", mew=0.3, ms=2.2)
    fac.append(f"{k:.2f}".replace(".", ","))
for k_, E in LINEAS.items():
    a.axvline(borde(E), color="#8a8984", ls=":", lw=0.8)
a.text(0.97, 0.97, "factor MEIGA:\n" + " / ".join(fac), transform=a.transAxes, ha="right", va="top", fontsize=11)
a.set_title("(b) MEIGA × factor frente a Sidelnik\n(punteadas: bordes Compton)")
for a in axs[0]:
    a.set_yscale("log"); a.set_ylim(10, 1e6); a.set_xlim(0, 10); a.set_xlabel("Energía [MeV]"); a.set_ylabel("Cuentas")
a = axs[1, 0]
for m in MED[1:]:
    c, h, N = esp[m]; c0, h0, _ = esp["pura"]
    hh = lambda x, y: np.convolve(y, np.ones(10) / 10, "same")
    a.plot(c, hh(c, h) / np.maximum(hh(c0, h0), 1e-9), "-", color=COL[m], lw=1.2)
    rp = np.interp(ref.E_MeV, ref.E_MeV[ref.pura_mediana.notna()], ref.pura_mediana.dropna())
    a.plot(ref.E_MeV, ref[f"{m}_mediana"] / rp, "o", color=COL[m], mec="#0b0b0b", mew=0.3, ms=2.5)
a.axhline(1, color="#0b0b0b", lw=0.8); a.set_ylim(0.5, 4); a.set_xlim(0, 10)
a.set_xlabel("Energía [MeV]"); a.set_ylabel("NaCl / agua pura")
a.set_title("(c) Efecto del NaCl\n(no depende de la normalización)")
a = axs[1, 1]
sb0 = out["saltos_bordes"]["pura"]; claves = list(sb0)
nb = 2 * len(MED); w_ = 0.84 / nb
for i, m in enumerate(MED):
    sb = out["saltos_bordes"][m]
    for j, k_ in enumerate(claves):
        if not sb[k_]["MEIGA"] or (m == "pura" and "Cl" in k_ and "O" not in k_):
            continue
        x0 = j - 0.42 + (2 * i + 0.5) * w_
        a.bar(x0, sb[k_]["MEIGA"], w_, color=COL[m])
        a.bar(x0 + w_, sb[k_]["Sidelnik"] or 0, w_, color=COL[m], alpha=0.4, hatch="//", edgecolor="#0b0b0b", lw=0.3)
a.set_xticks(np.arange(len(claves))); a.set_xticklabels([f"{sb0[k_]['borde_MeV']:.2f}".replace(".", ",") for k_ in claves]); a.set_xlabel("Borde Compton [MeV]")
a.axhline(1, color="#0b0b0b", lw=0.8); a.set_ylabel("Nivel antes / después"); a.set_ylim(0, 9.5)
a.set_title("(d) Saltos en los bordes Compton\n(líneas de 2,22; 6,1; 7,41; 7,79; 8,58 MeV)")
H = [Patch(color=COL[m], label={"pura": "Agua pura", "nacl25": "Agua + 2.5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}[m]) for m in MED] + \
    [Line2D([], [], color="#0b0b0b", lw=1.2, label="MEIGA (líneas, barras llenas)"),
     Line2D([], [], color="#0b0b0b", marker="o", ls="", ms=4, label="Sidelnik (puntos, barras rayadas)")]
fig.legend(handles=H, loc="lower center", ncol=3, bbox_to_anchor=(0.5, 0.0), columnspacing=1.2, handlelength=1.6)
fig.tight_layout(rect=(0, 0.06, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig13_espectro_electrones_tipo_sidelnik.{e}"))
print(json.dumps({k: out[k] for k in ("cociente_Sidelnik_MEIGA", "saltos_bordes")}, indent=1))
for m, v in out["NaCl_sobre_pura"].items():
    print(m, {E: (round(x["MEIGA"], 2), round(x["Sidelnik"], 2)) for E, x in v.items()})

In [ ]:
mostrar("fig13_espectro_electrones_tipo_sidelnik")

### Interpretación física: origen del gamma de cada electrón

`scripts/interpretacion_electrones.py` — Interpretacion fisica de las diferencias con la Fig. 5 de Sidelnik usando la descomposicion por origen (resultados/<medio>_origen_electrones.npz, de origen_electrones.py):  - salto en el borde Compton de 1,994 MeV: J = (C_H + F_antes) / F_despues, con C_H = electrones de la captura en H    y F = fondo de gammas de otros origenes (sobre todo dispersion inelastica de neutrones en O);  - factor k por el que habria que dividir el fondo de MEIGA para obtener el salto de Sidelnik;  - cociente NaCl/agua pura en 2-6 MeV con el fondo de MEIGA y con el fondo dividido por k. Escribe resultados/interpretacion_electrones.json y figuras/fig14_origen_electrones.{png,pdf}

In [ ]:
sys.argv = ['interpretacion_electrones.py']
__file__ = str(DIR_SCRIPTS / "interpretacion_electrones.py")
# ---- código de scripts/interpretacion_electrones.py ----
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
GR = [("captura H", "#2a78d6"), ("captura Cl", "#1baf7a"), ("captura otra", "#8fd3b6"), ("inelástica n", "#eda100"),
      ("otros hadrónicos", "#8a8984"), ("bremsstrahlung", "#e34948"), ("aniquilación", "#f4a3a3"), ("decaimiento radiactivo", "#4a3aa7")]
EL = json.load(open(os.path.join(RES, "espectro_electrones_tipo_sidelnik.json")))
D = {}
for m in ["pura", "nacl25"]:
    z = np.load(os.path.join(RES, f"{m}_origen_electrones.npz")); N = int(z["N"]); EB = z["EB"]
    D[m] = {g: z[f"h|{g}|todos"] * 150000 / N for g, _ in GR if f"h|{g}|todos" in z.files}
    D[m]["_sp"] = {g: z[f"h|{g}|sin_piones"] * 150000 / N for g, _ in GR if f"h|{g}|sin_piones" in z.files}
c = 0.5 * (EB[1:] + EB[:-1])
w = lambda h, a, b: float(h[(c > a) & (c < b)].mean())
CAP = ("captura H", "captura Cl", "captura otra")
def partes(m, a, b, sp=False):
    d = D[m]["_sp"] if sp else D[m]
    gs = [g for g in d if not g.startswith("_")]
    return sum(w(d[g], a, b) for g in gs if g in CAP), sum(w(d[g], a, b) for g in gs if g not in CAP)
out = {"composicion": {}}
for m in D:
    A = {g: w(D[m][g], 1.874, 1.964) for g in D[m] if g != "_sp"}; B = {g: w(D[m][g], 2.024, 2.114) for g in D[m] if g != "_sp"}
    out["composicion"][m] = {"antes_borde": A, "despues_borde": B,
                             "frac_inelastica_despues": B["inelástica n"] / sum(B.values()),
                             "salto": sum(A.values()) / sum(B.values())}
    ca, fa = partes(m, 1.874, 1.964); cb, fb = partes(m, 2.024, 2.114)
    cas, fas = partes(m, 1.874, 1.964, True); cbs, fbs = partes(m, 2.024, 2.114, True)
    out["composicion"][m]["salto_sin_piones"] = (cas + fas) / (cbs + fbs)
    Js = EL["saltos_bordes"][m]["2,223 (H)"]["Sidelnik"]
    # (ca + fa/k) / (cb + fb/k) = Js  ->  k
    k = (fa - Js * fb) / (Js * cb - ca)
    out["composicion"][m]["salto_Sidelnik"] = Js; out["composicion"][m]["k_fondo"] = k
kp = out["composicion"]["pura"]["k_fondo"]
out["NaCl_sobre_pura"] = {}
for a, b in [(0.5, 1.8), (2.05, 4.0), (4.0, 5.85), (7.0, 8.3)]:
    cp, fp = partes("pura", a, b); cn, fn = partes("nacl25", a, b)
    sid = [EL["NaCl_sobre_pura"]["nacl25"][e]["Sidelnik"] for e in EL["NaCl_sobre_pura"]["nacl25"] if a <= float(e) <= b]
    out["NaCl_sobre_pura"][f"{a}-{b}"] = {"MEIGA": (cn + fn) / (cp + fp), "MEIGA_fondo_entre_k": (cn + fn / kp) / (cp + fp / kp),
                                         "Sidelnik": float(np.median(sid)) if sid else None}
json.dump(out, open(os.path.join(RES, "interpretacion_electrones.json"), "w"), indent=1, ensure_ascii=False)

plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "savefig.dpi": 160})
estilo.aplicar()
fig, axs = estilo.tres_paneles(10.2)
for a, m, t in [(axs[0], "pura", "(a) Agua pura"), (axs[1], "nacl25", "(b) Agua + 2,5 % NaCl")]:
    base = np.zeros_like(c)
    for g, col in GR:
        if g in D[m] and D[m][g].sum() > 0:
            a.fill_between(c, base, base + D[m][g], step="mid", color=col, lw=0, label=g); base = base + D[m][g]
    a.set_yscale("log"); a.set_ylim(10, 1e6); a.set_xlim(0, 10); a.set_xlabel("Energía del electrón [MeV]"); a.set_ylabel("Cuentas / 20 keV")
    a.set_title(f"{t}:\norigen del gamma del electrón", fontsize=10.5)
    for E in (1.994, 5.875, 8.33):
        a.axvline(E, color="#0b0b0b", ls=":", lw=0.7)
a = axs[2]
lab = list(out["NaCl_sobre_pura"]); x = np.arange(len(lab))
for i, (key, col, nom) in enumerate([("Sidelnik", "#0b0b0b", "Sidelnik"), ("MEIGA", "#1baf7a", "MEIGA"),
                                      ("MEIGA_fondo_entre_k", "#8fd3b6", f"MEIGA con fondo / {kp:.2f}")]):
    v = [out["NaCl_sobre_pura"][l][key] or 0 for l in lab]
    a.bar(x + (i - 1) * 0.27, v, 0.27, color=col, label=nom)
    for xx, vv in zip(x + (i - 1) * 0.27, v):
        a.text(xx, vv + 0.04, f"{vv:.2f}".replace(".", ","), ha="center", fontsize=10.5)
a.set_xticks(x); a.set_xticklabels([l.replace("-", "–").replace(".", ",") + " MeV" for l in lab], fontsize=11); a.axhline(1, color="#0b0b0b", lw=0.8)
a.set_ylabel("Electrones NaCl / agua pura"); a.set_ylim(0, 1.25 * a.get_ylim()[1]); a.legend(fontsize=9, ncol=3, loc="upper left"); a.set_title("(c) Cociente 2,5 % NaCl / agua pura y efecto del fondo", fontsize=10.5)
fig.legend(*axs[1].get_legend_handles_labels(), loc="lower center", ncol=4, frameon=False, columnspacing=1.0, handlelength=1.4, title="origen del gamma (a, b):", title_fontsize=11.5)
fig.tight_layout(rect=(0, 0.10, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig14_origen_electrones.{e}"))
print(json.dumps({m: {k_: v for k_, v in d.items() if k_ not in ("antes_borde", "despues_borde")} for m, d in out["composicion"].items()}, indent=1, ensure_ascii=False))
print(json.dumps(out["NaCl_sobre_pura"], indent=1))

In [ ]:
mostrar("fig14_origen_electrones")

## 5. Luz Cherenkov y carga (Fig. 6 de Sidelnik)

### Fotones aceptados por la eficiencia cuántica y carga en el PMT

`scripts/histograma_tipo_sidelnik.py` — Histogramas al estilo de la Fig. 6 de Sidelnik et al. (2020).  (a) 'Fotones aceptados por la eficiencia cuantica': por neutron, fotones Cherenkov que pasan la QE del     R5912 al crearse (en el modo eFast de MEIGA son exactamente los fotones que se transportan),     SIN transporte optico. Es lo que grafica Sidelnik. (b) Carga en el PMT (fotoelectrones) despues del transporte optico: el aporte de este trabajo. Solo eventos con >= 1 foton / p.e. (la Fig. 6 no muestra el pico en cero). Ambos se comparan con la curva de agua pura de la Fig. 6 digitalizada. Uso: histograma_tipo_sidelnik.py [medio ...]   (por defecto los medios con <medio>_fotones.npz)

In [ ]:
sys.argv = ['histograma_tipo_sidelnik.py']
__file__ = str(DIR_SCRIPTS / "histograma_tipo_sidelnik.py")
# ---- código de scripts/histograma_tipo_sidelnik.py ----
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo

R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MED = [m for m in (sys.argv[1:] or ["pura", "nacl25", "nacl5", "nacl10"]) if (os.path.exists(os.path.join(RES, f"{m}_aceptados_por_run.npy")) or os.path.exists(os.path.join(RES, f"{m}_fotones.npz")))]
def aceptados_por_run(m):
    """Fotones aceptados por la QE (= transportados en eFast) por neutron. Usa resultados/<m>_aceptados_por_run.npy
    (pequeno, va en el repositorio); si no existe lo calcula una vez desde <m>_fotones.npz (destino_fotones.py)."""
    f = os.path.join(RES, f"{m}_aceptados_por_run.npy")
    if not os.path.exists(f):
        N = len(np.load(os.path.join(RES, f"{m}.npz"))["Q"])
        np.save(f, np.bincount(np.load(os.path.join(RES, f"{m}_fotones.npz"))["run"], minlength=N).astype(np.int32))
    return np.load(f)


ETQ = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
COL = {"pura": "#e34948", "nacl25": "#1baf7a", "nacl5": "#7b1fa2", "nacl10": "#eda100"}
COLREF = {"pura": "Pure_H2O_counts_digitized", "nacl25": "NaCl_2p5_counts_digitized",
          "nacl5": "NaCl_5_counts_digitized", "nacl10": "NaCl_10_counts_digitized"}
REF6 = os.path.join(R, "datos", "fig6_sidelnik_digitalizada_700pts.csv")   # Fig. 6 de Sidelnik digitalizada (700 puntos)
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "savefig.dpi": 160})
ref = pd.read_csv(REF6)
xb = np.arange(0, 1010, 10); xc = xb[:-1] + 5
res = {}; tablas = {}
estilo.aplicar()
fig, axs = plt.subplots(2, 2, figsize=(8, 10.4))
for m in MED:
    p = np.load(os.path.join(RES, f"{m}.npz")); N = len(p["Q"])
    acept = aceptados_por_run(m)                     # fotones aceptados por la QE (antes del transporte)
    q = p["Q"]                                       # fotoelectrones en el PMT (tras el transporte)
    ha = np.histogram(acept[acept >= 1], xb)[0]; hq = np.histogram(q[q >= 1], xb)[0]   # como la Fig. 6: sin eventos vacios
    yr = np.interp(xc, ref.Numero_de_fotones, ref[COLREF[m]])
    def forma(h):
        return h / h.sum()
    # correlacion de la forma en escala log, intervalo 0-1000
    corr = lambda a, b: float(np.corrcoef(np.log(forma(a) + 1e-9), np.log(forma(b) + 1e-9))[0, 1])
    res[m] = {"N": N,
              "aceptados_QE": {"media": float(acept.mean()), "mediana": float(np.median(acept)), "frac_0": float((acept == 0).mean()),
                               "frac_100_600": float(((acept >= 100) & (acept < 600)).mean()), "frac_ge600": float((acept >= 600).mean()),
                               "corr_forma_Sidelnik_0_1000": corr(ha, yr)},
              "carga_pe": {"media": float(q.mean()), "mediana": float(np.median(q)), "frac_0": float((q == 0).mean()),
                           "frac_ge1": float((q >= 1).mean()), "frac_ge100": float((q >= 100).mean()),
                           "corr_forma_Sidelnik_0_1000": corr(hq, yr)},
              "Sidelnik_media_0_1000": float(np.average(xc, weights=yr)),
              "MEIGA_aceptados_media_0_1000": float(np.average(xc, weights=ha)), "MEIGA_carga_media_0_1000": float(np.average(xc, weights=hq))}
    tablas[m] = pd.DataFrame({"bin_inf": xb[:-1], "bin_sup": xb[1:], "fotones_aceptados_QE": ha, "fotoelectrones_PMT": hq,
                              "Sidelnik_digitalizado": yr})
    tablas[m].to_csv(os.path.join(RES, f"histogramas_tipo_sidelnik_{m}.csv"), index=False)
    # (a) cuentas absolutas, como la Fig. 6; (b) normalizado; (c) carga en el PMT; (d) efecto del transporte optico
    axs[0, 0].step(xc, ha, where="mid", color=COL[m], lw=1.6)
    axs[0, 0].plot(xc, yr * ha.sum() / yr.sum(), "--", color="#0b0b0b", lw=1.1)
    axs[0, 1].step(xc, forma(ha), where="mid", color=COL[m], lw=1.6)
    axs[0, 1].plot(xc, forma(yr), "--", color="#0b0b0b", lw=1.1)
    axs[1, 0].step(xc, hq, where="mid", color=COL[m], lw=1.6)
    axs[1, 1].step(xc, forma(ha), where="mid", color=COL[m], lw=1.3, alpha=0.45)
    axs[1, 1].step(xc, forma(hq), where="mid", color=COL[m], lw=1.8)
    axs[1, 1].plot(xc, forma(yr), "--", color="#0b0b0b", lw=1.1)
axs[0, 0].set_title("(a) Fotones aceptados por la QE\n(como la Fig. 6 de Sidelnik)", fontsize=10.5)
axs[0, 0].set_xlabel("Fotones aceptados por la QE"); axs[0, 0].set_ylabel("Cuentas / 10 fotones")
axs[0, 1].set_title("(b) Igual, normalizado\n(escala logarítmica)", fontsize=10.5); axs[0, 1].set_yscale("log")
axs[0, 1].set_xlabel("Fotones aceptados por la QE"); axs[0, 1].set_ylabel("Fracción / 10 fotones")
axs[1, 0].set_title("(c) Aporte: carga en el PMT\n(con transporte óptico)", fontsize=10.5)
axs[1, 0].set_xlabel("Fotoelectrones en el PMT"); axs[1, 0].set_ylabel("Cuentas / 10 p.e."); axs[1, 0].set_yscale("log")
axs[1, 1].set_title("(d) Efecto del transporte óptico", fontsize=10.5); axs[1, 1].set_yscale("log")
axs[1, 1].set_xlabel("Fotones o fotoelectrones"); axs[1, 1].set_ylabel("Fracción / 10")
for a in axs.flat:
    a.set_xlim(0, 1000)
from matplotlib.lines import Line2D
h_ = [Line2D([], [], color=COL[m], lw=1.8, label=f"MEIGA, {ETQ[m]}" + ("" if res[m]["N"] == 150000 else f" ({res[m]['N']} n)")) for m in MED]
h_ += [Line2D([], [], color="#0b0b0b", ls="--", lw=1.1, label="Sidelnik, Fig. 6 (en (a), área igualada)"),
       Line2D([], [], color="#8a8984", lw=1.3, alpha=0.45, label="(d) trazo claro: aceptados por la QE; oscuro: p.e. en el PMT")]
fig.legend(handles=h_, loc="lower center", ncol=2, fontsize=9)
fig.tight_layout(rect=(0, 0.115, 1, 1))
fig.savefig(os.path.join(FIG, "fig09_histograma_tipo_sidelnik.png")); fig.savefig(os.path.join(FIG, "fig09_histograma_tipo_sidelnik.pdf"))
json.dump(res, open(os.path.join(RES, "histograma_tipo_sidelnik.json"), "w"), indent=1, ensure_ascii=False)
print(json.dumps(res, indent=1, ensure_ascii=False))

In [ ]:
mostrar("fig09_histograma_tipo_sidelnik")

### Fig. 6 de Sidelnik reproducida, con la separación de eventos con piones

`scripts/figura6_sidelnik_meiga.py` — Reproduce la Fig. 6 de Sidelnik et al. (2020) con los datos de MEIGA, en su mismo formato: fotones Cherenkov aceptados por la QE por neutron (antes del transporte optico), cuentas absolutas por bin de 10 fotones, 0-2500, escala log. Superpone la curva digitalizada de Sidelnik sin reescalar. Uso: figura6_sidelnik_meiga.py [medio ...]

In [ ]:
sys.argv = ['figura6_sidelnik_meiga.py']
__file__ = str(DIR_SCRIPTS / "figura6_sidelnik_meiga.py")
# ---- código de scripts/figura6_sidelnik_meiga.py ----
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo

R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
TODOS = ["pura", "nacl25", "nacl5", "nacl10"]
MED = [m for m in (sys.argv[1:] or TODOS) if (os.path.exists(os.path.join(RES, f"{m}_aceptados_por_run.npy")) or os.path.exists(os.path.join(RES, f"{m}_fotones.npz")))]
def aceptados_por_run(m):
    """Fotones aceptados por la QE (= transportados en eFast) por neutron. Usa resultados/<m>_aceptados_por_run.npy
    (pequeno, va en el repositorio); si no existe lo calcula una vez desde <m>_fotones.npz (destino_fotones.py)."""
    f = os.path.join(RES, f"{m}_aceptados_por_run.npy")
    if not os.path.exists(f):
        N = len(np.load(os.path.join(RES, f"{m}.npz"))["Q"])
        np.save(f, np.bincount(np.load(os.path.join(RES, f"{m}_fotones.npz"))["run"], minlength=N).astype(np.int32))
    return np.load(f)


ETQ = {"pura": "Pure H$_2$O", "nacl25": "2.5% NaCl in H$_2$O", "nacl5": "5% NaCl in H$_2$O", "nacl10": "10% NaCl in H$_2$O"}
COL = {"pura": "#e41a1c", "nacl25": "#009e73", "nacl5": "#7b1fa2", "nacl10": "#e69f00"}
EST = {"pura": "-", "nacl25": (0, (8, 4)), "nacl5": (0, (6, 2, 1, 2, 1, 2)), "nacl10": (0, (1, 3))}
COLREF = {"pura": "Pure_H2O_counts_digitized", "nacl25": "NaCl_2p5_counts_digitized",
          "nacl5": "NaCl_5_counts_digitized", "nacl10": "NaCl_10_counts_digitized"}
REF6 = os.path.join(R, "datos", "fig6_sidelnik_digitalizada_700pts.csv")   # Fig. 6 de Sidelnik digitalizada (700 puntos)
ref = pd.read_csv(REF6)
# artefactos de la digitalizacion: la leyenda capturada (1590-1800) y el borde final en las curvas NaCl
for c in list(COLREF.values())[1:]:
    mal = ((ref.Numero_de_fotones.between(1590, 1800)) & (ref[c] > 40)) | (ref.Numero_de_fotones > 2470)
    ref.loc[mal, c] = np.nan
DATOS = os.environ.get("CORRIDAS_SIDELNIK", "")   # carpeta con las corridas (solo para separar los eventos con piones)
CARP = {"pura": "150000N-500MeV-Sidelnik-H2O-pura-sinSab-20261012-061022",
        "nacl25": ["42649N-parcial-500MeV-Sidelnik-H2O-2.5NaCl-sinSab-20261013", "107351N-500MeV-Sidelnik-H2O-2.5NaCl-sinSab-20261013-135213"]}
xb = np.arange(0, 2510, 10); xc = xb[:-1] + 5
estilo.aplicar()
fig, axs = plt.subplots(1, 2, figsize=(8, 6.0), sharey=True)
filas = []; PI = {}
for m in TODOS:
    axs[0].plot(ref.Numero_de_fotones, ref[COLREF[m]], color=COL[m], ls=EST[m], lw=1.8, label=ETQ[m])
for m in MED:
    a = aceptados_por_run(m)
    k = 150000 / len(a)        # corridas parciales: cuentas escaladas a 150000 neutrones, como en el articulo
    h = np.histogram(a[a >= 1], xb)[0] * k
    axs[1].plot(xc, np.where(h > 0, h, np.nan), color=COL[m], ls=EST[m], lw=1.8,
                label=f"MEIGA {ETQ[m]}" + ("" if k == 1 else f" (×{k:.2f})".replace(".", ",")))
    f = os.path.join(DATOS, CARP.get(m, "-") if isinstance(CARP.get(m, "-"), str) else "-", "Datos-simulacion", "secundarias-inelastica-neutron.tsv")
    if m == "pura" and os.path.exists(f):  # separa eventos con produccion de piones en una inelastica
        sec = pd.read_csv(f, sep="\t", skiprows=1, header=None, usecols=[0, 7], names=["run", "part"], on_bad_lines="skip")
        sec = sec[pd.to_numeric(sec.run, errors="coerce") < len(a)]; sec["run"] = sec.run.astype(int)
        pic = np.zeros(len(a), bool); pic[sec.run[sec.part.isin(["pi+", "pi-"])].unique()] = True
        pi0 = np.zeros(len(a), bool); pi0[sec.run[sec.part == "pi0"].unique()] = True
        pi = pic | pi0
        PI[m] = {"frac_eventos_con_piones": float(pi.mean()), "frac_con_pi_cargado": float(pic.mean()), "frac_con_pi0": float(pi0.mean()),
                 "frac_ge500_todos": float((a >= 500).mean()), "frac_ge500_sin_piones": float((a[~pi] >= 500).mean()),
                 "frac_ge500_con_piones_entre_ge500": float(pi[a >= 500].mean()), "media_aceptados_con_piones": float(a[pi].mean()),
                 "media_aceptados_sin_piones": float(a[~pi].mean()), "intervalos": []}
        for lo, hi in [(1, 100), (100, 500), (500, 1000), (1000, 2500), (2500, None)]:
            sl = (a >= lo) & (a < (hi or 10**9))
            PI[m]["intervalos"].append({"intervalo": f"{lo}-{hi}" if hi else f">={lo}", "n": int(sl.sum()),
                                        "pi_cargado": float(pic[sl].mean()), "pi0": float(pi0[sl].mean()), "algun_pion": float(pi[sl].mean())})
        np.savez(os.path.join(RES, f"piones_hist_{m}.npz"), con=np.histogram(a[pi & (a >= 1)], xb)[0], sin=np.histogram(a[~pi & (a >= 1)], xb)[0],
                 frac=pi.mean())
        filas.append({"medio": m, "intervalo": ">=500 con piones (%)", "MEIGA_cuentas": round(100 * pi[a >= 500].mean(), 1), "Sidelnik_cuentas": None, "cociente": None})
    fp = os.path.join(RES, f"piones_hist_{m}.npz")
    if os.path.exists(fp):                  # histogramas guardados: la figura no necesita las corridas
        P_ = np.load(fp)
        for key, nom, ls in [("con", "con piones", ":"), ("sin", "sin piones", "-.")]:
            fr = float(P_["frac"]) if key == "con" else 1 - float(P_["frac"])
            axs[1].plot(xc, np.where(P_[key] > 0, P_[key], np.nan), color="#555555", ls=ls, lw=1.2, label=f"{ETQ[m]}: eventos {nom} ({100 * fr:.1f} %)".replace(".", ","))
    if m == "pura":
        axs[1].plot(ref.Numero_de_fotones, ref[COLREF[m]], color="#0b0b0b", lw=1.0, alpha=0.6, label="Sidelnik, Pure H$_2$O")
    ok = ref[COLREF[m]].notna(); yr = np.interp(xc, ref.Numero_de_fotones[ok], ref[COLREF[m]][ok])   # sin los artefactos enmascarados
    for x0, x1 in [(0, 100), (100, 500), (500, 1000), (1000, 2500)]:
        s = (xc >= x0) & (xc < x1)
        filas.append({"medio": m, "intervalo": f"{x0}-{x1}", "MEIGA_cuentas": round(h[s].sum()),
                      "Sidelnik_cuentas": round(float(yr[s].sum())), "cociente": round(h[s].sum() / yr[s].sum(), 2)})
    filas.append({"medio": m, "intervalo": ">2500", "MEIGA_cuentas": round((a > 2500).sum() * k), "Sidelnik_cuentas": None, "cociente": None})
for a, t in zip(axs, ["(a) Sidelnik et al. (2020), Fig. 6", "(b) MEIGA (cuentas para 150 000 n)"]):
    a.set_yscale("log"); a.set_ylim(1, 1e5); a.set_xlim(0, 2500); a.set_title(t, fontsize=10.5)
    a.set_xlabel("Fotones aceptados por la QE"); a.grid(alpha=0.3)
axs[0].set_ylabel("Cuentas / 10 fotones")
from matplotlib.lines import Line2D
ETQ_ES = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
H_ = [Line2D([], [], color=COL[m], ls=EST[m], lw=2, label=ETQ_ES[m]) for m in TODOS]
H_ += [l for l in axs[1].get_lines() if l.get_label().startswith(("Pure", "Sidelnik"))]
for l in H_[len(TODOS):]:
    l.set_label(l.get_label().replace("Pure H$_2$O: eventos", "(b) agua pura, eventos").replace("Sidelnik, Pure H$_2$O", "(b) Sidelnik, agua pura"))
fig.legend(handles=H_, loc="lower center", ncol=2, frameon=False, columnspacing=1.5, handlelength=2.2)
fig.tight_layout(rect=(0, 0.2, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig10_figura6_sidelnik_vs_meiga.{e}"), dpi=160)
if PI:                                   # sin las corridas se conserva el resultado guardado
    json.dump(PI, open(os.path.join(RES, "piones_cola.json"), "w"), indent=1)
t = pd.DataFrame(filas); t.to_csv(os.path.join(RES, "figura6_cuentas_por_intervalo.csv"), index=False); print(t.to_string(index=False))

In [ ]:
mostrar("fig10_figura6_sidelnik_vs_meiga")

### Diagnóstico óptico (requiere `resultados/<medio>_fotones.npz`)

`diagnostico_carga.py` (balance de luz, destino de los fotones) y `sensibilidad_tyvek.py` (modelo de rebotes en el Tyvek) usan el destino de cada
fotón óptico; si el archivo no está, se muestran las figuras ya guardadas.

In [ ]:
for m in MEDIOS:
    if (DIR_RES / f"{m}_fotones.npz").exists():
        ejecutar("diagnostico_carga.py", m)
        if m == "pura":
            ejecutar("sensibilidad_tyvek.py", m)
    if (DIR_RES / f"sensibilidad_tyvek_{m}.json").exists() and ((DIR_RES / f"tyvek_espectro_{m}.json").exists() or (DIR_RES / f"{m}_fotones.npz").exists()):
        ejecutar("tyvek_diagnostico.py", m)      # el estudio del Tyvek se hizo con agua pura
    else:
        print(f"{m}: falta {m}_fotones.npz; se usan los resultados guardados en resultados/diagnostico_{m}.json")
ejecutar("figura_destino_fotones_medios.py") if all((DIR_RES / f"{m}_fotones.npz").exists() for m in MEDIOS) else None
for f in ["fig06_balance_luz_y_fig6", "fig07_destino_fotones", "fig08_tyvek_diagnostico"]:
    if (DIR_FIG / f"{f}.png").exists():
        mostrar(f)

## 6. Medios con NaCl: captura por núcleo, cascadas del cloro y deuterones

### Cascadas de las capturas en ³⁵Cl, ²³Na y ¹H

`scripts/cascadas_cloro.py` — Cascadas de gammas de las capturas en 35Cl, 23Na y 1H (secundarias de nCapture guardadas en <medio>.npz). Compara la energia total emitida con el valor Q de la captura (35Cl: 8,579 MeV; 23Na: 6,959; 1H: 2,224). Escribe resultados/cascadas_cloro_<medio>.json y figuras/fig12_cascadas_cloro_<medio>.{png,pdf}. Uso: cascadas_cloro.py [medio]

In [ ]:
sys.argv = ['cascadas_cloro.py', "nacl25"]
__file__ = str(DIR_SCRIPTS / "cascadas_cloro.py")
# ---- código de scripts/cascadas_cloro.py ----
import os, sys, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
m = sys.argv[1] if len(sys.argv) > 1 else "nacl25"
z = np.load(os.path.join(R, "resultados", f"{m}.npz"))
cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "A": z["cap_target_A"]})
sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"]})
g = sc[sc.p == "gamma"].groupby(["run", "trk"]).agg(n=("E", "size"), S=("E", "sum"), Emax=("E", "max"))
c = cap.merge(g, left_on=["run", "trk"], right_index=True, how="left").fillna({"n": 0, "S": 0, "Emax": 0})
QV = {(1, 1): 2.2246, (11, 23): 6.9594, (17, 35): 8.5794}
out = {}
for (Z, A), q in QV.items():
    d = c[(c.Z == Z) & (c.A == A)]
    if len(d) < 20:
        continue
    out[f"Z{Z}A{A}"] = {"capturas": int(len(d)), "Q_MeV": q, "gammas_por_captura": float(d.n.mean()),
                        "suma_media_MeV": float(d.S.mean()), "suma_p5_p50_p95": np.percentile(d.S, [5, 50, 95]).tolist(),
                        "frac_suma_igual_Q_1pct": float((np.abs(d.S - q) < 0.01 * q).mean()),
                        "frac_suma_mayor_Q": float((d.S > 1.01 * q).mean()), "frac_suma_menor_Q": float((d.S < 0.99 * q).mean()),
                        "frac_un_solo_gamma": float((d.n == 1).mean()),
                        "frac_con_gamma_8579": float(((d.Emax > 8.57) & (d.Emax < 8.59)).mean()) if Z == 17 else None,
                        "n_gammas": {int(k): int(v) for k, v in d.n.value_counts().sort_index().items()}}
json.dump(out, open(os.path.join(R, "resultados", f"cascadas_cloro_{m}.json"), "w"), indent=1)
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.color": "#e4e3df", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "savefig.dpi": 160})
estilo.aplicar()
fig, axs = plt.subplots(1, 2, figsize=(8, 4.3))
for (Z, A), col, et in [((17, 35), "#1baf7a", "$^{35}$Cl (Q = 8,58 MeV)"), ((11, 23), "#4a3aa7", "$^{23}$Na (Q = 6,96 MeV)"), ((1, 1), "#2a78d6", "$^{1}$H (Q = 2,22 MeV)")]:
    d = c[(c.Z == Z) & (c.A == A)]
    if len(d) < 20:
        continue
    axs[0].hist(d.S, bins=np.arange(0, 25, 0.1), histtype="step", density=True, color=col, lw=1.4, label=et)
    axs[0].axvline(QV[(Z, A)], color=col, ls=":", lw=1)
    vc = d.n.value_counts(normalize=True).sort_index()
    axs[1].plot(vc.index, vc.values, "o-", color=col, label=et)
axs[0].set_yscale("log"); axs[0].set_xlabel("Energía total de los gammas [MeV]"); axs[0].set_ylabel("Densidad")
axs[0].set_title("(a) Energía emitida por captura\n(punteada: valor Q)", fontsize=10.5)
axs[1].set_xlabel("Número de gammas por captura"); axs[1].set_ylabel("Fracción de capturas")
axs[1].set_title("(b) Multiplicidad de la cascada", fontsize=10.5)
fig.legend(*axs[1].get_legend_handles_labels(), loc="lower center", ncol=3, frameon=False, columnspacing=1.2)
fig.tight_layout(rect=(0, 0.09, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(R, "figuras", f"fig12_cascadas_cloro_{m}.{e}"))
print(json.dumps({k: {kk: v[kk] for kk in ("capturas", "suma_media_MeV", "frac_suma_igual_Q_1pct", "frac_suma_mayor_Q", "frac_un_solo_gamma")} for k, v in out.items()}, indent=1))

In [ ]:
mostrar("fig12_cascadas_cloro_nacl25")

### Espectros de captura por núcleo (Fig. 4.1 de la tesis)

`scripts/espectros_captura_nucleo.py` — Espectros gamma de la captura de neutrones en cada nucleo (formato de la Fig. 4.1 de la tesis): (a) 16O(n,g)17O y (b) 1H(n,g)2H con ajuste gaussiano de la linea de 2,224 MeV en agua pura; (c) 35Cl(n,g)36Cl y (d) 37Cl(n,g)38Cl en agua + 2,5 % NaCl. Gammas emitidos en la captura (secundarias-captura-neutron, enlazadas con el nucleo de capturas-neutrones), solo capturas en el medio. Escribe figuras/fig15_espectros_captura_nucleo.{png,pdf} y resultados/lineas_captura_nucleo.csv

In [ ]:
sys.argv = ['espectros_captura_nucleo.py']
__file__ = str(DIR_SCRIPTS / "espectros_captura_nucleo.py")
# ---- código de scripts/espectros_captura_nucleo.py ----
import os
import numpy as np, pandas as pd
from scipy.optimize import curve_fit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
def gammas(m, Z, A):
    z = np.load(os.path.join(RES, f"{m}.npz"))
    cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "A": z["cap_target_A"], "mat": z["cap_material"]})
    cap = cap[(cap.Z == Z) & (cap.A == A) & cap.mat.str.startswith(("Water", "SaltyWater"))]
    sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"]})
    g = sc[sc.p == "gamma"].merge(cap[["run", "trk"]], on=["run", "trk"])
    return g.E.to_numpy() * 1e3, len(cap)          # keV
filas = []
def lineas(E, n, panel, nombre, top):
    """lineas = bines de 1 keV con cuentas; intensidad por 100 capturas y relativa a la mas intensa"""
    b = np.arange(0.5, E.max() + 2, 1.0); h, _ = np.histogram(E, b)   # bordes en x,5 keV: las lineas (x,0-x,9) no se parten
    idx = np.flatnonzero(h); Ec = np.array([E[(E >= b[i]) & (E < b[i + 1])].mean() for i in idx]); hc = h[idx]
    rel = 100 * hc / hc.max()
    for i in np.argsort(-hc)[:12]:
        filas.append({"reaccion": nombre, "E_keV": round(Ec[i], 1), "por_100_capturas": round(100 * hc[i] / n, 2), "relativa": round(rel[i], 1), "capturas": n})
    return Ec, rel, [i for i in np.argsort(-hc)[:top]]
# Referencias (mismas que en los esquemas de niveles y en la Tabla 4.2 unificada del informe), intensidad por 100 capturas:
# 16O: Firestone y Revay, PRC 93, 044311 (2016) (Fig. 4.2 de la tesis); 35Cl: ENSDF (Nica et al. 2012), Ig[b]*100/33,22;
# 37Cl: energias de la Tabla 4.2 (sin intensidades).
REF = {"16O": [(870.76, 96.6), (1087.89, 80.4), (2184.49, 80.4), (3272.02, 16.2), (4142.60, 3.36)],
       "35Cl": [(e, 100 * ib / 33.22) for e, ib in [(517.07, 7.58), (786.30, 3.419), (788.42, 5.42), (1164.86, 8.91), (1170.94, 1.82),
                                                    (1951.13, 6.33), (1959.34, 4.10), (2863.81, 1.818), (5715.36, 1.818), (6110.98, 6.59),
                                                    (6619.73, 2.530), (6627.94, 1.466), (7414.09, 3.29), (7790.45, 2.66), (8578.70, 0.883)]],
       "37Cl": [(637.69, None), (755.43, None), (788.42, None), (1692.11, None), (2166.90, None), (4134.13, None)]}
ETIQ = {"16O": [870.76, 1087.89, 2184.49, 3272.02, 4142.60], "35Cl": [517.07, 788.42, 1164.86, 1951.13, 1959.34, 6110.98, 8578.70],
        "37Cl": [637.69, 755.43, 788.42, 1692.11, 2166.90, 4134.13]}         # lineas de la Tabla 4.2 (energias unificadas)
plt.rcParams.update({"font.size": 12, "axes.spines.top": True, "axes.spines.right": True, "savefig.dpi": 200,
                     "axes.labelsize": 12.5, "xtick.labelsize": 11.5, "ytick.labelsize": 11.5})
fig, axs = plt.subplots(2, 2, figsize=(8.6, 9.8))   # ancho real de la pagina: las letras se imprimen casi a su tamano
MAG = "#c51b8a"
SN = {"16O": 4143.08, "35Cl": 8579.79, "37Cl": 6107.8}   # energia de separacion del nucleo compuesto [keV]


def stick(a, E, n, col, nombre, caja, iso, xmax, panel):
    Ec, rel, _ = lineas(E, n, panel, nombre, 0)
    I = rel * 0 + np.array([((E > e - 0.5) & (E < e + 0.5)).sum() for e in Ec]) * 100 / n   # por 100 capturas
    refs = REF[iso]; Emax_ref = max(e for e, _ in refs)
    # gamma unico de S_n (solo Geant4): linea por encima de todas las de referencia con > 2 por 100 capturas
    sn = [k for k in np.flatnonzero(np.abs(Ec - SN[iso]) < 0.6) if I[k] > 2]
    normal = np.ones(len(Ec), bool); normal[sn] = False
    a.vlines(Ec[normal], 0, I[normal], color=col, lw=1.1, label="MEIGA")
    a.vlines(Ec[sn], 0, I[sn], color=MAG, lw=1.8, label="MEIGA: γ único de $S_n$")
    con_I = [(e, r) for e, r in refs if r is not None]
    if con_I:
        a.plot([e for e, _ in con_I], [r for _, r in con_I], "D", mfc="none", mec="#0b0b0b", ms=6, mew=1.1, label="referencia")
    ymax = 1.3 * max(I.max(), max([r for _, r in con_I], default=0))
    for e, _ in refs:
        if all(r is None for _, r in refs):
            a.plot([e] * 2, [ymax * 0.0, ymax * 0.035], color="#0b0b0b", lw=2)
    if not con_I:
        a.plot([], [], color="#0b0b0b", lw=2, label="energía de referencia (Tabla 4.2)")
    etiquetas = [(e, None) for e in ETIQ[iso]] + [(Ec[k], "sn") for k in sn]
    for j_, (e, tipo) in enumerate(sorted(etiquetas)):
        k = np.argmin(np.abs(Ec - e)); hay = abs(Ec[k] - e) < 0.4 or tipo == "sn"
        x, y = (Ec[k] if hay else e), (I[k] if hay else 0)
        yt = ymax * (0.95 - 0.082 * j_); xt = x + 0.04 * xmax if x < 0.5 * xmax else x - 0.4 * xmax
        if tipo == "sn":
            txt, ec_ = f"{e:.1f}: γ único $S_n$".replace(".", ","), MAG
        else:
            txt, ec_ = (f"{e:.2f}".replace(".", ",") + ("" if hay else " (ausente)")), (col if hay else "#8a8984")
        a.annotate(txt, (x, y), (xt, yt), fontsize=9.5, color="#0b0b0b",
                   bbox=dict(boxstyle="square,pad=0.15", fc="white", ec=ec_, lw=0.9), arrowprops=dict(arrowstyle="-", color=ec_, lw=0.8))
    a.set_xlim(0, xmax); a.set_ylim(0, ymax); a.set_xlabel("Energía del fotón γ [keV]"); a.set_ylabel("Fotones por 100 capturas")
    a.text(0.98, 0.97, caja + f"\n$N_{{\\rm capt}}$ = {n}", transform=a.transAxes, ha="right", va="top", fontsize=9.5,
           bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#8a8984"))
    a.text(-0.16, 1.03, panel, transform=a.transAxes, fontweight="bold", fontsize=12.5)


E, n = gammas("pura", 8, 16); stick(axs[0, 0], E, n, "#e41a1c", "16O(n,g)17O", "Agua pura\n$^{16}$O(n,γ)$^{17}$O", "16O", 5000, "(a)")
# (b) 1H(n,g)2H: ajuste gaussiano
E, n = gammas("pura", 1, 1); x = E - 2224
b = np.arange(0.315, 0.4351, 0.01); h, _ = np.histogram(x, b); c = 0.5 * (b[1:] + b[:-1]); s = np.sqrt(np.maximum(h, 1))
f = lambda t, A, mu, sg: A * np.exp(-0.5 * ((t - mu) / sg) ** 2)
p, cv = curve_fit(f, c, h, p0=[h.max(), 0.373, 0.013], sigma=s, absolute_sigma=True)
chi2 = np.sum(((h - f(c, *p)) / s) ** 2) / (len(c) - 3)
a = axs[0, 1]; tt = np.linspace(0.31, 0.44, 400)
a.plot(tt, f(tt, *p), "-", color="#e41a1c", label="Ajuste gaussiano"); a.errorbar(c, h, s, fmt="s", color="#2a5aa6", ms=4, label="Simulación")
a.set_xlabel("Energía del fotón γ − 2224 [keV]"); a.set_ylabel("Fotones por intervalo"); a.legend(fontsize=11, loc="upper center", bbox_to_anchor=(0.5, 0.66), ncol=2, handlelength=1.3, columnspacing=1.0); a.set_ylim(-1500, 1.95 * h.max())
a.set_xticks([0.32, 0.36, 0.40, 0.44])
a.text(0.02, 0.97, f"Agua pura\n$^1$H(n,γ)$^2$H\n$N_{{\\rm capt}}$ = {n}", transform=a.transAxes, va="top", fontsize=9.5, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#8a8984"))
a.text(0.98, 0.97, f"μ = {2224 + p[1]:.5f} ± {np.sqrt(cv[1, 1]):.5f} keV\nσ = {abs(p[2]):.5f} ± {np.sqrt(cv[2, 2]):.5f} keV\n$\\chi^2_{{\\rm red}}$ = {chi2:.1f}",
       transform=a.transAxes, ha="right", va="top", fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#8a8984"))
a.text(0.98, 0.8, "Valor medido: 2223,25 keV\n(1,12 keV por debajo,\nfuera de la escala)", transform=a.transAxes, ha="right", va="top", fontsize=9.5, color="#555555")
a.text(-0.16, 1.03, "(b)", transform=a.transAxes, fontweight="bold", fontsize=12.5)
filas.append({"reaccion": "1H(n,g)2H", "E_keV": round(2224 + p[1], 5), "por_100_capturas": round(100 * len(E) / n, 2), "relativa": 100, "capturas": n})
E, n = gammas("nacl25", 17, 35); stick(axs[1, 0], E, n, "#1a8a1a", "35Cl(n,g)36Cl", "Agua + 2,5 % de NaCl\n$^{35}$Cl(n,γ)$^{36}$Cl", "35Cl", 9000, "(c)")
E, n = gammas("nacl25", 17, 37); stick(axs[1, 1], E, n, "#1a8a1a", "37Cl(n,g)38Cl", "Agua + 2,5 % de NaCl\n$^{37}$Cl(n,γ)$^{38}$Cl", "37Cl", 7000, "(d)")
from matplotlib.lines import Line2D
h_ = [Line2D([], [], color="#e41a1c", lw=1.5, label="MEIGA (agua pura)"), Line2D([], [], color="#1a8a1a", lw=1.5, label="MEIGA (agua + 2,5 % NaCl)"),
      Line2D([], [], color=MAG, lw=2, label="MEIGA: γ único de $S_n$ (no físico)"),
      Line2D([], [], ls="none", marker="D", mfc="none", mec="#0b0b0b", label="referencia (por 100 capturas)"),
      Line2D([], [], color="#0b0b0b", lw=2, label="energía de referencia ($^{37}$Cl)")]
fig.legend(handles=h_, loc="lower center", ncol=2, fontsize=11.5, frameon=False)
fig.tight_layout(rect=(0, 0.095, 1, 1))
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig15_espectros_captura_nucleo.{e}"))
t = pd.DataFrame(filas); t.to_csv(os.path.join(RES, "lineas_captura_nucleo.csv"), index=False)
print(f"H: mu = {2224 + p[1]:.5f} +- {np.sqrt(cv[1,1]):.5f}  sigma = {abs(p[2]):.5f}  chi2red = {chi2:.1f}")
print(t.groupby("reaccion").head(8).to_string(index=False))

In [ ]:
mostrar("fig15_espectros_captura_nucleo")

### Deuterones de la captura en H (Fig. 4.6 de la tesis)

`scripts/deuterones_captura.py` — Distribucion de energia de los deuterones de la captura 1H(n,g)2H en el medio (formato de la Fig. 4.6 de la tesis). Usa los productos de captura de <medio>.npz (secundarias-captura-neutron). Las cuentas de corridas parciales se escalan a 150000 neutrones primarios para que las alturas sean comparables. Media y RMS se calculan con todos los deuterones (como en la Fig. 4.6), y tambien en la ventana 1,2-1,45 keV. Escribe figuras/fig16_deuterones_captura.{png,pdf} y resultados/deuterones_captura.json

In [ ]:
sys.argv = ['deuterones_captura.py']
__file__ = str(DIR_SCRIPTS / "deuterones_captura.py")
# ---- código de scripts/deuterones_captura.py ----
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
MED = [m for m in ["pura", "nacl25", "nacl5", "nacl10"] if os.path.exists(os.path.join(RES, f"{m}.npz"))]
ETQ = {"pura": "Agua pura", "nacl25": "Agua + 2,5 % NaCl", "nacl5": "Agua + 5 % NaCl", "nacl10": "Agua + 10 % NaCl"}
COL = {"pura": "#e41a1c", "nacl25": "#1a8a1a", "nacl5": "#7b1fa2", "nacl10": "#f5a623"}
MD = 1875.61294                                  # masa del deuteron [MeV]
out = {}
b = np.arange(0.00120, 0.0014501, 0.0000005)    # 0,5 eV
estilo.aplicar()
fig, ax = plt.subplots(figsize=(8, 5.0))
for m in MED:
    z = np.load(os.path.join(RES, f"{m}.npz")); N = int(z["N"])
    cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "mat": z["cap_material"]})
    cap = cap[(cap.Z == 1) & cap.mat.str.startswith(("Water", "SaltyWater"))]
    sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"]})
    d = sc[sc.p == "deuteron"].merge(cap[["run", "trk"]], on=["run", "trk"]).E.to_numpy()
    g = sc[sc.p == "gamma"].merge(cap[["run", "trk"]], on=["run", "trk"]).E.to_numpy()
    w = (d > 0.0012) & (d < 0.00145)
    out[m] = {"N_primarios": N, "deuterones": int(len(d)), "media_MeV": float(d.mean()), "RMS_MeV": float(d.std()),
              "mediana_MeV": float(np.median(d)), "media_ventana_MeV": float(d[w].mean()), "sigma_ventana_MeV": float(d[w].std()),
              "frac_fuera_ventana": float(1 - w.mean()), "max_MeV": float(d.max()),
              "E_retroceso_teorico_MeV": float(np.mean(g) ** 2 / (2 * MD))}
    k = 150000 / N
    h, _ = np.histogram(d, b)
    ax.step(0.5e3 * (b[1:] + b[:-1]), h * k, where="mid", color=COL[m], lw=1.5,
            label=f"{ETQ[m]}: mediana {1e3 * np.median(d):.3f} keV, σ = {1e6 * d[w].std():.1f} eV".replace(".", ","))
ax.set_xlim(1.20, 1.45); ax.set_ylim(0, 1.45 * ax.get_ylim()[1]); ax.grid(alpha=0.4)
ax.set_xlabel("Energía cinética del deuterón [keV]"); ax.set_ylabel("Deuterones por 150 000 neutrones")
ax.legend(fontsize=11.5, loc="upper left", handlelength=1.5)
fig.tight_layout()
for e in ("png", "pdf"):
    fig.savefig(os.path.join(FIG, f"fig16_deuterones_captura.{e}"), dpi=160)
json.dump(out, open(os.path.join(RES, "deuterones_captura.json"), "w"), indent=1)
print(json.dumps(out, indent=1))

In [ ]:
mostrar("fig16_deuterones_captura")

### Tabla 4.2 de la tesis: líneas de captura y su energía en MEIGA

`scripts/tabla_parametros_nucleares.py` — Tabla 4.2 de la tesis (parametros nucleares y lineas gamma de captura termica) y energia de cada linea en MEIGA: para cada linea de la tabla se busca, entre los gammas de las capturas en ese isotopo (<medio>.npz), la linea de MEIGA mas intensa a menos de 3 keV y se da su energia media y su intensidad por 100 capturas. Escribe resultados/tabla_parametros_nucleares.json

In [ ]:
sys.argv = ['tabla_parametros_nucleares.py']
__file__ = str(DIR_SCRIPTS / "tabla_parametros_nucleares.py")
# ---- código de scripts/tabla_parametros_nucleares.py ----
import os, json
import numpy as np, pandas as pd
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES = os.path.join(R, "resultados")
# isotopo: (Z, A, abundancia %, sigma0 [mb], lineas [(E keV, valor original de la Tabla 4.2 si se corrigio)])
# Energias unificadas con las figuras del informe: 16O de Firestone y Revay (2016) como en la Fig. 4.2 de la tesis;
# 35Cl de la evaluacion ENSDF (Nica et al. 2012) como en el esquema de niveles de 36Cl.
TABLA = {"1H": (1, 1, "99.99", "332.60 \\pm 0.70", [(2223.25, None)]),
         "16O": (8, 16, "99.76", "0.170 \\pm 0.003", [(870.76, None), (1087.89, 1087.86), (2184.49, 2184.44), (3272.02, 3272.15), (4142.60, 4142.73)]),
         "35Cl": (17, 35, "75.78", "43840 \\pm 170", [(517.07, None), (788.42, None), (1164.86, None), (1951.13, 1959.13), (1959.34, "nueva"),
                                                     (6110.98, 6110.84), (8578.70, 8578.21)]),
         "23Na": (11, 23, "100", "527 \\pm 7", [(472.20, None)]),
         "37Cl": (17, 37, "24.22", "553 \\pm 17", [(637.69, None), (755.43, None), (788.42, None), (1692.11, None), (2166.90, None), (4134.13, None)])}
VENTANA = 0.4        # keV: misma transicion. El H es la excepcion: Geant4 emite la linea 1,1 keV por encima (2224,37 keV)
gam, ncap = {}, {}
for m in ["pura", "nacl25", "nacl5", "nacl10"]:
    f = os.path.join(RES, f"{m}.npz")
    if not os.path.exists(f):
        continue
    z = np.load(f)
    cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "A": z["cap_target_A"], "mat": z["cap_material"]})
    cap = cap[cap.mat.str.startswith(("Water", "SaltyWater"))]
    sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"] * 1e3})
    g = sc[sc.p == "gamma"].merge(cap, on=["run", "trk"])
    for (Z, A), d in g.groupby(["Z", "A"]):
        gam.setdefault((Z, A), []).append(d.E.to_numpy())
    for (Z, A), d in cap.groupby(["Z", "A"]):
        ncap[(Z, A)] = ncap.get((Z, A), 0) + len(d)
out = {}
for iso, (Z, A, ab, s0, lineas) in TABLA.items():
    E = np.concatenate(gam.get((Z, A), [np.array([])])); n = ncap.get((Z, A), 0); filas = []
    for El, orig in lineas:
        v_ = 1.5 if iso == "1H" else VENTANA
        cerca = E[np.abs(E - El) < v_]
        if len(cerca) == 0:
            filas.append({"tabla": El, "original": orig, "MEIGA": None, "por_100": 0.0}); continue
        v, c = np.unique(np.round(cerca, 1), return_counts=True); Ep = v[np.argmax(c)]
        sel = cerca[np.abs(cerca - Ep) < 0.3]
        filas.append({"tabla": El, "original": orig, "MEIGA": float(sel.mean()), "por_100": 100 * len(sel) / n})
    # gamma unico con toda la energia de separacion (solo Geant4): linea mas intensa por encima de la ultima de la tabla
    alto = E[E > max(l for l, _ in lineas) + 0.3]
    if len(alto):
        v, c = np.unique(np.round(alto, 1), return_counts=True); Ep = v[np.argmax(c)]
        sel = alto[np.abs(alto - Ep) < 0.3]
        if len(sel) / max(n, 1) > 0.02:
            filas_sn = {"MEIGA": float(sel.mean()), "por_100": 100 * len(sel) / n}
        else:
            filas_sn = None
    else:
        filas_sn = None
    out[iso] = {"abundancia": ab, "sigma0_mb": s0, "capturas_MEIGA": n, "lineas": filas, "gamma_unico_Sn": filas_sn}
json.dump(out, open(os.path.join(RES, "tabla_parametros_nucleares.json"), "w"), indent=1)
for iso, v in out.items():
    print(iso, v["capturas_MEIGA"], [(l["tabla"], round(l["MEIGA"], 2) if l["MEIGA"] else None, round(l["por_100"], 1)) for l in v["lineas"]], "Sn:", v["gamma_unico_Sn"])

### Esquema de niveles de 16O(n,γ)17O (Fig. 4.2 de la tesis) e intensidades de MEIGA

`scripts/esquema_niveles_O17.py` — Esquema de niveles y transiciones gamma de la captura 16O(n,g)17O con neutrones termicos (Fig. 4.2 de la tesis). Datos de referencia: Firestone y Revay, Phys. Rev. C 93, 044311 (2016), Sec. II.B.1, Fig. 1 y Tabla III; CapGam/NNDC, 16O(N,G):E=THERMAL. Ademas compara las intensidades (por 100 capturas) con las de MEIGA en las capturas en 16O de las corridas disponibles (<medio>.npz). Escribe figuras/fig18_esquema_niveles_O17.{png,pdf} (replica) y figuras/fig18b_esquema_niveles_O17_MEIGA.{png,pdf} (con MEIGA), y resultados/intensidades_O17.json

In [ ]:
sys.argv = ['esquema_niveles_O17.py']
__file__ = str(DIR_SCRIPTS / "esquema_niveles_O17.py")
# ---- código de scripts/esquema_niveles_O17.py ----
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import estilo
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
# niveles: energia [keV], espin-paridad, texto secundario
NIV = [(0.0, "5/2$^+$", "estable"), (870.78, "1/2$^+$", "180 ps"), (3055.37, "1/2$^-$", "0.08 ps"), (4143.27, "1/2$^+$", None)]
SN = 4143.27            # como en la Fig. 4.2 de la tesis
SN_G4 = 4143.09         # energia del gamma unico que emite Geant4 (= S_n de 17O, 4143,08 keV en AME2020)
# transiciones: E_gamma, nivel inicial, nivel final, multipolaridad, intensidad por 100 capturas, x, color, grosor
TR = [(4142.60, 4143.27, 0.0, "E2", 3.36, 0.12, "#555555", 1.2),
      (3272.02, 4143.27, 870.78, "M1", 16.2, 0.50, "#2a5aa6", 1.4),
      (1087.89, 4143.27, 3055.37, "E1", 80.4, 0.58, "#d62728", 3.2),
      (2184.49, 3055.37, 870.78, "E1", 80.4, 0.80, "#d62728", 3.2),
      (870.76, 870.78, 0.0, "E2", 96.6, 0.92, "#d62728", 3.2)]

# intensidades en MEIGA: gammas de las capturas en 16O en el medio, por 100 capturas
cap_n, cuentas = 0, {t[0]: 0 for t in TR}; suma = []; unico = 0
for m in ["pura", "nacl25", "nacl5", "nacl10"]:
    f = os.path.join(RES, f"{m}.npz")
    if not os.path.exists(f):
        continue
    z = np.load(f)
    cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "A": z["cap_target_A"], "mat": z["cap_material"]})
    cap = cap[(cap.Z == 8) & (cap.A == 16) & cap.mat.str.startswith(("Water", "SaltyWater"))]
    sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"] * 1e3})
    g = sc[sc.p == "gamma"].merge(cap[["run", "trk"]], on=["run", "trk"])
    cap_n += len(cap); suma += g.groupby(["run", "trk"]).E.sum().tolist()
    for t in TR:
        cuentas[t[0]] += int((np.abs(g.E - t[0]) < 0.3).sum())
    unico += int((np.abs(g.E - SN_G4) < 0.3).sum())
out = {"capturas_O16_MEIGA": cap_n, "suma_media_keV": float(np.mean(suma)) if suma else None,
       "frac_suma_igual_Sn": float(np.mean(np.abs(np.array(suma) - SN) < 5)) if suma else None,
       "gamma_unico_Sn_por_100": 100 * unico / cap_n if cap_n else None, "gamma_unico_Sn_error": 100 * np.sqrt(unico) / cap_n if cap_n else None,
       "transiciones": {}}
for t in TR:
    k = cuentas[t[0]]
    out["transiciones"][f"{t[0]:.2f}"] = {"referencia": t[4], "MEIGA": 100 * k / cap_n if cap_n else None,
                                          "MEIGA_error": 100 * np.sqrt(k) / cap_n if cap_n else None, "cuentas": k}
json.dump(out, open(os.path.join(RES, "intensidades_O17.json"), "w"), indent=1)


def dibujar(con_meiga, nombre):
    plt.rcParams.update({"font.size": 10, "font.family": "DejaVu Sans"})
    estilo.aplicar()
    fig, ax = plt.subplots(figsize=(8, 9.6))
    x0, x1 = 0.03, 1.0
    for E, jp, txt in NIV:
        es_cap = abs(E - SN) < 1
        ax.hlines(E, x0, x1, color="#0b0b0b", lw=3.2 if E == 0 else 1.3, ls="--" if es_cap else "-")
        ax.text(x0 - 0.015, E, jp, ha="right", va="center", fontsize=12.5)
        ax.text(x1 + 0.015, E + 40, f"{E:.2f} keV" if E > 0 else "0", ha="left", va="bottom", fontsize=12.5)
        if txt:
            ax.text(x1 + 0.015, E - 40, txt, ha="left", va="top", fontsize=11.5, color="#555555")
    ax.text(x1 + 0.015, SN + 190, "estado de captura ($S_n$)", ha="left", va="bottom", fontsize=11.5, color="#555555")
    for Eg, Ei, Ef, mult, I, x, col, lw in TR:
        ax.annotate("", xy=(x, Ef + 8), xytext=(x, Ei - 8),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, mutation_scale=8 + 3 * lw, shrinkA=0, shrinkB=0))
        ym = Ei - 0.18 * (Ei - Ef) if Ei - Ef > 1500 else (Ei + Ef) / 2 + 120
        if Eg == 4142.60:
            ym = 3750
        if Eg == 3272.02:
            ym = 2250
        etiqueta = f"{Eg:.2f} keV\n{mult}, {I:g} %".replace(".", ",", 0)
        if con_meiga:
            tt = out["transiciones"][f"{Eg:.2f}"]
            etiqueta += (f"\nMEIGA: {tt['MEIGA']:.1f} ± {tt['MEIGA_error']:.1f} %" if tt["cuentas"] else "\nMEIGA: ausente")
        izq = Eg in (870.76, 3272.02)           # a la izquierda de su flecha, para no tapar el nivel fundamental
        ax.text(x - 0.015 if izq else x + 0.015, 830 if Eg == 870.76 else ym, etiqueta, color=col if col != "#555555" else "#0b0b0b", fontsize=11.5, va="top", ha="right" if izq else "left")
    if con_meiga:
        xu = 0.06
        ax.annotate("", xy=(xu, 8), xytext=(xu, SN - 8),
                    arrowprops=dict(arrowstyle="-|>", color="#c51b8a", lw=2.0, ls="--", mutation_scale=13, shrinkA=0, shrinkB=0))
        ax.text(xu + 0.012, 230, f"solo MEIGA: γ único de {SN_G4:.2f} keV (= $S_n$)\n{out['gamma_unico_Sn_por_100']:.1f} ± {out['gamma_unico_Sn_error']:.1f} %",
                ha="left", va="center", fontsize=11.5, color="#c51b8a", bbox=dict(fc="white", ec="none", alpha=0.9, pad=1))
    ax.set_xlim(-0.1, 1.27); ax.set_ylim(-250, 4700)
    ax.set_xticks([]); ax.set_ylabel("Energía de excitación de $^{17}$O [keV]", fontsize=12.5)
    for s in ("top", "right", "bottom"):
        ax.spines[s].set_visible(False)
    ax.spines["left"].set_bounds(-250, 4700)
    tit = "$^{16}$O(n,$\\gamma$)$^{17}$O con neutrones térmicos"
    if con_meiga:
        tit += f"\n(intensidades por 100 capturas; MEIGA: {cap_n} capturas en $^{{16}}$O)"
    ax.set_title(tit, fontsize=13)
    fig.tight_layout()
    for e in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{nombre}.{e}"), dpi=170)
    plt.close(fig)


dibujar(False, "fig18_esquema_niveles_O17")
dibujar(True, "fig18b_esquema_niveles_O17_MEIGA")
print(json.dumps(out, indent=1))

In [ ]:
mostrar("fig18_esquema_niveles_O17")
mostrar("fig18b_esquema_niveles_O17_MEIGA")

### Esquema de niveles de 35Cl(n,γ)36Cl (ENSDF) e intensidades de MEIGA

`scripts/esquema_niveles_Cl36.py` — Esquema de niveles y principales transiciones gamma de la captura 35Cl(n,g)36Cl con neutrones termicos. Datos de referencia: evaluacion ENSDF 35Cl(n,g) E=thermal (N. Nica, J. Cameron y B. Singh, Nucl. Data Sheets 113, 1 (2012); intensidades de 2007ChZX/PGAA-IAEA). Las intensidades Ig de la evaluacion son secciones eficaces parciales; por 100 capturas se multiplican por 100/sigma = 100/33,22 b. Se incluyen las transiciones con mas de ~2,5 por 100 capturas. Ademas compara con las intensidades de MEIGA en las capturas en 35Cl (<medio>.npz). Los niveles se dibujan sin escala. Escribe figuras/fig19_esquema_niveles_Cl36.{png,pdf}, figuras/fig19b_esquema_niveles_Cl36_MEIGA.{png,pdf} y resultados/intensidades_Cl36.json

In [ ]:
sys.argv = ['esquema_niveles_Cl36.py']
__file__ = str(DIR_SCRIPTS / "esquema_niveles_Cl36.py")
# ---- código de scripts/esquema_niveles_Cl36.py ----
import os, json
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
R = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
RES, FIG = os.path.join(R, "resultados"), os.path.join(R, "figuras")
SIG = 33.22                                      # b; Ig(por 100 capturas) = 100 * Ig(b) / SIG
SN = 8579.7945
# niveles: energia [keV], J^pi, T1/2, posicion en el dibujo (sin escala)
NIV = [(0.0, "2$^+$", "3,01×10$^5$ a", 0.0), (788.4329, "3$^+$", None, 2.2), (1164.8802, "1$^+$", None, 4.2),
       (1951.1857, "2", "1,25 ps", 6.2), (1959.3945, "2", "44 fs", 7.3), (2468.2594, "3$^-$", "0,83 ps", 9.0),
       (2863.9311, "(2,3)$^+$", "15 fs", 10.5), (SN, "2$^+$", None, 14.6)]
Y = {E: y for E, _, _, y in NIV}
# transiciones: E_gamma [keV], Ig [b], nivel inicial, nivel final, multipolaridad
TR = [(8578.696, 0.883, SN, 0.0, "M1+E2"), (7790.454, 2.66, SN, 788.4329, "M1+E2"), (7414.09, 3.29, SN, 1164.8802, "M1+E2"),
      (6627.945, 1.466, SN, 1951.1857, ""), (6619.732, 2.530, SN, 1959.3945, ""), (6110.9802, 6.59, SN, 2468.2594, "E1"),
      (5715.356, 1.818, SN, 2863.9311, "M1+E2"),
      (2863.807, 1.818, 2863.9311, 0.0, ""), (517.06962, 7.58, 2468.2594, 1951.1857, ""), (1170.941, 1.82, 1959.3945, 788.4329, ""),
      (1959.337, 4.10, 1959.3945, 0.0, ""), (786.29643, 3.419, 1951.1857, 1164.8802, ""), (1951.12647, 6.33, 1951.1857, 0.0, ""),
      (1164.860, 8.91, 1164.8802, 0.0, "M1+E2"), (788.4236, 5.42, 788.4329, 0.0, "M1+E2")]
COLM = {"E1": "#d62728", "M1+E2": "#2a5aa6", "": "#444444"}

# MEIGA: gammas de las capturas en 35Cl en el medio, por 100 capturas
cap_n, cuentas, suma, unico = 0, {t[0]: 0 for t in TR}, [], 0
for m in ["nacl25", "nacl5", "nacl10"]:
    f = os.path.join(RES, f"{m}.npz")
    if not os.path.exists(f):
        continue
    z = np.load(f)
    cap = pd.DataFrame({"run": z["cap_run_id"], "trk": z["cap_track_id"], "Z": z["cap_target_Z"], "A": z["cap_target_A"], "mat": z["cap_material"]})
    cap = cap[(cap.Z == 17) & (cap.A == 35) & cap.mat.str.startswith(("Water", "SaltyWater"))]
    sc = pd.DataFrame({"run": z["sc_run"], "trk": z["sc_ptrk"], "p": z["sc_part"], "E": z["sc_E"] * 1e3})
    g = sc[sc.p == "gamma"].merge(cap[["run", "trk"]], on=["run", "trk"])
    cap_n += len(cap); suma += g.groupby(["run", "trk"]).E.sum().tolist()
    for t in TR:
        cuentas[t[0]] += int((np.abs(g.E - t[0]) < 0.3).sum())
    unico += int((np.abs(g.E - SN) < 0.3).sum())     # gamma unico con toda la energia de separacion (sin retroceso)
out = {"capturas_Cl35_MEIGA": cap_n, "suma_media_keV": float(np.mean(suma)) if suma else None,
       "gamma_unico_Sn_por_100": 100 * unico / cap_n if cap_n else None, "transiciones": {}}
for t in TR:
    k = cuentas[t[0]]
    out["transiciones"][f"{t[0]:.2f}"] = {"referencia": 100 * t[1] / SIG, "MEIGA": 100 * k / cap_n if cap_n else None,
                                          "MEIGA_error": 100 * np.sqrt(k) / cap_n if cap_n else None, "Ei": t[2], "Ef": t[3]}
json.dump(out, open(os.path.join(RES, "intensidades_Cl36.json"), "w"), indent=1)


def dibujar(con_meiga, nombre):
    plt.rcParams.update({"font.size": 10, "font.family": "DejaVu Sans"})
    fig, ax = plt.subplots(figsize=(13.5, 8.0))   # apaisada: va en una pagina horizontal del informe
    x0, x1 = 0.0, 1.0
    for E, jp, t12, y in NIV:
        cap = E == SN
        ax.hlines(y, x0, x1, color="#0b0b0b", lw=3.2 if E == 0 else 1.2, ls="--" if cap else "-")
        ax.text(x0 - 0.01, y, jp, ha="right", va="center", fontsize=14.4)
        ax.text(x1 + 0.01, y + 0.08, f"{E:.2f} keV".replace(".", ",") if E > 0 else "0", ha="left", va="bottom", fontsize=13.8)
        if t12:
            ax.text(x1 + 0.01, y - 0.08, t12, ha="left", va="top", fontsize=12, color="#555555")
    ax.text(x1 + 0.01, Y[SN] + 0.55, "estado de captura ($S_n$)", ha="left", va="bottom", fontsize=12.6, color="#555555")
    yb = (Y[2863.9311] + Y[SN]) / 2                  # corte del eje: los niveles no estan a escala
    for d in (-0.12, 0.12):
        ax.plot([-0.035, -0.015], [yb + d - 0.1, yb + d + 0.1], color="#0b0b0b", lw=1.2, clip_on=False)
    xs = np.linspace(0.025, 0.925, len(TR))
    for x, (Eg, Ib, Ei, Ef, mult) in zip(xs, TR):
        I = 100 * Ib / SIG; col = COLM[mult]; lw = 0.8 + I / 6
        ax.annotate("", xy=(x, Y[Ef] + 0.04), xytext=(x, Y[Ei] - 0.04),
                    arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, mutation_scale=7 + 2.5 * lw, shrinkA=0, shrinkB=0))
        et = f"{Eg:.2f} keV\n{I:.1f} %".replace(".", ",")
        if con_meiga:
            et += f" [{out['transiciones'][f'{Eg:.2f}']['MEIGA']:.1f}]".replace(".", ",")
        if Ei == SN:
            ym, va_ = Y[Ei] - 0.25, "top"
        elif Y[Ei] - Y[Ef] < 3.2:                   # transicion corta: etiqueta desde el nivel final hacia arriba
            ym, va_ = Y[Ef] + 0.15, "bottom"
        else:
            ym, va_ = (Y[Ei] + Y[Ef]) / 2, "center"
        ax.text(x + 0.003 + 0.0025 * lw, ym, et, rotation=90, ha="left", va=va_, fontsize=12.6, color=col, linespacing=1.0,
                bbox=dict(boxstyle="square,pad=0.1", fc="white", ec="none", alpha=0.85))
    if con_meiga:                                   # gamma unico de energia S_n que emite Geant4 (no existe en la naturaleza)
        xu = 0.975
        ax.annotate("", xy=(xu, Y[0.0] + 0.04), xytext=(xu, Y[SN] - 0.04),
                    arrowprops=dict(arrowstyle="-|>", color="#c51b8a", lw=2.2, ls="--", mutation_scale=14, shrinkA=0, shrinkB=0))
        ax.text(xu - 0.015, Y[SN] - 2.6, f"solo MEIGA:\nγ único de {SN:.1f} keV\n(= $S_n$, sin retroceso)\n[{out['gamma_unico_Sn_por_100']:.1f} % de las capturas]".replace(".", ","),
                ha="right", va="center", fontsize=12.6, color="#c51b8a", linespacing=1.15)
    from matplotlib.lines import Line2D
    h = [Line2D([], [], color=COLM["E1"], lw=2.5, label="E1"), Line2D([], [], color=COLM["M1+E2"], lw=2.5, label="M1+E2"),
         Line2D([], [], color=COLM[""], lw=2.5, label="multipolaridad no asignada")]
    if con_meiga:
        h.append(Line2D([], [], color="#c51b8a", lw=2.2, ls="--", label="γ único de $S_n$ (solo MEIGA)"))
    ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, frameon=False, fontsize=13.2)
    ax.set_xlim(-0.07, 1.14); ax.set_ylim(-0.5, 15.6); ax.axis("off")
    tit = "$^{35}$Cl(n,$\\gamma$)$^{36}$Cl con neutrones térmicos\n(niveles sin escala; intensidades por 100 capturas"
    tit += f"; entre corchetes: MEIGA, {cap_n} capturas)" if con_meiga else ")"
    ax.set_title(tit, fontsize=15)
    fig.tight_layout()
    for e in ("png", "pdf"):
        fig.savefig(os.path.join(FIG, f"{nombre}.{e}"), dpi=170)
    plt.close(fig)


dibujar(False, "fig19_esquema_niveles_Cl36")
dibujar(True, "fig19b_esquema_niveles_Cl36_MEIGA")
for k, v in out["transiciones"].items():
    print(f"{k:>9}  ref {v['referencia']:5.2f}  MEIGA {v['MEIGA']:5.2f} ± {v['MEIGA_error']:.2f}")
print("capturas", cap_n, "suma media", out["suma_media_keV"], "gamma unico Sn %", out["gamma_unico_Sn_por_100"])

In [ ]:
mostrar("fig19_esquema_niveles_Cl36")
mostrar("fig19b_esquema_niveles_Cl36_MEIGA")

## 7. Informe

In [ ]:
# Informe principal: copia fiel de la Sec. 4.1.1 de la tesis (seccion_4_1_1.tex + apendice_procesos.tex, preambulo de la tesis).
# Las notas tecnicas (Informe-Sidelnik-500MeV-tecnico) se generan con escribir_informe.py a partir de resultados/.
ejecutar("escribir_informe.py")
ejecutar("construir_informe.py")          # cuerpo del informe a partir de la fuente comun con la tesis
for doc in ("Informe-Sidelnik-500MeV", "Informe-Sidelnik-500MeV-tecnico"):
    r = subprocess.run(["latexmk", "-pdf", "-interaction=nonstopmode", doc + ".tex"],
                       cwd=BASE / "informe", capture_output=True, text=True, errors="replace")
    print(doc, "->", "correcto" if r.returncode == 0 else r.stdout[-2000:])
subprocess.run(["latexmk", "-c"], cwd=BASE / "informe", capture_output=True)

## 8. Resultados y discusión (contenido completo del informe)

Esta sección reproduce el texto, las ecuaciones, las tablas y las figuras de `informe/Informe-Sidelnik-500MeV.pdf`, con el mismo contenido que la Sec. 4.1.1 de la tesis, con la numeración del informe.
Se genera automáticamente desde el informe (`scripts/informe_tesis_a_markdown.py`) cada vez que se regenera el notebook, así que siempre coincide con el PDF.
Si se reprocesan los datos, ejecute la sección 7 y luego `python3 scripts/construir_notebook.py` para actualizar este texto.

### Resumen

Se reprodujeron con el simulador MEIGA (Geant4 10.7.p04, `QGSP_BERT_HP_NoThermal`, sin $S(\alpha,\beta)$) la geometría, los materiales y la irradiación del detector Cherenkov de agua dopada con NaCl de Sidelnik et al. (2020), con $1.5\times10^{5}$ neutrones de 500 MeV en agua pura y en agua con 2.5, 5 y 10 % de NaCl. La validación sigue la cadena física completa: interacción del neutrón primario, moderación, captura, gammas de captura, electrones secundarios, luz Cherenkov y carga en el PMT. El modelo reproduce la moderación ($\xi_{\mathrm{H}}=1.000$, $\xi_{\mathrm{O}}=0.118$), la probabilidad de captura (24.08 % frente a 25.18 % en agua pura), el tiempo de captura y su reducción con la sal (de 192 a 75 µs), las líneas $\gamma$ de captura, el espectro de electrones en escala absoluta (dentro del 4–8 %) y la distribución de fotones Cherenkov aceptados entre 0 y 500 fotones sin reescalar; las curvas de razón NaCl/agua pura coinciden en forma con las del artículo ($r=0.88$–$0.91$). Las diferencias se atribuyen a la mayor producción de piones del modelo de Bertini, a un fondo de desexcitación nuclear mayor, a un aumento de la captura con la sal más débil y al modelo de captura de Geant4, que no conserva la energía en las cascadas del Cl. Este informe reproduce, con la misma redacción, figuras y tablas, la Sec. 4.1.1 de la tesis doctoral del autor.

### 1. Introducción y configuración de la simulación

Con el propósito de evaluar y validar el modelo Monte Carlo desarrollado en este trabajo, se reprodujeron las principales condiciones geométricas, físicas y de irradiación del detector Cherenkov de agua dopada con NaCl descrito por Sidelnik et al. (2020, Sec. 2, pp. 2217–2218, Fig. 1). *(Nota: Los resultados de este informe se obtuvieron con corridas del simulador MEIGA, con $1.5\times10^{5}$ neutrones de 500 MeV por medio, la misma geometría y la misma física en todos los medios. El análisis completo, los programas y los datos reducidos se encuentran en el repositorio del proyecto (Betancourt, 2026), carpeta `analysis/replica-Sidelnik-MEIGA_seccion-4.1.1`. Se simularon los cuatro medios: agua pura, agua + 2.5 % de NaCl, agua + 5 % de NaCl y agua + 10 % de NaCl.)* La estrategia de validación se realizó de manera progresiva, siguiendo la cadena de procesos físicos que conduce desde la interacción inicial del neutrón con el medio detector hasta la generación de la señal óptica: la interacción del neutrón primario de alta energía, la moderación de los neutrones secundarios mediante dispersiones elásticas, su captura, la emisión de fotones gamma de captura $\gamma_{\mathrm{Cap}}$, la producción de electrones y positrones secundarios y, finalmente, la generación de radiación Cherenkov y la respuesta del tubo fotomultiplicador.

Los principales parámetros geométricos, físicos y de simulación adoptados para realizar esta comparación se resumen en la Tabla (1.1). En ella se identifican tanto las condiciones reproducidas directamente del modelo de referencia como aquellas implementadas específicamente en el presente trabajo.

**Tabla 1.1:** Parámetros geométricos, materiales y de simulación del modelo Geant4 (MEIGA) desarrollado en este trabajo, frente a la configuración del detector WCD de Sidelnik et al. (2020). La última columna indica dónde se encuentra cada dato en dicho trabajo.

| **Parámetro** | **Este trabajo** | **Sidelnik et al. (2020)** | **Referencia** |
|:--|:--|:--|:--|
| **Geometría del detector** |  |  |  |
| Altura / diámetro del tanque | 133 cm / 96 cm | 133 cm / 96 cm | Sec. 2, p. 2217 |
| Radio del tanque | 48 cm | 48 cm$^{a}$ | Sec. 2, p. 2217 |
| Volumen activo | $\approx 1~\mathrm{m^{3}}$ | $\approx 1~\mathrm{m^{3}}$ | Sec. 1, p. 2217 |
| Tanque | Acero inoxidable de 0.5 mm (lados, tapa y fondo) | Acero inoxidable de 0.5 mm | Sec. 2, p. 2217 |
| **Revestimiento interno** |  |  |  |
| Material y espesor | Tyvek, 0.12 mm | Tyvek, $\approx$ 0.12 mm | Sec. 2, p. 2217 |
| Modelo óptico | Reflector difuso (`groundbackpainted`) con la reflectividad medida del Tyvek$^{b}$ | Propiedades reflectivas y refractivas del Tyvek | Sec. 2, pp. 2217–2218 |
| **Tubo fotomultiplicador** |  |  |  |
| Modelo | Hamamatsu R5912, 8 pulgadas | Hamamatsu R5912, 8 pulgadas | Sec. 2, p. 2217 |
| Superficie sensible | Sección esférica: radio 10.16 cm, altura 5 cm | Sección esférica: radio 10.16 cm, altura 5 cm | Sec. 2, p. 2217 |
| Posición | Centro de la tapa superior | Centro de la tapa superior | Sec. 2; Fig. 1 |
| **Fuente de neutrones y simulación** |  |  |  |
| Partícula primaria | Neutrones de 500 MeV | Neutrones de 500 MeV | Sec. 2, p. 2218 |
| Fuente | Haz horizontal a media altura (66.562 cm), 1 cm fuera de la pared | Haz horizontal sobre la pared lateral | Sec. 2; Fig. 1 |
| Neutrones por medio | $1.5\times10^{5}$ | $1.5\times10^{5}$ | Sec. 2, p. 2218 |
| Versión de Geant4 | 10.7.p04 | No especificada | – |
| Lista física | QGSP_BERT_HP_ NoThermal, sin $S(\alpha,\beta)$; G4NDL 4.6 | No especificada | – |
| Detección de la luz | Eficiencia cuántica del R5912 aplicada al crearse cada fotón (modo `eFast`) | Eficiencia cuántica del PMT | Sec. 3, Fig. 6 |
| **Medio detector** |  |  |  |
| Configuraciones | Agua pura; agua + 2.5 % NaCl; agua + 5 % NaCl; agua + 10 % NaCl | Las mismas cuatro | Sec. 2, p. 2218 |

*Notas:* $^{a}$Derivado del diámetro reportado ($D=96$ cm). $^{b}$Implementación propia de este trabajo; no reproduce un modelo de superficie óptica especificado por Sidelnik et al. Las corridas no incluyen la dispersión térmica $S(\alpha,\beta)$ (modelo de gas libre para los neutrones térmicos), como en la primera campaña de simulación.

La Figura (1.1) muestra un evento real de la simulación sobre la geometría del detector. El neutrón de 500 MeV ingresa a media altura y sufre una interacción inelástica; los neutrones secundarios se moderan y algunos de ellos son capturados. Las capturas emiten fotones gamma cuyos electrones secundarios, de pocos milímetros de recorrido, generan la luz Cherenkov que finalmente alcanza el PMT. Cada eslabón de esta cadena se compara a continuación con el trabajo de referencia, cuyos resultados se presentan en la Sec. 3 (pp. 2218–2221) (Sidelnik et al., 2020, Sec. 3, pp. 2218–2221).

![Figura 1.1](figuras/fig00_esquema_detector.png)

**Figura 1.1:** Esquema del WCD simulado (corte $y=0$, a escala; el espesor de las paredes está exagerado) con un evento real de la corrida de agua + 2.5 % de NaCl. El neutrón de 500 MeV entra a media altura y tiene una interacción inelástica; los neutrones secundarios se moderan y dos de ellos se capturan, uno en $^{1}$H (gamma de 2224 keV) y otro en $^{35}$Cl, con la cascada de 6111 + 1951 + 517 keV que conserva la energía de separación del neutrón (8.579 MeV). Los electrones de los gammas recorren pocos milímetros y emiten la luz Cherenkov. Equivalente a la Fig. 1 de Sidelnik et al. (2020), con la geometría y las trayectorias del presente modelo.

La comparación se organiza en tres niveles de exigencia: (i) magnitudes que deben coincidir en valor absoluto porque dependen solo de la física nuclear y de la geometría (energías de las líneas gamma, tiempos de captura, escala del espectro de electrones); (ii) magnitudes que se comparan mediante cocientes o formas normalizadas, porque los ejes del trabajo de referencia están en unidades arbitrarias o su normalización no se especifica (alturas relativas de líneas, espectros acumulados, histogramas de fotones); y (iii) tendencias con la concentración de NaCl, que no dependen de la normalización y constituyen la prueba más directa del objetivo de este trabajo.

### 2. Interacciones del neutrón primario de 500 MeV

Antes de analizar la captura conviene seguir al neutrón incidente. A 500 MeV el neutrón primario solo experimenta tres procesos en el detector: dispersión elástica (`hadElastic`), interacción inelástica (`neutronInelastic`) y, raramente, captura. El núcleo blanco de cada interacción inelástica se reconstruyó por conservación de la carga y del número bariónico de los productos; los isótopos $^{17}$O (0.04 %) y $^{18}$O (0.21 %) aparecen con su abundancia natural, lo que valida la reconstrucción. La Tabla (2.1) resume el destino del neutrón primario en los cuatro medios.

**Tabla 2.1:** Destino del neutrón primario de 500 MeV y localización de su interacción inelástica ($1.5\times10^{5}$ neutrones por medio).

| **Destino del neutrón primario** | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Atraviesa el tanque sin interactuar | 21.8 % | 21.3 % | 21.0 % | 20.0 % |
| Interacción inelástica (la traza termina) | 61.7 % | 62.6 % | 62.9 % | 64.1 % |
| Solo dispersiones elásticas y sale | 15.6 % | 15.3 % | 15.3 % | 15.1 % |
| Solo dispersiones elásticas y es capturado | 0.88 % | 0.81 % | 0.78 % | 0.76 % |
| Inelástica en el medio | 99.0 % | 99.0 % | 99.1 % | 99.1 % |
| Inelástica en el acero / aire | 0.76 / 0.21 % | 0.75 / 0.20 % | 0.70 / 0.20 % | 0.69 / 0.17 % |
| Blanco en el medio: $^{16}$O | 97.5 % | 95.2 % | 92.9 % | 88.3 % |
| Blanco en el medio: $^{1}$H | 2.2 % | 2.1 % | 2.2 % | 2.0 % |
| Blanco en el medio: $^{35}$Cl + $^{37}$Cl + $^{23}$Na | – | 2.33 % | 4.64 % | 9.40 % |
| Profundidad media de la inelástica (cm) | 40.2 | 40.2 | 40.0 | 39.8 |

En la interacción inelástica, el modelo de cascada intranuclear de Bertini (`QGSP_BERT_HP`, por encima de $\approx$ 20 MeV) termina la traza del primario y crea trazas nuevas para todos los productos. La interacción ocurre casi siempre a la energía inicial: el 80 % de los primarios no sufre ninguna dispersión elástica previa. El 99 % de las inelásticas ocurre en el agua, distribuidas en todo el tanque y decreciendo con la profundidad (camino libre medio inelástico $\approx$ 1 m; Figura 2.1a); el blanco es $^{16}$O en el 97 % de los casos y $^{1}$H en el 2.2 %, ya que la colisión neutrón–protón inelástica requiere producir un pion. Con NaCl, los núcleos de Cl y Na son el blanco en solo el 2.3 % (2.5 %), 4.6 % (5 %) y 9.4 % (10 %) de las inelásticas: la sal no modifica la fase de alta energía.

Cada inelástica produce en promedio 2.12 neutrones, 1.35 protones, 0.33 partículas $\alpha$, 0.12 deuterones, 2.0 gammas de desexcitación y 0.18 piones, y deja un núcleo residual en el 89 % de los casos ($^{12}$C, $^{15}$N, $^{15}$O, $^{14}$N, $^{8}$Be; Figura 2.1d). La energía se reparte principalmente entre neutrones (280 MeV por inelástica) y protones (125 MeV). El neutrón más energético conserva en mediana el 48 % de la energía incidente y puede producir a su vez otra inelástica, de modo que cada evento contiene en promedio 3.44 neutrones en agua pura. Los neutrones que terminan capturados son los de evaporación, de pocos MeV, que se moderan en el agua. Los núcleos residuales emisores $\beta^{+}$ ($^{15}$O, $^{13}$N y $^{11}$C) son además la fuente de la línea de aniquilación de larga vida discutida en la Sec. 6, y los gammas de desexcitación de $^{12}$C, $^{15}$N y $^{16}$O forman el fondo de 4–7 MeV discutido en la Sec. 7. Las multiplicidades y los núcleos residuales coinciden en los cuatro medios dentro del 2 %.

![Figura 2.1](figuras/fig20_inelasticas_primario.png)

**Figura 2.1:** Interacciones inelásticas del neutrón primario de 500 MeV: (a) profundidad a lo largo del haz; (b) multiplicidad media de los productos; (c) espectro de energía de los productos en agua pura; (d) núcleos residuales más frecuentes.

**El neutrón capturado no es el primario.**  Solo el 0.88 % (agua pura), 0.81 % (2.5 %), 0.78 % (5 %) y 0.76 % (10 %) de los primarios se captura como la misma traza, después de frenarse únicamente por dispersiones elásticas. En cambio, en el 97 % de los eventos con captura el primario tuvo antes una interacción inelástica: los neutrones capturados son los secundarios. Este comportamiento no es propio del presente modelo, sino de cualquier simulación con Geant4, incluida la del trabajo de referencia, porque los modelos de cascada intranuclear terminan la traza incidente; además, los neutrones son partículas indistinguibles y, después de la inelástica, carece de sentido preguntar si se capturó “el primario”. En consecuencia, la probabilidad de captura debe definirse sobre el conjunto de neutrones del evento, como se discute en la Sec. 4.

**Inventario de procesos.**  El Apéndice A reúne, para cada medio, todas las reacciones de los neutrones en el volumen del detector, primarios y secundarios (Tablas A.1–A.4): la ecuación de la reacción, su valor $Q$, la energía típica del neutrón, su porcentaje dentro de cada tipo de proceso y los gammas que emite. Estas tablas constituyen el inventario a partir del cual se interpretan las secciones siguientes. En agua pura, la dispersión elástica en $^{1}$H representa el 84 % de las colisiones con blanco identificado y es la responsable de la moderación; la captura ocurre en $^{1}$H en el 99.9 % de los casos; y entre las interacciones inelásticas dominan la dispersión inelástica $^{16}$O(n,n$'$) y los canales $^{16}$O(n,n$\alpha$)$^{12}$C, $^{16}$O(n,np)$^{15}$N y $^{16}$O(n,2n)$^{15}$O, que emiten las líneas de desexcitación de 4.44, 6.13, 6.92 y 7.12 MeV y producen los emisores $\beta^{+}$ de larga vida. Con NaCl se añaden la captura en $^{35}$Cl, con su cascada de 517 a 8579 keV, y, en mucha menor proporción, las reacciones en $^{23}$Na y $^{37}$Cl. Las diferencias con el trabajo de referencia en el fondo de gammas de 4–7 MeV y en la línea de 0.511 MeV (Secs. 6 y 7) tienen su origen en estos canales inelásticos.

### 3. Moderación de los neutrones: dispersiones elásticas

Las dispersiones elásticas son el mecanismo que modera a los neutrones antes de la captura. Se reconstruyeron todos los vértices elásticos de todos los neutrones: 15 573 769 en agua pura (104 por neutrón incidente), 12 519 457 en agua + 2.5 % de NaCl (83 por neutrón), 10 732 779 en agua + 5 % de NaCl (72 por neutrón) y 8 834 631 en agua + 10 % de NaCl (59 por neutrón). Con sal hay menos vértices porque el cloro captura antes a los neutrones térmicos: la fracción de vértices por debajo de 1 eV es del 74 %, 69 %, 65 % y 59 %, respectivamente. Cada vértice corresponde a la reacción

$$ n(E_{\mathrm{in}},\hat{d}_{\mathrm{in}}) + (A,Z) \longrightarrow n(E_{\mathrm{out}},\hat{d}_{\mathrm{out}}) + (A,Z)_{\mathrm{retroceso}} . \qquad (3.1) $$

La Figura (3.1) resume la reconstrucción de estos vértices. Cada panel responde a una pregunta sobre la moderación: con qué núcleo choca el neutrón (paneles a y b), cuánta energía pierde y en qué dirección sale (paneles c, d y e) y si la simulación respeta las leyes de conservación (panel f).

![Figura 3.1](figuras/fig21_elasticas.png)

**Figura 3.1:** Reconstrucción de los vértices de dispersión elástica: (a) núcleo blanco según la energía del neutrón; (b) masa del blanco deducida solo de la cinemática del neutrón; (c) fracción de energía perdida por colisión en H y $^{16}$O frente a la teoría; (d) ángulo de dispersión en el laboratorio; (e) decremento logarítmico medio $\xi$; (f) balance de energía e impulso en cada vértice.

**Con quién: el núcleo blanco.**  A partir de la transferencia de impulso $\vec{q}=\vec{p}_{\mathrm{in}}-\vec{p}_{\mathrm{out}}$ y de energía $T=E_{\mathrm{in}}-E_{\mathrm{out}}$ del neutrón, la masa del núcleo blanco se obtiene sin usar el núcleo de retroceso como

$$ M = \frac{q^{2}-T^{2}}{2T}. \qquad (3.2) $$

La distribución de $M$ (Figura 3.1b) presenta picos estrechos en la masa del protón (938.27 MeV) y del núcleo de $^{16}$O (14 895.1 MeV) y, en los medios con NaCl, en las de $^{23}$Na y $^{35}$Cl. En el 100 % de los vértices con retroceso registrado, el blanco reconstruido coincide con la partícula de retroceso. Por debajo de 20 MeV las masas reconstruidas son las *atómicas* (938.78 y 14 899.1 MeV), porque el modelo de alta precisión `G4NeutronHP` trabaja con masas atómicas, mientras que por encima de 20 MeV los modelos hadrónicos usan masas nucleares. La composición por energía (Figura 3.1a y Tabla 3.1) sigue las secciones eficaces elásticas: el hidrógeno es el blanco en el 86–88 % de las colisiones por debajo de 1 MeV, mientras que por encima de 20 MeV el oxígeno pasa a ser el blanco más frecuente (44.8 %), porque la sección eficaz n–p decrece rápidamente con la energía.

**Tabla 3.1:** Vértices de dispersión elástica por intervalo de energía en agua pura, núcleo blanco reconstruido por cinemática y decremento logarítmico medio $\xi$. Por debajo de 1 eV (11 466 643 vértices) el blanco térmico no está en reposo y no se reconstruye.

| **Energía del neutrón** | **Vértices** | **Blanco H / O** | $\boldsymbol{\xi_{\mathrm{H}}}$ | $\boldsymbol{\xi_{\mathrm{O}}}$ |
|:--|:--|:--|:--|:--|
| 1 eV – 1 keV | 1 002 307 | 88.2 % / 5.5 % | 1.017 | 0.131 |
| 1 keV – 1 MeV | 1 083 824 | 85.6 % / 14.2 % | 1.000 | 0.118 |
| 1 – 20 MeV | 413 486 | 70.7 % / 28.4 % | 1.007 | 0.094 |
| $>$ 20 MeV | 194 087 | 33.6 % / 44.8 % | 1.106 | 0.011 |
| Teoría (dispersión isótropa en el CM) |  |  | 1.000 | 0.121 |

**Energía y ángulo.**  En una colisión elástica con un núcleo en reposo y dispersión isótropa en el sistema del centro de masas (CM), la energía final del neutrón se distribuye uniformemente entre $\alpha E$ y $E$, con

$$ \alpha=\left(\frac{A-1}{A+1}\right)^{2}, \qquad (3.3) $$

de modo que la fracción de energía perdida $\Delta E/E$ es uniforme entre 0 y $1-\alpha$. La Figura (3.1c) muestra exactamente este comportamiento entre 1 keV y 1 MeV: para el H ($\alpha=0$) la pérdida es uniforme entre 0 y 1, es decir, en una sola colisión el neutrón puede perder toda su energía, y para el $^{16}$O ($\alpha=0.777$) es uniforme entre 0 y $1-\alpha=0.223$. La misma cinemática fija el ángulo de dispersión en el laboratorio (Figura 3.1d). La Figura (3.2) explica gráficamente estos resultados: en la colisión con $^{1}$H, de masa casi igual a la del neutrón, el neutrón y el protón salen en direcciones perpendiculares ($\theta+\phi=90^{\circ}$), por lo que el neutrón nunca retrocede (panel a); en la colisión con $^{16}$O el neutrón puede rebotar hacia atrás, pero conserva al menos el 78 % de su energía (panel b). El diagrama de velocidades (panel c) muestra el origen de esta diferencia: la velocidad final en el laboratorio es la suma de la velocidad del centro de masas, $V_{CM}=v/(A+1)$, y de la velocidad del neutrón en el CM, $u'=Av/(A+1)$, que, al ser la dispersión isótropa en el CM, puede apuntar en cualquier dirección sobre un círculo de radio $u'$. Para el H ($A=1$) el círculo pasa por el origen y la velocidad final nunca apunta hacia atrás; para el $^{16}$O ($A\approx16$) el origen queda dentro del círculo y cualquier dirección es posible. Para el H, el neutrón no puede retroceder: $\cos\theta_{\mathrm{lab}}$ está entre 0 y 1, con densidad $2\cos\theta_{\mathrm{lab}}$ y valor medio $\langle\cos\theta_{\mathrm{lab}}\rangle=2/(3A)=0.67$. Para el $^{16}$O la distribución es casi plana entre $-1$ y 1, con $\langle\cos\theta_{\mathrm{lab}}\rangle=0.04$. Por encima de 20 MeV el comportamiento cambia para el oxígeno: su distribución se concentra hacia $\cos\theta_{\mathrm{lab}}\approx1$, porque la dispersión deja de ser isótropa en el CM y se vuelve frontal (difracción), de modo que el neutrón rápido apenas cambia de dirección ni de energía al chocar con el O. En el H, en cambio, la distribución sigue cercana a la de la dispersión isótropa en el CM (Figura 3.2f).

![Figura 3.2](figuras/fig23_esquema_cinematica_elastica.png)

**Figura 3.2:** Cinemática de la dispersión elástica de neutrones en el laboratorio. (a) Colisión con $^{1}$H: el neutrón y el protón salen perpendiculares y el neutrón no puede retroceder (región sombreada). (b) Colisión con $^{16}$O: el neutrón puede rebotar y conserva al menos el 78 % de su energía; la velocidad del núcleo de retroceso se muestra multiplicada por 3. (c) Diagrama de velocidades: $\vec v\,'_{\mathrm{lab}}=\vec V_{CM}+\vec u\,'$, con $\vec u\,'$ isótropa sobre el círculo de radio $u'$. (d) Distribución de la energía final, uniforme entre $\alpha E$ y $E$. (e) Distribución de $\cos\theta_{\mathrm{lab}}$ entre 1 keV y 1 MeV: teoría (dispersión isótropa en el CM) frente a la simulación en agua pura. (f) Por encima de 20 MeV: el $^{16}$O dispersa hacia adelante (difracción), mientras que el H permanece cercano a la dispersión isótropa en el CM.

El decremento logarítmico medio resume la pérdida de energía por colisión:

$$ \xi = \left\langle \ln\frac{E}{E'}\right\rangle = 1 + \frac{\alpha\ln\alpha}{1-\alpha}. \qquad (3.4) $$

Entre 1 keV y 1 MeV se obtiene $\xi_{\mathrm{H}}=1.000$ (teoría: 1) y $\xi_{\mathrm{O}}=0.118$ (teoría: 0.121), y ambos valores se mantienen constantes desde el eV hasta unos MeV (Figura 3.1e y Tabla 3.1). Con ellos, el número medio de colisiones necesarias para moderar un neutrón de evaporación de 2 MeV hasta la energía térmica (0.025 eV) es $n=\ln(E_{0}/E_{\mathrm{th}})/\xi\approx18$ en H y $\approx150$ en $^{16}$O. Por encima de 20 MeV $\xi_{\mathrm{O}}$ cae a 0.011, por la dispersión frontal, mientras que $\xi_{\mathrm{H}}$ sube (1.1). En consecuencia, los neutrones rápidos se moderan casi exclusivamente en el hidrógeno. Con NaCl, el Na y el Cl son el blanco en solo el 0.6 % (2.5 %), 1.0 % (5 %) y 1.7 % (10 %) de las colisiones, y la cinemática (masas reconstruidas, $\xi$ y conservación) coincide en los cuatro medios dentro de 0.002: *la sal modifica la captura, no la moderación*.

**Neutrones térmicos.**  El 74 % de los vértices en agua pura (11 466 643) ocurre por debajo de 1 eV, cuando el neutrón ya está termalizado y difunde por el agua hasta ser capturado. En este régimen el núcleo blanco no está en reposo: su energía térmica ($\sim kT=0.025$ eV) es comparable a la del neutrón, y la simulación la describe con el modelo de gas libre, porque estas corridas no incluyen la dispersión térmica $S(\alpha,\beta)$ que tiene en cuenta los enlaces moleculares del agua. Por ello el 44 % de las colisiones *aumenta* la energía del neutrón, que intercambia energía con el medio hasta alcanzar el equilibrio térmico (mediana de la energía al capturarse: 30.5 meV, Tabla 4.2). La reconstrucción con el blanco en reposo no es válida en este régimen y no se aplica; además, Geant4 no registra el núcleo de retroceso por debajo de $\approx$ 1 keV. Las pequeñas desviaciones de $\xi$ entre 1 eV y 1 keV (1.017 para el H y 0.131 para el O) son el comienzo de este efecto térmico.

**Leyes de conservación.**  En cada vértice con retroceso registrado se verificó el balance de energía y de impulso entre el estado inicial (neutrón incidente y núcleo en reposo) y el final (neutrón dispersado y núcleo de retroceso) (Figura 3.1f). La energía se conserva dentro de una parte en $10^{3}$ en el 99.998 % de los vértices; el residuo típico, de $\sim5\times10^{-4}$, proviene de la convención de masas atómicas del modelo de alta precisión. El impulso se conserva dentro del 1 % en el 96.7 % de los vértices (mediana del desbalance relativo $1.2\times10^{-4}$); el resto corresponde a núcleos de retroceso cuyo primer paso es tan corto que su dirección queda mal definida, no a una violación física. La carga, el número bariónico y el número leptónico se conservan exactamente, porque el estado inicial y el final contienen el mismo neutrón y el mismo núcleo, y no aparecen núcleos de retroceso distintos de los del medio ($p$, $^{16,17,18}$O, $^{23}$Na, $^{35,37}$Cl), salvo en el Tyvek ($^{12}$C), el acero ($^{56}$Fe, $^{52}$Cr), el aire ($^{14}$N) y algunos deuterones del $^{2}$H natural del agua. Estos controles confirman que la moderación simulada respeta la cinemática de la dispersión elástica de dos cuerpos.

### 4. Probabilidad, tiempo y distancia de captura

Sidelnik et al. (2020, Sec. 3, Tabla 2, p. 2218) reportan la fracción de capturas de neutrones dentro del volumen activo sin especificar sobre qué total se calcula. La Tabla (4.1) compara tres definiciones posibles. La captura de la traza primaria (0.88 %) mide la numeración de las trazas en Geant4 y no es comparable con el trabajo de referencia. La magnitud comparable es el número de capturas en el agua dividido por todos los neutrones del evento (primario y secundarios): 24.08 % en agua pura, frente al 25.18 % de Sidelnik et al. En otras palabras, de cada cuatro neutrones presentes en el tanque, uno termina capturado en el agua; los demás escapan o desaparecen en interacciones inelásticas. Para la eficiencia del detector, la magnitud relevante es, en cambio, la fracción de neutrones incidentes que producen al menos una captura: 42.4 % en agua pura.

**Tabla 4.1:** Tres definiciones de la probabilidad de captura de neutrones en el volumen activo. Solo la tercera es comparable con la Tabla 2 de Sidelnik et al. (2020); entre paréntesis, el cambio relativo respecto al agua pura.

|  | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Primario capturado (misma traza) | 0.88 % | 0.81 % | 0.78 % | 0.76 % |
| Neutrones incidentes con $\geq 1$ captura en el agua | 42.4 % | 43.3 % (+2.0 %) | 44.3 % (+4.3 %) | 45.9 % (+8.3 %) |
| **Capturas / neutrones del evento (este trabajo)** | **24.08 %** | **24.59 %** (+2.1 %) | **25.19 %** (+4.6 %) | **26.08 %** (+8.3 %) |
| *Sidelnik et al.: probabilidad de captura* | *25.18 %* | *26.73 %* (+6.2 %) | *27.65 %* (+9.8 %) | *29.26 %* (+16.2 %) |
| Capturas en el agua por neutrón incidente | 0.829 | 0.864 (+4.3 %) | 0.903 (+8.9 %) | 0.972 (+17.3 %) |
| Neutrones por evento (primario + secundarios) | 3.44 | 3.51 | 3.58 | 3.73 |

En agua pura la probabilidad de captura queda 1.1 puntos por debajo del trabajo de referencia (5 % relativo), una diferencia razonable entre dos simulaciones con versiones y modelos hadrónicos distintos. Con NaCl la tendencia es la misma que en el trabajo de referencia, monótona con la concentración, pero aproximadamente dos veces más débil: +2.1 %, +4.6 % y +8.3 % frente a +6.2 %, +9.8 % y +16.2 %. Esta diferencia no puede atribuirse a la estadística (con 2.5 % de NaCl el aumento es significativo a $\approx$ 6$\sigma$) ni a un error de configuración, porque el tiempo de captura sigue exactamente la predicción de las secciones eficaces térmicas (Tabla 4.2). Puede deberse a diferencias en las secciones eficaces del cloro o en la normalización de Sidelnik et al., que el artículo no detalla.

**Efecto del NaCl en la captura.**  La Tabla (4.2) resume el efecto de la sal. El tiempo característico de captura, obtenido del ajuste exponencial de la distribución de tiempos (Figura 4.1), puede estimarse a partir de las secciones eficaces térmicas de captura de los núcleos del medio como

$$ \tau = \frac{1}{v_{0}\sum_{i} N_{i}\,\sigma_{0,i}}, \qquad (4.1) $$

donde $v_{0}=2200$ m/s es la velocidad de referencia de los neutrones térmicos, $N_{i}$ la densidad de núcleos del isótopo $i$ y $\sigma_{0,i}$ su sección eficaz térmica de captura (Tabla 5.1). La simulación reproduce la reducción de $\tau$ con la concentración dentro del 6 % en los cuatro medios (de 192 a 75 µs, frente a 205 y 78 µs). La fracción de capturas en $^{35}$Cl es, en cambio, mayor que la predicha por las mismas secciones eficaces (35.8, 51.5 y 67.7 % frente a 28.0, 44.2 y 62.2 %), un exceso relativo que disminuye con la concentración y ya observado en las campañas anteriores con G4NDL 4.6.

**Tabla 4.2:** Captura de neutrones en los distintos medios ($1.5\times10^{5}$ neutrones de 500 MeV por medio) frente a las predicciones de las secciones eficaces térmicas y a las tendencias de Sidelnik et al. (2020). Entre paréntesis, el cambio relativo respecto al agua pura.

|  | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Capturas en $^{1}$H | 99.94 % | 63.5 % | 47.5 % | 31.0 % |
| Capturas en $^{35}$Cl (este trabajo) | – | 35.8 % | 51.5 % | 67.7 % |
| teoría (secciones eficaces térmicas) | – | 28.0 % | 44.2 % | 62.2 % |
| Capturas en $^{23}$Na / $^{37}$Cl | – | 0.6 / 0.1 % | 0.8 / 0.2 % | 1.1 / 0.2 % |
| Tiempo de captura $\tau$ (µs), este trabajo | 192 | 140 ($-$27.0 %) | 109 ($-$43.2 %) | 75 ($-$61.0 %) |
| teoría, Ec. (4.1) (µs) | 205 | 147 ($-$28.0 %) | 114 ($-$44.2 %) | 78 ($-$62.1 %) |
| Energía del neutrón al capturarse, mediana (meV) | 30.5 | 30.6 | 31.0 | 31.7 |
| Energía gamma de captura por neutrón incidente (MeV) | 1.85 | 4.39 | 5.71 | 7.40 |

![Figura 4.1](figuras/fig05_tiempo_energia_captura.png)

**Figura 4.1:** Distribución del tiempo de captura (izquierda) y de la energía del neutrón en el momento de la captura (derecha; la línea punteada marca la energía térmica de referencia, 25.3 meV, $kT$ a 20 °C) para agua pura, agua + 2.5 % de NaCl, agua + 5 % de NaCl y agua + 10 % de NaCl.

**Distancia de máxima absorción.**  Sidelnik et al. (2020, Sec. 3, Fig. 2) reportan la distribución de capturas en función de la distancia desde el borde del tanque, con un máximo a 8.4 cm en agua pura. Medida a lo largo del haz desde la cara de entrada, la distribución de capturas del presente modelo es, en cambio, una joroba ancha con máximo cerca del centro del tanque (46.5 cm), porque el primario realiza su primera inelástica a 40 cm de profundidad en promedio y los neutrones de la cascada se capturan en todo el volumen (Figura 4.2). La forma de la Fig. 2 de Sidelnik et al., con un máximo cercano y una cola larga, aparece si la distancia se mide desde el punto donde nace el neutrón capturado (Tabla 4.3). Esto sugiere que la distancia reportada en el trabajo de referencia describe la escala de moderación. Con esta definición, la distancia disminuye con la concentración de NaCl, como en el trabajo de referencia.

**Tabla 4.3:** Distancia de máxima absorción con tres definiciones (máximo del histograma suavizado con bines de 0.5 cm), frente a la Fig. 2 de Sidelnik et al. (2020).

| **Distancia de máxima absorción (cm)** | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Cara de entrada $\to$ captura, a lo largo del haz | 46.5 | 45.3 | 36.1 | 47.8 |
| Vértice del neutrón capturado $\to$ captura | 11.1 | 10.0 ($-$10 %) | 9.8 ($-$12 %) | 9.6 ($-$14 %) |
| Inelástica del primario $\to$ captura | 12.7 | 11.3 | 11.2 | 10.4 |
| *Sidelnik et al.* | *8.4* | *8.0* ($-$5 %) | *7.7* ($-$8 %) | *7.3* ($-$13 %) |

![Figura 4.2](figuras/fig01_distancia_captura.png)

**Figura 4.2:** Izquierda: capturas en función de la distancia a lo largo del haz desde la cara de entrada. Derecha: distribuciones normalizadas con las tres definiciones de distancia de la Tabla 4.3, frente a la curva de agua pura digitalizada de la Fig. 2 de Sidelnik et al. (2020) (puntos).

### 5. Contribución de los principales núcleos a la captura de neutrones

La identificación de los núcleos que participan en los procesos de captura constituye un aspecto fundamental para comprender la modificación de la respuesta del WCD cuando el agua es dopada con NaCl. La captura conduce a la formación de un núcleo compuesto excitado que posteriormente se desexcita mediante la emisión de uno o varios fotones gamma. En el caso del agua pura, la absorción de neutrones ocurre principalmente en los núcleos de $^{1}$H y, en menor medida, en $^{16}$O. Cuando se incorpora NaCl al volumen detector aparecen nuevos núcleos capaces de participar en el proceso de captura, entre los cuales se encuentran $^{35}$Cl, $^{37}$Cl y $^{23}$Na. Se consideran los medios de agua pura (rojo), agua + 2.5 % de NaCl (verde), agua + 5 % de NaCl (violeta) y agua + 10 % de NaCl (naranja).

La Figura (5.1) presenta los gammas emitidos en cada captura, enlazados con el núcleo que capturó el neutrón. A diferencia de la versión anterior de esta figura, la intensidad se expresa en fotones por 100 capturas y se compara directamente con los valores de referencia: Firestone y Révay (2016) para $^{16}$O y la evaluación ENSDF (Nica et al., 2012) para $^{35}$Cl.

![Figura 5.1](figuras/fig15_espectros_captura_nucleo.png)

**Figura 5.1:** Espectros gamma producidos por la captura de neutrones en los núcleos presentes en el detector: (a) $^{16}\mathrm{O}(n,\gamma)^{17}\mathrm{O}$ en agua pura; (b) $^{1}\mathrm{H}(n,\gamma)^{2}\mathrm{H}$ con su ajuste gaussiano; (c) $^{35}\mathrm{Cl}(n,\gamma)^{36}\mathrm{Cl}$ y (d) $^{37}\mathrm{Cl}(n,\gamma)^{38}\mathrm{Cl}$ en agua + 2.5 % de NaCl. Intensidad en fotones por 100 capturas; los rombos indican los valores de referencia (Firestone y Révay, 2016; Nica et al., 2012) y las etiquetas, en keV, las energías de la Tabla 5.1. En magenta se indica el gamma único con toda la energía de separación del neutrón $S_n$ que emite Geant4 (véase el texto).

**Captura de neutrones en agua pura.** 

En agua pura, la captura radiativa de un neutrón por el hidrógeno,

$$ {}^{1}\mathrm{H}+n \longrightarrow {}^{2}\mathrm{H}+\gamma , \qquad (5.1) $$

produce la línea gamma característica de 2223.25 keV, utilizada por Firestone y R\'evay como línea de referencia para la normalización de sus medidas (Firestone y Révay, 2016, Sec. II.A, Tabla I, p. 2). Como se observa en la Figura 5.1(b), el máximo de la distribución simulada se encuentra en

$$ \mu = 2224.373~\mathrm{keV}, \qquad (5.2) $$

con una dispersión $\sigma \approx 0.013$ keV, del orden de la precisión con que se registran las energías en la salida de la simulación (0.01 keV). La línea simulada está 1.1 keV por encima del valor medido: Geant4 emite el gamma con $E_{\gamma}\approx S_{n}(^{2}\mathrm{H})=2224.57$ keV, sin descontar la energía de retroceso del deuterón (1.32 keV; Sec. 8). La diferencia es irrelevante para la producción de luz, pero debe tenerse en cuenta al comparar la posición de la línea.

El oxígeno constituye el segundo núcleo del agua que puede capturar neutrones. Para la reacción

$$ {}^{16}\mathrm{O}(n,\gamma){}^{17}\mathrm{O}, \qquad (5.3) $$

Firestone y R\'evay identifican cinco transiciones principales, con energías de 870.76, 1087.89, 2184.49, 3272.02 y 4142.60 keV (Firestone y Révay, 2016, Sec. II.B.1, Fig. 1, Tabla III, pp. 2–3), representadas en el esquema de niveles de la Figura (5.2). Entre corchetes se indican las intensidades obtenidas en la simulación, que coinciden con las de referencia dentro del error estadístico (190 capturas en $^{16}$O en los cuatro medios). La contribución de la captura en $^{16}$O es considerablemente menor que la del hidrógeno debido a su reducida sección eficaz radiativa térmica,

$$ \sigma_{0}(^{16}\mathrm{O}) = 0.170\pm0.003~\mathrm{mb}, \qquad (5.4) $$

reportada en la Sec. II.B.1 (Firestone y Révay, 2016, Sec. II.B.1, p. 3); la transición de 870.76 keV tiene una probabilidad absoluta de emisión $P_{\gamma}=96.6\pm0.5$ % y una sección eficaz de producción individual $\sigma_{\gamma}=0.164\pm0.003$ mb.

![Figura 5.2](figuras/fig18b_esquema_niveles_O17_MEIGA.png)

**Figura 5.2:** Esquema de niveles y principales transiciones $\gamma$ de la reacción $^{16}\mathrm{O}(n,\gamma){}^{17}\mathrm{O}$ con neutrones térmicos, elaborado a partir de los datos experimentales de Firestone y R\'evay (Sec. II.B.1, Fig. 1 y Tabla III) (Firestone y Révay, 2016) y contrastado con la base CapGam del NNDC (National Nuclear Data Center, 2025). Debajo de cada transición se indica, como «MEIGA», la intensidad por 100 capturas obtenida en la simulación; en magenta, el gamma único de energía $S_n$ que emite Geant4.

**Captura de neutrones en agua dopada con NaCl.** 

La incorporación de NaCl introduce canales adicionales de captura. Para los isótopos naturales del cloro,

$$ {}^{35}\mathrm{Cl}+n \longrightarrow {}^{36}\mathrm{Cl}+\gamma , \qquad
{}^{37}\mathrm{Cl}+n \longrightarrow {}^{38}\mathrm{Cl}+\gamma . \qquad (5.5) $$

A diferencia de la captura sobre hidrógeno, caracterizada por una única línea, la captura en $^{35}$Cl libera $Q=8.579$ MeV que se reparten en una cascada de tres a cuatro gammas distribuidos en un intervalo amplio de energías. La Figura (5.5) muestra el esquema de niveles de $^{36}$Cl con las quince transiciones más intensas (Nica et al., 2012) y, entre corchetes, las intensidades de la simulación en 214 748 capturas en $^{35}$Cl. Trece de las quince transiciones coinciden con la evaluación dentro de pocos puntos porcentuales, incluidas las primarias de 6110.98 keV (19.4 frente a 19.8 por 100 capturas) y 7414.09 keV (10.0 frente a 9.9) y la transición directa al estado fundamental de 8578.70 keV (2.6 frente a 2.7). Las excepciones son la línea de 1170.94 keV, casi ausente en la simulación (0.5 frente a 5.5), y la de 788.42 keV, algo baja.

\beginsidewaysfigure
\beginadjustwidth0cm0cm

\beginminipage\linewidth

\includegraphics[width=\linewidth,height=0.70\textwidth,keepaspectratio]fig19b_esquema_niveles_Cl36_MEIGA.pdf

\captionsetupwidth=\textwidth,justification=justified,singlelinecheck=false
\caption[Esquema de desexcitación gamma tras la captura de neutrones en $^{35}$Cl]Esquema de niveles de $^{36}$Cl y transiciones $\gamma$ más intensas de la captura $^{35}\mathrm{Cl}(n,\gamma)^{36}\mathrm{Cl}$ con neutrones térmicos (evaluación ENSDF (Nica et al., 2012); niveles sin escala). Intensidades por 100 capturas según la evaluación; entre corchetes, los valores de la simulación (214 748 capturas en $^{35}$Cl). En magenta, el gamma único de energía $S_n$ que emite Geant4.

\endminipage
\endadjustwidth
\endsidewaysfigure

La Tabla (5.1) reúne los parámetros nucleares de los isótopos del volumen activo, las energías de las líneas de captura y la línea correspondiente en la simulación. Respecto a la versión anterior de esta tabla se unificaron las energías con las de los esquemas de niveles: las del $^{16}$O siguen ahora a Firestone y R\'evay (Fig. 1), y el valor de 1959.13 keV del $^{35}$Cl, que mezclaba dos líneas reales, se separó en 1951.13 y 1959.34 keV.

**Tabla 5.1:** Parámetros nucleares de los principales isótopos del volumen activo y energías de las líneas $\gamma$ de captura de neutrones térmicos, contrastadas con la base CapGam del NNDC (National Nuclear Data Center, 2025), con Firestone y R\'evay para $^{16}$O (Firestone y Révay, 2016, Sec. II.B.1, Tabla III) y con la evaluación ENSDF para $^{35}$Cl (Nica et al., 2012). La última columna da la misma transición en la simulación y su intensidad por 100 capturas en ese isótopo (capturas en el agua de los cuatro medios: $^{1}$H: 316 177; $^{16}$O: 190; $^{35}$Cl: 214 748; $^{23}$Na: 3 483; $^{37}$Cl: 707). $\sigma_{0}$ es la sección eficaz total de captura radiativa térmica del isótopo y no la de producción de cada línea. En magenta, el gamma único con toda la energía de separación que emite Geant4, sin equivalente en la naturaleza.

| **Parámetros nucleares y líneas $\gamma$ de captura** |  |  |  |  |
|:--|:--|:--|:--|:--|
| **Isótopo** | **Abundancia** |  |  |  |
| isotópica (%) | **$\sigma_{0}$ (mb)** | **$E_{\gamma}$ (keV)** | **Simulación: $E_{\gamma}$ (keV)** |  |
| (por 100 capturas) |  |  |  |  |
| $^{1}$H | 99.99 | $332.60 \pm 0.70$ | 2223.25 | 2224.37  (100.0 %) |
| $^{16}$O | 99.76 | $0.170 \pm 0.003$ | 870.76 | 870.80  (94.2 %) |
|  |  |  | 1087.89 | 1088.00  (87.9 %) |
|  |  |  | 2184.49 | 2184.20  (84.7 %) |
|  |  |  | 3272.02 | 3272.20  (15.8 %) |
|  |  |  | 4142.60 | – |
|  | $\gamma$ único de $S_n$ (solo Geant4) | 4143.09  (21.6 %) |  |  |
| $^{35}$Cl | 75.78 | $43840 \pm 170$ | 517.07 | 517.08  (22.9 %) |
|  |  |  | 788.42 | 788.43  (15.2 %) |
|  |  |  | 1164.86 | 1164.87  (25.6 %) |
|  |  |  | 1951.13 | 1951.14  (18.2 %) |
|  |  |  | 1959.34 | 1959.36  (11.8 %) |
|  |  |  | 6110.98 | 6110.85  (19.4 %) |
|  |  |  | 8578.70 | 8578.59  (2.6 %) |
|  | $\gamma$ único de $S_n$ (solo Geant4) | 8579.81  (24.3 %) |  |  |
| $^{23}$Na | 100 | $527 \pm 7$ | 472.20 | 472.26  (0.6 %) |
|  | $\gamma$ único de $S_n$ (solo Geant4) | 6959.44  (11.4 %) |  |  |
| $^{37}$Cl | 24.22 | $553 \pm 17$ | 637.69 | 637.50  (11.5 %) |
|  |  |  | 755.43 | 755.47  (26.0 %) |
|  |  |  | 788.42 | – |
|  |  |  | 1692.11 | 1692.15  (19.9 %) |
|  |  |  | 2166.90 | – |
|  |  |  | 4134.13 | – |
|  | $\gamma$ único de $S_n$ (solo Geant4) | 6107.88  (28.1 %) |  |  |

**Limitación del modelo de captura de Geant4.**  Además de las transiciones reales, Geant4 emite en el 24.3 % de las capturas en $^{35}$Cl un único gamma de 8579.8 keV, igual a $S_{n}$ sin la corrección por retroceso del núcleo, distinto de la transición real al estado fundamental (8578.70 keV, 2.7 %, bien reproducida). El mismo comportamiento aparece en el $^{16}$O (21.6 % de las capturas con un único gamma de 4143.09 keV, mientras que la transición real de 4142.60 keV no aparece), en el $^{37}$Cl (28 %) y en el $^{23}$Na (11 %), cuya línea de 472.20 keV prácticamente no se produce. Que la fracción sea similar en núcleos distintos indica que el origen está en el código de captura de alta precisión (`G4NeutronHP`) y no en los datos nucleares. En el resto de las capturas los gammas de la cascada se sortean de forma independiente a partir de las tablas de producción de fotones de la evaluación, sin imponer la conservación de la energía en cada captura (Figura 5.3): las 46 355 capturas en $^{35}$Cl del medio con 2.5 % de NaCl emiten en promedio 2.80 gammas con una energía total media de 10.14 MeV (percentiles 5–95: 2.8–20.9 MeV), en lugar de 8.579 MeV; solo el 26 % de las cascadas suma $Q$ dentro del 1 %. Estos valores son los mismos en los tres medios con NaCl (10.12–10.14 MeV en promedio), como corresponde a una propiedad del modelo y no del medio. Las intensidades por captura de las líneas coinciden con la evaluación, pero la energía emitida por captura no se conserva. Geant4 permite usar en su lugar `G4PhotonEvaporation`, que genera cascadas que sí conservan la energía (variable de entorno `G4NEUTRONHP_USE_ONLY_PHOTONEVAPORATION`).

![Figura 5.3](figuras/fig12_cascadas_cloro_nacl25.png)

**Figura 5.3:** Energía total emitida por captura y multiplicidad de la cascada gamma en $^{35}$Cl, $^{23}$Na y $^{1}$H en agua + 2.5 % de NaCl. La línea vertical indica la energía de separación del neutrón, $Q=8.579$ MeV, en $^{36}$Cl.

**Implicaciones para la respuesta Cherenkov del WCD.**  La cadena de procesos que conduce de la captura a la señal es

$$ \begin{gathered}n + {}^{A}\mathrm{X} \longrightarrow {}^{A+1}\mathrm{X}^{*} \longrightarrow {}^{A+1}\mathrm{X} + \sum_{k}\gamma_{k}, \\
\gamma_{k} + e^{-}_{\mathrm{at}} \longrightarrow e^{-} \;(\text{Compton, fotoeléctrico, pares}) \longrightarrow \gamma_{\mathrm{Ch}} .\end{gathered} \qquad (5.6–5.7) $$

La energía gamma de captura por neutrón incidente pasa de 1.85 MeV en agua pura a 4.39 MeV (2.5 %), 5.71 MeV (5 %) y 7.40 MeV (10 %), es decir, se multiplica por 2.4, 3.1 y 4.0 (Tabla 4.2). Este es el origen del aumento de la luz Cherenkov con la concentración de NaCl. Debido a la limitación descrita, en los medios con NaCl la energía de las capturas en Cl está sobreestimada en promedio en un 18 %, lo que afecta a la escala absoluta de la luz y de la carga de esos medios, aunque no al signo de las tendencias.

### 6. Comparación de los espectros de fotones gamma generados por captura de neutrones

La Figura (6.1) compara los espectros de energía de los fotones $\gamma$ creados en el agua después de la interacción de neutrones monocromáticos de 500 MeV. La subfigura (a) corresponde al espectro reportado por Sidelnik et al. (2020, Fig. 3, pp. 2219–2220); la subfigura (b), al espectro obtenido en este trabajo con el mismo formato.

![Figura 6.1](informe/Figuras/espectro-gamma-ivan.png)

*(a) Trabajo previo*

**Figura 6.1:** Comparación entre los espectros de radiación $\gamma$ creados en el agua por neutrones de 500 MeV. (a) Fig. 3 de Sidelnik et al. (2020) para agua pura (rojo), agua + 2.5 % de NaCl (verde), agua + 5 % de NaCl (violeta) y agua + 10 % de NaCl (naranja). (b) Presente trabajo en agua pura: espectro con el mismo formato, comparación con la curva digitalizada con el pico de 2.223 MeV igualado, composición por proceso creador y cocientes de líneas.

![Figura 6.1](figuras/fig11_espectro_gamma_tipo_sidelnik.png)

*(b) Presente trabajo*

**Figura 6.1:** (Continuación) (b) Presente trabajo; véase la descripción en la parte (a) de esta figura.

**Normalización.**  El eje vertical de la Fig. 3 del trabajo de referencia está en unidades arbitrarias: su línea de 2.223 MeV contiene unas $4.2\times10^{5}$ cuentas, más que el número de neutrones simulados ($1.5\times10^{5}$). En la presente simulación la línea contiene 124 397 cuentas, una por captura en H. Por ello la comparación se realiza con magnitudes que no dependen de la normalización: cocientes entre alturas de líneas y la forma del continuo con el pico de 2.223 MeV igualado. Con bines de 2 keV el cociente pico/continuo entre 3 y 5 MeV (1 777–3 659) coincide con el del artículo (1 946–3 075).

La Tabla (6.1) resume la producción de gammas y de electrones en el agua. La Tabla (6.2) compara los cocientes de las líneas principales en agua pura.

**Tabla 6.1:** Gammas y electrones creados dentro del agua por neutrón incidente de 500 MeV.

|  | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Gammas creados en el agua | 5.98 | 7.03 | 7.68 | 8.65 |
| de inelásticas de neutrones | 2.63 | 2.70 | 2.77 | 2.93 |
| de captura | 0.83 | 1.43 | 1.76 | 2.19 |
| de bremsstrahlung / aniquilación | 1.23 / 0.67 | 1.42 / 0.76 | 1.56 / 0.82 | 1.74 / 0.90 |
| Energía gamma de captura (MeV) | 1.855 | 4.393 | 5.713 | 7.404 |
| e$^{\pm}$ creados por gammas | 44.1 | 51.4 | 55.8 | 62.2 |
| Compton / fotoeléctrico | 37.1 / 6.80 | 41.4 / 9.57 | 43.6 / 11.74 | 46.3 / 15.25 |
| fracción sobre el umbral Cherenkov (0.264 MeV) | 8.0 % | 8.3 % | 8.5 % | 8.7 % |

**Tabla 6.2:** Cocientes entre las alturas de las líneas gamma principales en agua pura. $^{*}$Estimado restando la fracción de positrones procedentes de desintegraciones $\beta^{+}$ de larga vida.

| **Cociente de alturas** | **Sidelnik et al.** | **Este trabajo** | **Este trabajo, sin** |
|:--|:--|:--|:--|
| decaimiento radiactivo |  |  |  |
| 0.511 MeV / 2.223 MeV | 0.128 | 0.763 | 0.344$^{*}$ |
| 6.13 MeV / 2.223 MeV | 0.022 | 0.029 | 0.022 |

**Líneas de 2.223 y 6.13 MeV.**  La línea de 6.13 MeV del $^{16}$O la producen la dispersión inelástica de neutrones y la desintegración del $^{16}$N ($T_{1/2}=7.1$ s), producido por la reacción $^{16}\mathrm{O}(n,p)$. Sin esta última, el cociente con la línea de captura en H es 0.022, igual al del trabajo de referencia: la captura en H y la dispersión inelástica en O se reproducen con la misma proporción.

**Línea de aniquilación (0.511 MeV).**  En la presente simulación es unas seis veces más intensa, relativa a la línea de 2.223 MeV, que en el trabajo de referencia. Los positrones que la producen proceden de desintegraciones $\beta^{+}$ de núcleos de espalación del oxígeno ($^{15}$O, $^{13}$N y $^{11}$C, con vidas medias de minutos; 55 %), de la conversión de gammas de alta energía en pares (39 %) y de la desintegración de muones positivos procedentes de $\pi^{+}$ (6 %). Geant4 simula las desintegraciones radiactivas sin límite de tiempo; sin ellas, el cociente bajaría a 0.34, y el exceso restante es coherente con la mayor producción de piones discutida en la Sec. 9.

**Continuo.**  Entre 2.5 y 9 MeV la forma del continuo coincide con la del trabajo de referencia (cola de la dispersión inelástica de neutrones). Por debajo de 1.5 MeV el continuo simulado es varias veces más alto, dominado por el bremsstrahlung de electrones y positrones de alta energía asociados a la producción de piones.

**Líneas del cloro.**  La Tabla (6.3) compara las alturas de las líneas respecto a la de 2.223 MeV en los medios con NaCl. Las líneas de la cascada del $^{36}$Cl de 1.165, 6.11, 7.41 y 7.79 MeV coinciden con el trabajo de referencia dentro del 2–10 % en las tres concentraciones: la proporción de capturas en Cl respecto a H y la intensidad de esas transiciones son las mismas en las dos simulaciones. Las diferencias se concentran en la línea de 0.511 MeV, por las causas ya descritas, y en la de 8.58 MeV, unas diez veces más intensa en este trabajo por el gamma único de $S_n$ de Geant4.

**Tabla 6.3:** Altura de las líneas gamma respecto a la de 2.223 MeV en los medios con NaCl, frente a la Fig. 3 de Sidelnik et al. (2020) digitalizada. La digitalización (25.6 keV por píxel) no separa las líneas de 0.511/0.517 MeV ni las de 6.11/6.13 MeV.

|  | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |  |  |  |
|:--|:--|:--|:--|:--|:--|:--|
| **Línea (MeV) / 2.223 MeV** | **Sidelnik** | **Este trabajo** | **Sidelnik** | **Este trabajo** | **Sidelnik** | **Este trabajo** |
| 0.511 / 0.517 | 0.423 | 1.305 | 0.678 | 1.788 | 1.181 | 2.796 |
| 1.165 (Cl) | 0.165 | 0.147 | 0.312 | 0.277 | 0.574 | 0.561 |
| 1.951 (Cl) | 0.179 | 0.104 | 0.339 | 0.200 | 0.624 | 0.397 |
| 6.11 / 6.13 | 0.115 | 0.109 | 0.217 | 0.212 | 0.412 | 0.425 |
| 7.41 (Cl) | 0.059 | 0.057 | 0.112 | 0.107 | 0.206 | 0.216 |
| 7.79 (Cl) | 0.047 | 0.046 | 0.089 | 0.084 | 0.165 | 0.172 |
| 8.58 (Cl) | 0.016 | 0.151 | 0.029 | 0.292 | 0.054 | 0.583 |

La Figura (6.2) muestra el espectro gamma completo en agua + 2.5 % de NaCl, con el mismo formato y los mismos cuatro paneles que el del agua pura (Figura 6.1). Respecto al agua pura aparecen las líneas de la cascada del $^{36}$Cl entre 0.5 y 8.58 MeV (panel a), y la componente de captura ya no se limita a la línea de 2.223 MeV del H, sino que se extiende hasta la energía de separación del neutrón en $^{36}$Cl (panel c); el resto de las componentes (inelásticas de neutrones y protones, bremsstrahlung, aniquilación y decaimiento radiactivo) no cambia con la sal. Para la comparación con la curva digitalizada se iguala de nuevo el pico de 2.223 MeV (panel b); el factor necesario, 3.38, es prácticamente el mismo que en agua pura (3.40), lo que confirma que se trata de la normalización arbitraria del eje del artículo y no de una diferencia física. Con ese factor, las líneas del Cl aparecen en las mismas energías y con alturas relativas comparables; entre 2.5 y 9 MeV la curva digitalizada queda algo por encima del continuo simulado, porque a la resolución de la digitalización (25.6 keV por píxel) las numerosas líneas del Cl se confunden con el continuo. Por encima de 10 MeV ambos espectros quedan en el nivel de unas pocas cuentas. El panel (d) resume los cocientes de la Tabla (6.3): las líneas de 1.165, 6.11, 7.41 y 7.79 MeV coinciden con el artículo dentro del 2–10 %, la de 1.951 MeV queda por debajo porque en la figura del artículo se superpone con la de 1.959 MeV, la de 0.511 MeV es unas tres veces más intensa por los emisores $\beta^{+}$ de larga vida y los piones, y la de 8.58 MeV es unas diez veces más intensa por el gamma único de $S_n$ que emite Geant4. Los espectros de 5 y 10 % de NaCl tienen la misma estructura y escalan con la fracción de capturas en Cl.

![Figura 6.2](figuras/fig11_espectro_gamma_tipo_sidelnik_nacl25.png)

**Figura 6.2:** Espectro de energía de los gammas creados en agua + 2.5 % de NaCl por neutrones de 500 MeV: (a) formato de la Fig. 3 de Sidelnik et al. (2020) (intervalos de 2 keV, $1.5\times10^{5}$ neutrones); (b) comparación con la curva digitalizada del artículo, con el pico de 2.223 MeV igualado; (c) composición por proceso creador; (d) cocientes de las alturas de las líneas principales respecto a la de 2.223 MeV (Tabla 6.3).

**Espectro acumulado.**  Sidelnik et al. (2020, Fig. 4, p. 2220) presentan los mismos gammas acumulados de derecha a izquierda, $N(\geq E)$. La Figura (6.3) muestra el espectro completo de los cuatro medios y su acumulado, y la Figura (6.4) la comparación con la Fig. 4 digitalizada. Tres cocientes, independientes de la normalización, resumen la comparación:


- **Línea de 2.223 MeV frente a los gammas de más de 3 MeV**, $[N(\geq2)-N(\geq3)]/N(\geq3)$: 2.32 en el artículo y 1.27 en este trabajo. Por cada captura en H, la presente simulación produce $\approx$ 1.6–1.8 veces más gammas de desexcitación nuclear tras la dispersión inelástica de neutrones, conclusión que coincide con la obtenida del espectro de electrones (Sec. 7).

- **Gammas de más de 20 MeV**, $N(\geq20)/N(\geq3)$: 0.029 en el artículo, 0.163 en este trabajo y 0.0031 sin los eventos con piones. Los gammas de alta energía proceden casi exclusivamente de los piones ($\pi^{0}\to\gamma\gamma$ y bremsstrahlung de sus cascadas); el trabajo de referencia queda entre las dos curvas, con unas seis veces menos piones en este observable.

- **Efecto del NaCl**, $N(\geq E)_{\mathrm{NaCl}}/N(\geq E)_{\mathrm{pura}}$: entre 3 y 6 MeV vale 1.46–1.58 en este trabajo y 1.91–2.01 en el artículo (el fondo mayor de la simulación diluye la contribución del Cl); a partir de 9 MeV ambos valen 1: la sal no modifica los gammas de alta energía.

![Figura 6.3](figuras/fig02_espectro_gamma.png)

**Figura 6.3:** Espectro de energía de los gammas creados en el agua (izquierda, formato de la Fig. 3 de Sidelnik et al. (2020)) y cuentas acumuladas de derecha a izquierda $N(\geq E)$ (derecha, formato de su Fig. 4) para agua pura, agua + 2.5 % de NaCl, agua + 5 % de NaCl y agua + 10 % de NaCl.

**Exceso de cuentas acumuladas entre 2.2 y 8.6 MeV.**  En la Figura (6.3, derecha) las curvas con NaCl presentan una protuberancia entre 2.2 y 8.6 MeV respecto al agua pura. $N(\geq E)$ cuenta, por neutrón, los gammas de energía mayor o igual que $E$: cada línea produce un escalón en su energía y los gammas comprendidos entre dos energías forman la meseta que las separa. En agua pura casi todos los gammas de captura proceden del H y tienen 2.22 MeV, de modo que por encima de ese escalón no queda ninguno. Con NaCl, el $^{35}$Cl emite su cascada entre 2.3 y 8.58 MeV: esos gammas forman una meseta adicional que termina en un escalón abrupto en 8.58 MeV, la energía de separación del neutrón en $^{36}$Cl, por encima de la cual no puede haber gammas de captura. La Tabla (6.4) muestra que la captura explica el 96 % (2.5 %), el 94 % (5 %) y el 93 % (10 %) del exceso a 4 MeV, mientras que los gammas de las inelásticas apenas cambian con la sal. El escalón mayor, en 8.58 MeV, corresponde al 27 % de las capturas en $^{35}$Cl y es consecuencia del gamma único de $S_n$; con cascadas realistas sería mucho menor y la meseta contendría más gammas de menor energía. Por encima de 8.7 MeV las curvas coinciden, porque solo quedan gammas de las inelásticas, del bremsstrahlung y de las desintegraciones de $\pi^{0}$ y muones, que no dependen de la sal.

**Tabla 6.4:** Descomposición de las cuentas acumuladas $N(\geq E)$ de la Figura 6.3 (derecha) por proceso creador. Gammas creados en el agua por neutrón incidente.

|  | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| $N(\geq 4~\mathrm{MeV})$ total | 0.694 | 1.041 | 1.222 | 1.450 |
| de captura | 0.000 | 0.332 | 0.498 | 0.705 |
| de inelásticas de neutrones | 0.418 | 0.428 | 0.437 | 0.458 |
| Exceso sobre el agua pura en $N(\geq 4~\mathrm{MeV})$ | – | 0.346 | 0.528 | 0.756 |
| fracción del exceso debida a la captura | – | 96 % | 94 % | 93 % |
| Gammas de captura entre 2.3 y 8.7 MeV | 0.000 | 0.463 | 0.694 | 0.984 |
| por captura en $^{35}$Cl | – | 1.50 | 1.49 | 1.50 |
| Escalón de 8.58 MeV (gamma único) | 0.000 | 0.083 | 0.126 | 0.176 |
| fracción de las capturas en $^{35}$Cl | – | 27 % | 27 % | 27 % |
| $N(\geq 8.7~\mathrm{MeV})$ total | 0.216 | 0.216 | 0.219 | 0.217 |

![Figura 6.4](figuras/fig17_gammas_acumulados.png)

**Figura 6.4:** Gammas creados en el medio acumulados de derecha a izquierda, en el formato de la Fig. 4 de Sidelnik et al. (2020): (a) hasta 100 MeV y (b) por debajo de 10 MeV, este trabajo (símbolos llenos) y artículo (símbolos vacíos), cuentas por $1.5\times10^{5}$ neutrones; (c) forma normalizada a $N(\geq3~\mathrm{MeV})$, con y sin los eventos con piones; (d) cociente NaCl / agua pura.

### 7. Comparación de los espectros de electrones y positrones secundarios

Los fotones gamma transfieren su energía a electrones y positrones del medio por efecto Compton, efecto fotoeléctrico y producción de pares; solo los que superan el umbral Cherenkov (0.264 MeV de energía cinética en agua) emiten luz. La Figura (7.1) compara el espectro de los electrones producidos por gammas con la Fig. 5 de Sidelnik et al. (2020, Sec. 3, Fig. 5, p. 2221).

![Figura 7.1](informe/Figuras/espectro-e-ivan.png)

*(a) Trabajo previo*

**Figura 7.1:** Espectros de energía de los electrones producidos por gammas en el agua. (a) Fig. 5 de Sidelnik et al. (2020). (b) Presente trabajo: espectro con el mismo formato (bines de 20 keV, $1.5\times10^{5}$ neutrones), comparación con la curva digitalizada y los bordes Compton, cociente NaCl / agua pura y saltos en los bordes Compton.

![Figura 7.1](figuras/fig13_espectro_electrones_tipo_sidelnik.png)

*(b) Presente trabajo*

**Figura 7.1:** (Continuación) (b) Presente trabajo; véase la descripción en la parte (a) de esta figura.

**Escala absoluta.**  A diferencia de los espectros gamma y de fotones, aquí la comparación no requiere ninguna normalización: ambas figuras representan electrones por $1.5\times10^{5}$ neutrones con intervalos de 20 keV. El factor $k$ del panel (b) de la Figura 7.1 es la mediana del cociente Sidelnik/presente trabajo en 11 energías entre 0.5 y 7 MeV: $k=1.06$ (agua pura), 1.09 (2.5 %), 1.07 (5 %) y 1.04 (10 %). Es decir, la presente simulación produce entre un 4 y un 8 % menos de electrones que la del artículo, con la misma forma desde 0.3 hasta 10 MeV. Este es el resultado de validación más directo de la cadena gamma $\to$ electrón. El factor es exclusivo de los electrones: el factor 3.40 del espectro gamma (Figura 6.1) es una normalización, porque el eje de la Fig. 3 del artículo está en unidades arbitrarias.

**Origen de la diferencia de amplitud.**  Se identifican cuatro causas.


- **Probabilidad de captura.** Es la causa principal: el cociente entre las probabilidades de captura del artículo y de la presente simulación (Tabla 4.1) es 1.05, 1.09, 1.10 y 1.12 para los cuatro medios, prácticamente igual a $k$ en agua pura y con 2.5 % de NaCl. Con 5 y 10 % el factor $k$ queda por debajo de ese cociente porque cada captura en $^{35}$Cl emite en la simulación más energía que la real (10.1 frente a 8.58 MeV; Sec. 5) y genera más electrones, lo que compensa en parte las capturas faltantes; el efecto crece con la concentración.

- **Diferencias de forma.** El factor $k$ es un promedio: punto a punto el cociente varía entre 0.85 y 1.35 en agua pura. El artículo está por encima justo debajo del borde Compton de 1.994 MeV y la simulación entre 2 y 3 MeV, donde domina el fondo de desexcitación nuclear; con NaCl, entre 7.4 y 8.3 MeV la simulación queda muy por encima por el gamma único de 8.58 MeV.

- **Digitalización.** La Fig. 5 del artículo se digitalizó por color en escala logarítmica, con una incertidumbre del 5–10 % en las cuentas.

- **Modelos de simulación.** Las versiones de Geant4 y las listas físicas son distintas, y el artículo no las especifica.

La diferencia de amplitud es, por tanto, pequeña y se explica casi por completo por la probabilidad de captura; las diferencias físicamente relevantes están en la forma del espectro (fondo de desexcitación nuclear y gamma único de 8.58 MeV), que se analizan a continuación.

**Bordes Compton.**  En ambas simulaciones se identifican los bordes Compton de la línea de aniquilación (0.34 MeV), de la captura en H (1.994 MeV) y de la línea de 6.13/6.11 MeV (5.87 MeV), cuya energía es

$$ T_{\max} = \frac{2E_{\gamma}^{2}}{m_{e}c^{2}+2E_{\gamma}} . \qquad (7.1) $$

El salto en el borde de 1.994 MeV es 4.5 en este trabajo y 6.5 en el artículo; el de 5.87 MeV, 1.74 y 1.63.

**Interpretación de las diferencias.**  Para entender las diferencias se clasificó cada electrón según el origen del gamma que lo produce (Figura 7.2). Por debajo del borde de 1.994 MeV el 76 % de los electrones procede de la línea de 2.223 MeV; por encima, el 78 % procede de gammas de la dispersión inelástica de neutrones en el oxígeno (desexcitación de $^{16}$O$^{*}$, $^{12}$C$^{*}$ y $^{15}$N$^{*}$). El salto vale por tanto $J\approx1+C_{\mathrm{H}}/F$, donde $C_{\mathrm{H}}$ son los electrones de la captura en H y $F$ ese fondo. Para obtener el salto del artículo habría que dividir el fondo de la simulación por 1.60 en agua pura y por 1.76 con 2.5 % de NaCl, un factor similar en ambos medios, como corresponde a un fondo que no depende de la sal. Es decir, por cada captura en H la presente simulación produce $\approx$ 1.6 veces más gammas de desexcitación nuclear, en acuerdo con el espectro gamma acumulado. Los piones no son la causa (sin ellos el salto pasa de 4.46 a 4.58); la diferencia depende del modelo de cascada y de la desexcitación de los núcleos residuales, que cambiaron entre versiones de Geant4.

**Efecto del NaCl.**  El cociente NaCl / agua pura no depende de la normalización. Por debajo del borde de 1.994 MeV vale $\approx$ 0.84 en las dos simulaciones, porque el Cl compite con el H y las capturas en H por neutrón caen de 0.83 a 0.55; que el valor coincida indica que la competencia H/Cl es la misma en ambas. Entre 2 y 6 MeV, donde contribuyen los electrones Compton de las líneas del Cl, el cociente vale 1.42–1.96 en este trabajo y 1.63–2.19 en el artículo: el fondo más alto diluye la contribución del Cl. Entre 7 y 8.3 MeV el cociente simulado (2.46) supera al del artículo (1.58) por los electrones Compton del gamma único de 8.58 MeV. Las intensidades de las líneas intermedias del Cl coinciden con el artículo porque las dos limitaciones de Geant4 se compensan: hay $\approx$ 1.27 veces más capturas en Cl, pero solo el $\approx$ 76 % sigue una cascada ($1.27\times0.76\approx0.97$).

![Figura 7.2](figuras/fig14_origen_electrones.png)

**Figura 7.2:** Origen del gamma que produce cada electrón en agua pura y agua + 2.5 % de NaCl (áreas apiladas), y cociente NaCl / agua pura en cuatro intervalos de energía: Sidelnik et al. (2020), este trabajo y este trabajo con el fondo no debido a capturas dividido por el factor que reproduce el salto en el borde de 1.994 MeV del artículo.

### 8. Distribución de energía de los deuterones producidos en la captura de neutrones sobre hidrógeno

En la captura $n+p\to d+\gamma$ el deuterón retrocede con una energía

$$ E_{d} \simeq \frac{E_{\gamma}^{2}}{2M_{d}c^{2}} = \frac{(2.2244~\mathrm{MeV})^{2}}{2\times1875.6~\mathrm{MeV}} = 1.319~\mathrm{keV}. \qquad (8.1) $$

La distribución simulada (Figura 8.1) tiene su máximo exactamente en ese valor (mediana 1.319 keV) y una anchura $\sigma=9.9$ eV, originada en el movimiento térmico del neutrón y del protón en el instante de la captura, la misma causa que da a la línea de 2.224 MeV su anchura. Con NaCl el pico desciende en proporción a la fracción de capturas en H (64, 48 y 31 % con 2.5, 5 y 10 % de NaCl); su posición no cambia (1.319 keV) y su anchura crece (10.5, 10.8 y 11.6 eV). La media y el valor cuadrático medio de la distribución no describen el pico: los dominan unos pocos deuterones (0.12 %) procedentes de capturas de neutrones aún rápidos, con energías de hasta 6.9 MeV (media 1.43 keV y RMS 23.2 keV en agua pura). 

![Figura 8.1](figuras/fig16_deuterones_captura.png)

**Figura 8.1:** Energía cinética de los deuterones producidos en la captura $^{1}\mathrm{H}(n,\gamma)^{2}\mathrm{H}$ para los cuatro medios; cuentas por $1.5\times10^{5}$ neutrones primarios.

### 9. Comparación de la distribución de fotones Cherenkov detectados para neutrones de 500 MeV

Sidelnik et al. (2020, Sec. 3, Fig. 6, p. 2221) presentan la distribución del número de fotones Cherenkov aceptados por la eficiencia cuántica del PMT. Para comparar con esa figura se utilizó el mismo observable: para cada neutrón se cuentan los fotones Cherenkov aceptados por la eficiencia cuántica del R5912 en el momento de crearse, antes del transporte óptico (en el modo `eFast` del simulador son exactamente los fotones que se transportan); se descartan los eventos sin ningún fotón aceptado (39 % de los neutrones), porque la figura de referencia no muestra el pico en cero; y se histograma con bines de 10 fotones entre 0 y 2500 fotones, en escala logarítmica y en cuentas absolutas, sin normalizar ni reescalar. La Figura (9.1) muestra el resultado.

![Figura 9.1](figuras/fig10_figura6_sidelnik_vs_meiga.png)

**Figura 9.1:** Distribución del número de fotones Cherenkov aceptados por la eficiencia cuántica del PMT para neutrones de 500 MeV. Izquierda: Fig. 6 de Sidelnik et al. (2020), digitalizada. Derecha: la misma figura construida con el presente modelo ($1.5\times10^{5}$ neutrones por medio) para los cuatro medios, con la curva del artículo superpuesta y los eventos de agua pura separados según haya o no producción de piones.

**Coincidencia hasta 500 fotones.**  Entre 0 y 500 fotones la distribución simulada coincide con la del trabajo de referencia sin ningún reescalamiento: la pendiente es la misma, el quiebre aparece en 400–500 fotones y las cuentas absolutas coinciden dentro del 6–23 % (Tabla 9.1). La correlación de la forma en escala logarítmica entre 0 y 1000 fotones es 0.985. El número de neutrones, la eficiencia cuántica y la producción de luz Cherenkov son, por tanto, comparables en las dos simulaciones.

**Tabla 9.1:** Cuentas absolutas por intervalo de fotones aceptados en agua pura (eventos con al menos un fotón) y fracción de eventos con producción de piones en alguna interacción inelástica de un neutrón. Un mismo evento puede tener piones cargados y neutros.

| **Fotones aceptados** | **Este trabajo** | **Sidelnik et al.** |  |  |  |  |
|:--|:--|:--|:--|:--|:--|:--|
| (digitalizada) | **Razón** | **Con $\pi^{\pm}$** | **Con $\pi^{0}$** | **Con algún pion** |  |  |
| 1 – 100 | 52 395 | 67 941 | 0.77 | 4.0 % | 1.5 % | 5.5 % |
| 100 – 500 | 28 902 | 30 773 | 0.94 | 13.5 % | 2.6 % | 16.1 % |
| 500 – 1000 | 3 748 | 1 758 | 2.13 | 68.6 % | 26.5 % | 95.1 % |
| 1000 – 2500 | 4 783 | 1 495 | 3.20 | 35.5 % | 63.1 % | 98.6 % |
| $\geq$ 2500 | 1 707 | – | – | 9.7 % | 89.3 % | 98.8 % |

**La cola por encima de 500 fotones: producción de piones.**  Por encima de 500 fotones la simulación presenta entre dos y tres veces más eventos que el trabajo de referencia. Solo el 11.9 % de los neutrones produce piones, pero esos eventos forman la cola: el 97.3 % de los eventos con 500 fotones o más contiene piones. Un evento con piones produce en promedio 944 fotones aceptados, frente a 49 en uno sin piones. Los $\pi^{\pm}$ emiten luz Cherenkov a lo largo de su trayectoria y, al desintegrarse, producen muones que también la emiten; los $\pi^{0}$ se desintegran en dos gammas de $\approx$ 70 MeV que inician cascadas electromagnéticas. Sin los eventos con piones, la distribución cae a cero hacia 600 fotones. La energía de 500 MeV está apenas por encima del umbral de producción de piones en colisiones nucleón–nucleón ($\approx$ 290 MeV), donde la sección eficaz depende fuertemente del modelo hadrónico; la explicación más probable es que la simulación de referencia usara otra lista física u otra versión de Geant4, que el artículo no especifica. La cola no afecta al intervalo de 0 a 500 fotones, que corresponde a la moderación y la captura.

**Efecto del NaCl en la luz Cherenkov.**  Con el mismo observable, el presente modelo reproduce el efecto del NaCl de la Fig. 6 del artículo: las curvas con NaCl quedan por encima de la de agua pura, más cuanto mayor es la concentración, entre $\approx$ 150 y 550 fotones, por la luz adicional de las capturas en Cl; por encima de 600 fotones las curvas coinciden (piones). Entre 100 y 500 fotones las cuentas aumentan un 30.1 % (2.5 %), un 42.8 % (5 %) y un 59.2 % (10 %) en este trabajo, frente al 52.9 %, 66.9 % y 53.9 % del artículo, y la correlación de forma con la curva correspondiente de Sidelnik et al. es 0.989, 0.994 y 0.982. En la presente simulación el aumento es monótono con la concentración; en la figura digitalizada del artículo, en cambio, la curva de 10 % no supera a la de 5 % en este intervalo. Los fotones Cherenkov creados por neutrón pasan de 1 441 (agua pura) a 1 662 (2.5 %), 1 806 (5 %) y 1 994 (10 %).

**Contribución de los piones al déficit del efecto del NaCl.**  Para cuantificar qué parte de la diferencia con el artículo se debe a la mayor producción de piones, se marcó cada neutrón cuyo árbol de interacciones contiene una inelástica con producción de $\pi^{\pm}$ o $\pi^{0}$. Estos eventos son el 11.9–12.1 % en los cuatro medios. Al excluirlos, el aumento de cuentas entre 100 y 500 fotones pasa de 30.1 a 33.5 % (2.5 %), de 42.8 a 48.8 % (5 %) y de 59.2 a 68.7 % (10 %), frente a 52.9, 66.9 y 53.9 % en el artículo (Figura 9.2). Los piones explican, por tanto, solo unos 3 y 6 de los 23 y 24 puntos de diferencia con 2.5 y 5 % de NaCl: el exceso de cuentas que aporta la sal casi no cambia al excluirlos, y lo que se modifica es el fondo del agua pura con el que se compara. El resto del déficit se atribuye al fondo de desexcitación nuclear tras las interacciones inelásticas, $\approx$ 1.6 veces mayor por captura en H en la presente simulación (Sec. 7), y al menor aumento de la probabilidad de captura con la sal (Sec. 4). Ambos dependen del modelo hadrónico y de los datos de captura empleados, y su verificación requiere corridas adicionales con otras listas físicas. Con 10 % de NaCl la simulación ya supera al artículo, y la diferencia crece al excluir los piones, lo que es consistente con la anomalía de la curva de 10 % del artículo, que apenas supera a la de 5 %.

![Figura 9.2](figuras/fig24_piones_efecto_nacl.png)

**Figura 9.2:** Efecto del NaCl en la distribución de fotones Cherenkov aceptados, con y sin los eventos con producción de piones. (a) Razón NaCl/agua pura para todos los eventos (línea continua) y sin los eventos con piones (línea discontinua, hasta 520 fotones). (b) Aumento de cuentas entre 100 y 500 fotones respecto al agua pura: presente trabajo con todos los eventos y sin los eventos con piones, frente a la Fig. 6 de Sidelnik et al. (2020).

### 10. Validación cuantitativa de los histogramas de carga

Con el propósito de cuantificar la concordancia del efecto del NaCl, se calculó la razón entre el histograma de fotones aceptados de cada solución salina y el del agua pura, tanto para la presente simulación como para las curvas digitalizadas de la Fig. 6 de Sidelnik et al. (2020). Como ambos conjuntos corresponden al mismo número de neutrones ($1.5\times10^{5}$ por medio), la razón se calcula con cuentas absolutas, sin normalizar por el área. Para reducir las fluctuaciones de Poisson los histogramas se agruparon en bines de 50 fotones, y se excluyó el primer bin (0–50 fotones), afectado por la digitalización. La Figura (10.1) muestra las curvas de razón.

![Figura 10.1](figuras/fig22_razon_histogramas.png)

**Figura 10.1:** Razón entre los histogramas de fotones Cherenkov aceptados por la eficiencia cuántica de cada solución salina y del agua pura, en bines de 50 fotones: presente trabajo (puntos con barras de error estadístico) y Fig. 6 de Sidelnik et al. (2020) digitalizada (líneas discontinuas), para agua + 2.5 % de NaCl, agua + 5 % de NaCl y agua + 10 % de NaCl.

Las curvas de los dos trabajos presentan las mismas características:


- Para un número reducido de fotones ($N<150$) la razón permanece por debajo de la unidad: la captura en Cl desplaza eventos desde la región de baja señal hacia valores mayores.

- Entre 150 y 400 fotones la razón crece para todas las concentraciones, manteniendo el orden 10 % $>$ 5 % $>$ 2.5 %, como consecuencia de la luz adicional de las capturas en $^{35}$Cl.

- El máximo aparece entre 375 y 475 fotones en este trabajo y entre 325 y 425 fotones en el artículo, con valores de 2.72, 3.58 y 4.74 (este trabajo) frente a 2.44, 3.17 y 4.16 (Sidelnik et al.) para 2.5, 5 y 10 % de NaCl. En ambos trabajos el máximo se desplaza hacia más fotones al aumentar la concentración.

- Por encima de $\approx$ 700 fotones las razones tienden a la unidad: la cola está dominada por los eventos con piones, que no dependen de la sal.

- La razón media entre 100 y 500 fotones coincide dentro del 1 % para 2.5 y 5 % de NaCl: 1.92 frente a 1.93 (2.5 %) y 2.36 frente a 2.37 (5 %). Con 10 % es 3.02 en este trabajo y 2.39 en el artículo, donde la curva digitalizada de 10 % apenas supera a la de 5 % en este intervalo.

**Análisis cuantitativo de las curvas de razón.**  La concordancia se evaluó con métricas de correlación (coeficientes de Pearson $r$ y de Spearman $\rho$) y de error (RMSE y MAE) entre las curvas de razón de los dos trabajos, en los 19 bines entre 50 y 1000 fotones (Tabla 10.1).

**Tabla 10.1:** 

| **NaCl** | $\mathbf{r}$ | $\mathbf{p}$ | $\boldsymbol{\rho}$ | **RMSE** | **MAE** | **Razón media** |  |
|:--|:--|:--|:--|:--|:--|:--|:--|
| 100–500 | **Nivel** |  |  |  |  |  |  |
| 2.5 % | 0.881 | $6.2\times10^{-7}$ | 0.863 | 0.276 | 0.234 | 1.92 / 1.93 | Alta |
| 5 % | 0.906 | $9.1\times10^{-8}$ | 0.867 | 0.383 | 0.325 | 2.36 / 2.37 | Alta |
| 10 % | 0.895 | $2.2\times10^{-7}$ | 0.930 | 0.857 | 0.569 | 3.02 / 2.39 | Alta en forma |

Las tres concentraciones presentan correlaciones altas ($r=0.88$, $0.91$ y $0.90$; $\rho=0.86$, $0.87$ y $0.93$) con $p<10^{-6}$: la forma de las curvas de razón, incluida la posición del máximo y su desplazamiento con la concentración, se reproduce en todos los casos. Las diferencias de amplitud, cuantificadas por RMSE y MAE, crecen con la concentración y se concentran en la región del máximo, donde la razón simulada es un 11–14 % mayor que la del artículo y está desplazada 0–50 fotones hacia valores mayores. Este exceso es coherente con la sobreestimación de la energía de las capturas en Cl descrita en la Sec. 5, que aumenta la luz por captura en Cl. Con 10 % de NaCl, además, la diferencia en la razón media (3.02 frente a 2.39) proviene principalmente de la curva digitalizada del artículo, cuyo efecto entre 100 y 500 fotones es prácticamente igual al de 5 %; en la presente simulación el efecto crece de forma monótona con la concentración, como lo hacen la probabilidad de captura, el tiempo de captura y la energía gamma de captura (Tabla 4.2). A diferencia de la comparación anterior, la correlación no disminuye al aumentar la concentración.

**Del número de fotones aceptados a la carga en el PMT.**  El eje de la Fig. 6 del trabajo de referencia corresponde a los fotones Cherenkov aceptados por la eficiencia cuántica. Su escala y su forma coinciden con los fotones aceptados en el momento de su creación, antes del transporte óptico, y no con la carga en el PMT, que es unas 16 veces menor; esto indica que la figura no incluye la eficiencia de colección óptica, aunque el artículo no lo especifica. La magnitud que mide el detector es, en cambio, la carga en el PMT, que incluye el transporte de cada fotón por el agua y sus reflexiones en el Tyvek hasta alcanzar el fotocátodo. La Tabla (10.3) resume el balance de luz en agua pura: de los 1 441 fotones Cherenkov creados por neutrón, 155.6 (10.8 %) son aceptados por la eficiencia cuántica y 9.81 (6.3 % de los transportados) alcanzan el PMT. La pérdida dominante es la absorción en la pared, y puede describirse con un modelo de rebotes ajustado a las trayectorias simuladas. En cada encuentro con la pared el fotón alcanza el PMT con probabilidad $s$; si no, choca con el Tyvek, donde se absorbe con probabilidad $a$; y entre dos encuentros puede absorberse en el agua con probabilidad $w$. La fracción de fotones que llega al PMT es entonces

$$ P_{\mathrm{PMT}} = \frac{s}{s+(1-s)\,a+w}. \qquad (10.1) $$

En agua pura se miden $s=0.8$ %, $a=9.8$ % y $w=2.0$ %, con los que la Ec. (10.1) da $P_{\mathrm{PMT}}=6.3$ %, igual al valor observado. Un fotón sobrevive en promedio $1/a\approx10$ choques con el Tyvek (7.1 rebotes medios, porque también se pierde en el agua), mientras que para encontrar el PMT por azar necesitaría del orden de $1/s\approx130$ encuentros con la pared (Figura 10.2).

La absorción por choque, $a=9.8$ %, tiene tres componentes. La reflectividad medida del Tyvek, tabulada en el código de MEIGA según la energía del fotón (0.99–1.00 en el azul y 0.78–0.85 en el ultravioleta), aporta $1-R_{\mathrm{tabla}}\approx2.1$ puntos. En el código, sin embargo, esa tabla se multiplica por un factor de escala `scaleLinerRef` = 0.94 (`Materials.cc`), de modo que la reflectividad empleada es $R=0.94\,R_{\mathrm{tabla}}$; este factor añade $\approx5.9$ puntos, porque impone una pérdida mínima del 6 % en cada choque aun donde el Tyvek tabulado refleja el 100 %. Los $\approx1.9$ puntos restantes son una pérdida adicional del modelo de superficie `groundbackpainted` de Geant4. El factor 0.94 es, por tanto, el principal responsable de la absorción en la pared. Es un parámetro del modelo sin una justificación documentada; factores de este tipo se emplean a veces para representar un revestimiento envejecido o contaminado.

La Tabla (10.2) estima, con la Ec. (10.1) aplicada por intervalos de energía del fotón, la carga media que se obtendría con otros modelos ópticos. La escala absoluta de la carga es incierta en un factor de hasta $\approx2.5$. Esta incertidumbre no afecta a la comparación con la Fig. 6 del artículo, que usa los fotones aceptados antes del transporte, ni a las diferencias entre medios, porque el transporte óptico es el mismo en los cuatro. En esta campaña se conservó el modelo del Tyvek para mantener la comparabilidad entre todas las corridas; su revisión queda como trabajo futuro.

La distribución de fotones aceptados es, además, la que mejor reproduce la forma de la Fig. 6 del artículo (correlación en escala logarítmica de 0.966, frente a 0.878 para la carga en el PMT), incluido el quiebre a 400–450 fotones. Ni siquiera un Tyvek de reflectividad 0.985 con una longitud de absorción de 100 m acercaría la carga a la escala del artículo: la aumentaría como mucho unas 2.5 veces, lejos del factor $\approx$ 16 necesario.

**Tabla 10.2:** Carga media en el PMT estimada para distintos modelos ópticos en agua pura, con el modelo de rebotes de la Ec. (10.1) aplicado por intervalos de energía del fotón. Se conserva la pérdida adicional del modelo `groundbackpainted`.

| **Modelo óptico** | **Carga media (p.e.)** | **Factor** |
|:--|:--|:--|
| Actual: $R=0.94\,R_{\mathrm{tabla}}$ (simulado) | 9.8 | 1.0 |
| Tabla sin el factor 0.94: $R=R_{\mathrm{tabla}}$ | 20.9 | 2.1 |
| $R=0.985$ (LHAASO) | 19.8 | 2.0 |
| Actual + longitud de absorción del agua de 100 m | 11.0 | 1.1 |
| $R=0.985$ + agua de 100 m | 25.2 | 2.6 |

**Tabla 10.3:** Producción de luz Cherenkov, balance de luz y respuesta del PMT por neutrón incidente de 500 MeV.

|  | **Agua pura** | **2.5 % NaCl** | **5 % NaCl** | **10 % NaCl** |
|:--|:--|:--|:--|:--|
| Fotones Cherenkov creados | 1 441 | 1 662 | 1 806 | 1 994 |
| Aceptados por la eficiencia cuántica (transportados) | 155.6 | 179.3 | 194.8 | 214.9 |
| Fotones que alcanzan el PMT (carga media, p.e.) | 9.78 | 11.38 | 12.42 | 13.89 |
| Fotoelectrones por fotón creado | 0.68 % | 0.68 % | 0.69 % | 0.70 % |
| Eventos con $Q\geq1$ p.e. | 53.0 % | 54.9 % | 56.0 % | 58.1 % |
| Carga en $t<1$ µs | 85.7 % | 75.6 % | 71.7 % | 67.4 % |
| Carga en 1 µs – 1 ms (capturas) | 12.7 % | 22.9 % | 26.9 % | 31.1 % |
| Carga en $t>1$ ms (desintegraciones) | 1.55 % | 1.49 % | 1.45 % | 1.50 % |

![Figura 10.2](figuras/fig07_destino_fotones.png)

**Figura 10.2:** Destino de los fotones Cherenkov transportados en agua pura, agua + 2.5 % de NaCl, agua + 5 % de NaCl y agua + 10 % de NaCl: (a) destino final; (b) número de reflexiones en el Tyvek antes del destino; (c) fracción que llega al PMT según la energía del fotón; (d) fracción absorbida en la pared (línea continua) y en el agua (línea discontinua) según la energía.

**Histograma de carga.**  La Figura (10.3) muestra el histograma de carga en el PMT, aporte de este trabajo respecto al de referencia. El 53 % de los neutrones deja al menos un fotoelectrón; la mediana es 1 p.e. y la media 9.8 p.e. en agua pura, y solo el 2.5 % de los eventos supera 100 p.e. La carga media aumenta un 16.4 % (2.5 %), un 27.0 % (5 %) y un 42.0 % (10 %) respecto al agua pura, y la fracción de la carga asociada a las capturas (1 µs–1 ms) pasa del 12.7 % al 22.9 %, 26.9 % y 31.1 %. El transporte óptico es prácticamente el mismo en todos los medios (Figura 10.2): llega al PMT entre el 6.30 y el 6.50 % de los fotones transportados, se absorbe en la pared el 76.2–76.7 % y en el agua el 16.4–16.8 %, con 7.1–7.3 reflexiones medias y la misma dependencia con la energía del fotón, de modo que el modelo del Tyvek fija una escala común de la carga pero no altera las diferencias entre medios. Su escala absoluta depende del modelo óptico (Tabla 10.2).

![Figura 10.3](figuras/fig09_histograma_tipo_sidelnik.png)

**Figura 10.3:** (a, b) Fotones aceptados por la eficiencia cuántica antes del transporte óptico, frente a la Fig. 6 de Sidelnik et al. (2020) (área igualada y normalizado, en escala logarítmica). (c) Histograma de carga en el PMT, con transporte óptico. (d) Efecto del transporte óptico: fotones aceptados frente a fotoelectrones. Eventos con al menos un fotón o fotoelectrón; $1.5\times10^{5}$ neutrones de 500 MeV por medio.

### 11. Síntesis de la validación

La Tabla (11.1) resume el resultado de la validación para cada eslabón de la cadena de procesos, según los tres niveles de comparación definidos en la introducción (Sec. 1).

**Tabla 11.1:** Síntesis de la validación del modelo Monte Carlo frente a Sidelnik et al. (2020). Nivel: (i) valor absoluto; (ii) cociente o forma normalizada; (iii) tendencia con la concentración de NaCl.

| **Observable** | **Nivel** | **Resultado** | **Concordancia** |
|:--|:--|:--|:--|
| Moderación (elásticas) | (i) | $\xi_{\mathrm{H}}=1.000$, $\xi_{\mathrm{O}}=0.118$ (teoría 0.121); la sal no modifica la moderación | Completa |
| Probabilidad de captura | (i) | 24.08 % frente a 25.18 % en agua pura | Buena |
|  | (iii) | Mismo signo y monótono; aumento con NaCl $\approx$ 2 veces menor (+8.3 % frente a +16.2 % con 10 %) | Parcial |
| Tiempo de captura | (i) | 192 / 140 / 109 / 75 µs frente a 205 / 147 / 114 / 78 µs (teoría) | Buena |
| Distancia de captura | (iii) | Disminuye con NaCl si se mide desde el vértice del neutrón | Cualitativa |
| Líneas gamma de captura | (i) | Energías e intensidades por captura; gamma único de $S_n$ en $\approx$ 24 % de las capturas | Buena, con limitación |
| Espectro gamma | (ii) | 6.13/2.223 = 0.022 en ambos; líneas del Cl dentro del 3–10 %; exceso en 0.511 y 8.58 MeV | Buena, con diferencias explicadas |
| Espectro de electrones | (i) | Escala absoluta (cociente 1.06) y bordes Compton | Muy buena |
| Deuterones de captura | (i) | Mediana 1.319 keV (cinemática: 1.319 keV) | Completa |
| Fotones aceptados (Fig. 6) | (i) | Coincidencia sin reescalar entre 0 y 500 fotones; cola de piones mayor | Buena hasta 500 fotones |
| Efecto del NaCl en la luz | (iii) | Curvas de razón: $r=0.88$–$0.91$ en los tres medios con NaCl; razón media 100–500 dentro del 1 % con 2.5 y 5 % | Alta |

En conjunto, el modelo reproduce la física de moderación y captura del detector de referencia, el campo gamma de captura y su conversión en electrones, la producción de luz Cherenkov y, sobre todo, el efecto relativo del NaCl en la señal. Las diferencias identificadas tienen un origen físico concreto: (i) la mayor producción de piones del modelo de Bertini a 500 MeV, que forma la cola de la distribución de fotones por encima de 500 y que no afecta a la región de captura; (ii) una mayor producción de gammas de desexcitación nuclear tras la dispersión inelástica de neutrones, que eleva el fondo y diluye parcialmente el efecto del Cl; y (iii) el modelo de captura de Geant4, que no conserva la energía en las cascadas del Cl y emite un gamma único con toda la energía de separación en una cuarta parte de las capturas. Esta última limitación sobreestima la luz de las capturas en Cl y debe tenerse en cuenta en la interpretación de la respuesta absoluta del detector a neutrones en los medios con NaCl.

### Apéndice A. Ecuaciones, energías y porcentajes de los procesos de los neutrones de 500 MeV

Las Tablas A.1–A.4 reúnen, para cada medio simulado en la Sec. 1, todas las reacciones de los neutrones en el volumen del detector (primario y secundarios): la ecuación de la reacción, el valor $Q$ del estado fundamental (calculado con los excesos de masa de la evaluación AME2020; $Q<0$ es la energía umbral que se gasta en romper el núcleo), la energía típica del neutrón incidente (mediana y, entre paréntesis, la fracción de reacciones por debajo de 20 MeV), el porcentaje dentro de cada tipo de proceso y los gammas que emite. El núcleo blanco de cada interacción inelástica se deduce por conservación de la carga y del número bariónico de los productos. En las inelásticas, las líneas $\gamma$ solo se atribuyen si pertenecen al núcleo residual o al blanco de esa misma reacción (ventana de $\pm10$ keV); se listan los canales con más del 0.4 % de las inelásticas y, en los medios con NaCl, también los del Na y el Cl.

Estas tablas explican el origen de los gammas y, por tanto, de los electrones y de la luz Cherenkov: la dispersión elástica en $^{1}$H modera los neutrones ($\xi=1$); la captura en $^{1}$H y en $^{35}$Cl produce los gammas de captura; y las interacciones inelásticas en $^{16}$O producen el fondo de desexcitación nuclear (líneas de 4.44, 6.13, 6.92 y 7.12 MeV y un continuo) y los núcleos emisores $\beta^{+}$ ($^{15}$O, $^{13}$N, $^{11}$C) responsables de la línea de aniquilación de larga vida.

**Tabla A.1:** Agua pura: ecuaciones, energías y porcentajes de los procesos de los neutrones (150 000 neutrones de 500 MeV).

| Reacción | $Q$ (MeV) | $E_n$ típica | % | Gammas emitidos (keV) |
|:--|:--|:--|:--|:--|
| **Dispersión elástica** (104 por neutrón incidente; % de las colisiones con blanco reconstruido, $E>1$ eV) |  |  |  |  |
| $^1$H(n,n)$^1$H | 0 | todas | 83.74 % | pierde 0–100 % de $E$; $\xi = 1.00$ |
| $^{16}$O(n,n)$^{16}$O | 0 | todas | 15.95 % | pierde 0–22 %; $\xi = 0.12$ ($<$ 1 MeV) |
| **Captura radiativa** (0.829 por neutrón incidente; % de las capturas en el agua) |  |  |  |  |
| $^1$H(n,$\gamma$)$^2$H | +2.225 | térmica | 99.94 % | 2224 (MEIGA: 2224.4) |
| $^{16}$O(n,$\gamma$)$^{17}$O | +4.143 | térmica | 0.06 % | 871, 1088, 2184; $\gamma$ único 4143 |
| **Inelástica** (1.29 por neutrón incidente; % de las inelásticas; canales con más del 0.4 %) |  |  |  |  |
| $^{16}$O(n,np)$^{15}$N | -12.13 | 354 MeV | 12.9 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n')$^{16}$O | 0.00 | 21 MeV (48 % $<$ 20) | 11.3 % | 2.8 $\gamma$/reacc.; 6130 (14 %), 7117 (6 %), 2741 (4 %) |
| $^{16}$O(n,n$\alpha$)$^{12}$C | -7.16 | 39 MeV (16 % $<$ 20) | 9.5 % | 1.7 $\gamma$/reacc.; 4439 (6 %) |
| $^{16}$O(n,2n)$^{15}$O | -15.66 | 500 MeV | 9.4 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2np)$^{14}$N | -22.96 | 377 MeV | 9.1 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p)$^{12}$C | -35.46 | 500 MeV | 6.7 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n2p)$^{13}$C | -30.51 | 477 MeV | 4.0 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,$\alpha$)$^{13}$C | -2.22 | 10 MeV (100 % $<$ 20) | 3.1 % | 2.4 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p)$^{14}$C | -22.33 | 384 MeV | 2.9 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p$\alpha$)$^{8}$Be | -42.82 | 500 MeV | 2.1 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd)$^{12}$C | -33.23 | 380 MeV | 1.9 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p3$\alpha$) | -42.73 | 500 MeV | 1.4 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3np)$^{13}$N | -33.51 | 500 MeV | 1.1 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2$\alpha$)$^{8}$Be | -14.53 | 68 MeV | 1.0 % | 1.5 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n3p)$^{11}$B | -51.41 | 500 MeV | 0.8 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^-$)$^{15}$O | -154.45 | 500 MeV | 0.7 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,p)$^{16}$N | -9.64 | 19 MeV (54 % $<$ 20) | 0.7 % | 2.0 $\gamma$/reacc.; continuo |
| $^{1}$H(n,np$\pi^0$) | -134.98 | 500 MeV | 0.7 % | 0.0 $\gamma$/reacc.; sin $\gamma$ directos |
| $^{16}$O(n,2npd$\alpha$)$^{8}$Be | -40.60 | 500 MeV | 0.7 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n)$^{14}$O | -28.89 | 500 MeV | 0.7 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^0$)$^{15}$N | -147.10 | 500 MeV | 0.6 % | 2.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3pd2$\alpha$) | -68.80 | 500 MeV | 0.5 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3p)$^{10}$B | -62.87 | 500 MeV | 0.5 % | 1.8 $\gamma$/reacc.; continuo |
| $^{16}$O(n,5n4p2$\alpha$) | -71.03 | 500 MeV | 0.5 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n3p$\pi^-$)$^{12}$C | -174.25 | 500 MeV | 0.5 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n2p)$^{11}$C | -54.18 | 500 MeV | 0.5 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p$\pi^-$)$^{14}$N | -161.75 | 500 MeV | 0.4 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd3$\alpha$) | -40.51 | 500 MeV | 0.4 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p$\pi^0$)$^{12}$C | -170.43 | 500 MeV | 0.4 % | 2.2 $\gamma$/reacc.; continuo |

**Tabla A.2:** Agua + 2.5 % NaCl: ecuaciones, energías y porcentajes de los procesos de los neutrones (150 000 neutrones de 500 MeV).

| Reacción | $Q$ (MeV) | $E_n$ típica | % | Gammas emitidos (keV) |
|:--|:--|:--|:--|:--|
| **Dispersión elástica** (83 por neutrón incidente; % de las colisiones con blanco reconstruido, $E>1$ eV) |  |  |  |  |
| $^1$H(n,n)$^1$H | 0 | todas | 83.55 % | pierde 0–100 % de $E$; $\xi = 1.00$ |
| $^{16}$O(n,n)$^{16}$O | 0 | todas | 15.82 % | pierde 0–22 %; $\xi = 0.12$ ($<$ 1 MeV) |
| $^{23}$Na(n,n)$^{23}$Na | 0 | todas | 0.47 % | pierde 0–16 % |
| $^{35.37}$Cl(n,n)$^{35.37}$Cl | 0 | todas | 0.16 % | pierde 0–11 % |
| **Captura radiativa** (0.864 por neutrón incidente; % de las capturas en el agua) |  |  |  |  |
| $^1$H(n,$\gamma$)$^2$H | +2.225 | térmica | 63.53 % | 2224 (MEIGA: 2224.4) |
| $^{35}$Cl(n,$\gamma$)$^{36}$Cl | +8.580 | térmica | 35.75 % | 517, 788, 1165, 1951, 6111, 7414; $\gamma$ único 8580 |
| $^{23}$Na(n,$\gamma$)$^{24}$Na | +6.959 | térmica | 0.57 % | 472 (0.9 %); $\gamma$ único 6959 |
| $^{37}$Cl(n,$\gamma$)$^{38}$Cl | +6.108 | térmica | 0.11 % | 638, 755, 1692; $\gamma$ único 6108 |
| $^{16}$O(n,$\gamma$)$^{17}$O | +4.143 | térmica | 0.03 % | 871, 1088, 2184; $\gamma$ único 4143 |
| **Inelástica** (1.33 por neutrón incidente; % de las inelásticas; canales con más del 0.4 % y los del Na y el Cl) |  |  |  |  |
| $^{16}$O(n,np)$^{15}$N | -12.13 | 350 MeV | 12.6 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n')$^{16}$O | 0.00 | 21 MeV (49 % $<$ 20) | 11.0 % | 2.8 $\gamma$/reacc.; 6130 (14 %), 7117 (6 %), 6917 (4 %) |
| $^{16}$O(n,n$\alpha$)$^{12}$C | -7.16 | 39 MeV (16 % $<$ 20) | 9.3 % | 1.7 $\gamma$/reacc.; 4439 (6 %) |
| $^{16}$O(n,2n)$^{15}$O | -15.66 | 500 MeV | 9.0 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2np)$^{14}$N | -22.96 | 373 MeV | 8.8 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p)$^{12}$C | -35.46 | 500 MeV | 6.4 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n2p)$^{13}$C | -30.51 | 471 MeV | 4.0 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,$\alpha$)$^{13}$C | -2.22 | 10 MeV (100 % $<$ 20) | 3.1 % | 2.4 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p)$^{14}$C | -22.33 | 406 MeV | 2.8 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p$\alpha$)$^{8}$Be | -42.82 | 500 MeV | 2.0 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd)$^{12}$C | -33.23 | 399 MeV | 1.8 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p3$\alpha$) | -42.73 | 500 MeV | 1.5 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3np)$^{13}$N | -33.51 | 500 MeV | 1.1 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2$\alpha$)$^{8}$Be | -14.53 | 70 MeV | 1.0 % | 1.5 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n3p)$^{11}$B | -51.41 | 500 MeV | 0.8 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^-$)$^{15}$O | -154.45 | 500 MeV | 0.7 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,p)$^{16}$N | -9.64 | 19 MeV (54 % $<$ 20) | 0.7 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n)$^{14}$O | -28.89 | 500 MeV | 0.6 % | 2.2 $\gamma$/reacc.; continuo |
| $^{1}$H(n,np$\pi^0$) | -134.98 | 500 MeV | 0.6 % | 0.0 $\gamma$/reacc.; sin $\gamma$ directos |
| $^{16}$O(n,2npd$\alpha$)$^{8}$Be | -40.60 | 472 MeV | 0.6 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^0$)$^{15}$N | -147.10 | 500 MeV | 0.6 % | 2.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3pd2$\alpha$) | -68.80 | 500 MeV | 0.5 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3p)$^{10}$B | -62.87 | 500 MeV | 0.5 % | 1.8 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n3p$\pi^-$)$^{12}$C | -174.25 | 500 MeV | 0.5 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,5n4p2$\alpha$) | -71.03 | 500 MeV | 0.5 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n2p)$^{11}$C | -54.18 | 500 MeV | 0.5 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p$\pi^-$)$^{14}$N | -161.75 | 500 MeV | 0.4 % | 2.0 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,n')$^{23}$Na | 0.00 | 7 MeV (85 % $<$ 20) | 0.3 % | 2.6 $\gamma$/reacc.; 440 (74 %) |
| $^{35}$Cl(n,p)$^{35}$S | 0.62 | térmica (96 % $<$ 20) | 0.3 % | 0.4 $\gamma$/reacc.; continuo |

**Tabla A.3:** Agua + 5 % NaCl: ecuaciones, energías y porcentajes de los procesos de los neutrones (150 000 neutrones de 500 MeV).

| Reacción | $Q$ (MeV) | $E_n$ típica | % | Gammas emitidos (keV) |
|:--|:--|:--|:--|:--|
| **Dispersión elástica** (72 por neutrón incidente; % de las colisiones con blanco reconstruido, $E>1$ eV) |  |  |  |  |
| $^1$H(n,n)$^1$H | 0 | todas | 83.33 % | pierde 0–100 % de $E$; $\xi = 1.00$ |
| $^{16}$O(n,n)$^{16}$O | 0 | todas | 15.71 % | pierde 0–22 %; $\xi = 0.12$ ($<$ 1 MeV) |
| $^{23}$Na(n,n)$^{23}$Na | 0 | todas | 0.68 % | pierde 0–16 % |
| $^{35.37}$Cl(n,n)$^{35.37}$Cl | 0 | todas | 0.28 % | pierde 0–11 % |
| **Captura radiativa** (0.903 por neutrón incidente; % de las capturas en el agua) |  |  |  |  |
| $^{35}$Cl(n,$\gamma$)$^{36}$Cl | +8.580 | térmica | 51.47 % | 517, 788, 1165, 1951, 6111, 7414; $\gamma$ único 8580 |
| $^1$H(n,$\gamma$)$^2$H | +2.225 | térmica | 47.53 % | 2224 (MEIGA: 2224.4) |
| $^{23}$Na(n,$\gamma$)$^{24}$Na | +6.959 | térmica | 0.80 % | 472 (0.9 %); $\gamma$ único 6959 |
| $^{37}$Cl(n,$\gamma$)$^{38}$Cl | +6.108 | térmica | 0.17 % | 638, 755, 1692; $\gamma$ único 6108 |
| $^{16}$O(n,$\gamma$)$^{17}$O | +4.143 | térmica | 0.03 % | 871, 1088, 2184; $\gamma$ único 4143 |
| **Inelástica** (1.36 por neutrón incidente; % de las inelásticas; canales con más del 0.4 % y los del Na y el Cl) |  |  |  |  |
| $^{16}$O(n,np)$^{15}$N | -12.13 | 326 MeV | 12.1 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n')$^{16}$O | 0.00 | 20 MeV (50 % $<$ 20) | 10.8 % | 2.8 $\gamma$/reacc.; 6130 (14 %), 7117 (6 %), 6917 (4 %) |
| $^{16}$O(n,n$\alpha$)$^{12}$C | -7.16 | 38 MeV (16 % $<$ 20) | 9.2 % | 1.7 $\gamma$/reacc.; 4439 (6 %) |
| $^{16}$O(n,2n)$^{15}$O | -15.66 | 500 MeV | 8.7 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2np)$^{14}$N | -22.96 | 352 MeV | 8.7 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p)$^{12}$C | -35.46 | 500 MeV | 6.2 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n2p)$^{13}$C | -30.51 | 465 MeV | 3.8 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,$\alpha$)$^{13}$C | -2.22 | 10 MeV (100 % $<$ 20) | 3.0 % | 2.4 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p)$^{14}$C | -22.33 | 354 MeV | 2.7 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p$\alpha$)$^{8}$Be | -42.82 | 500 MeV | 2.0 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd)$^{12}$C | -33.23 | 362 MeV | 1.8 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p3$\alpha$) | -42.73 | 500 MeV | 1.4 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3np)$^{13}$N | -33.51 | 500 MeV | 1.1 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2$\alpha$)$^{8}$Be | -14.53 | 71 MeV | 1.0 % | 1.5 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n3p)$^{11}$B | -51.41 | 500 MeV | 0.7 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^-$)$^{15}$O | -154.45 | 500 MeV | 0.6 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd$\alpha$)$^{8}$Be | -40.60 | 487 MeV | 0.6 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,p)$^{16}$N | -9.64 | 18 MeV (55 % $<$ 20) | 0.6 % | 1.9 $\gamma$/reacc.; continuo |
| $^{1}$H(n,np$\pi^0$) | -134.98 | 500 MeV | 0.6 % | 0.0 $\gamma$/reacc.; sin $\gamma$ directos |
| $^{16}$O(n,3n)$^{14}$O | -28.89 | 500 MeV | 0.6 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^0$)$^{15}$N | -147.10 | 500 MeV | 0.5 % | 2.6 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,n')$^{23}$Na | 0.00 | 7 MeV (84 % $<$ 20) | 0.5 % | 2.5 $\gamma$/reacc.; 440 (72 %), 2076 (2 %) |
| $^{16}$O(n,4n3pd2$\alpha$) | -68.80 | 500 MeV | 0.5 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n2p)$^{11}$C | -54.18 | 500 MeV | 0.5 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3p)$^{10}$B | -62.87 | 500 MeV | 0.5 % | 1.8 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,p)$^{35}$S | 0.62 | térmica (96 % $<$ 20) | 0.5 % | 0.5 $\gamma$/reacc.; continuo |
| $^{16}$O(n,5n4p2$\alpha$) | -71.03 | 500 MeV | 0.4 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n3p$\pi^-$)$^{12}$C | -174.25 | 500 MeV | 0.4 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p$\pi^-$)$^{14}$N | -161.75 | 500 MeV | 0.4 % | 2.1 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,np)$^{22}$Ne | -8.79 | 155 MeV | 0.3 % | 2.2 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,np)$^{34}$S | -6.37 | 122 MeV | 0.3 % | 2.5 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,n')$^{35}$Cl | 0.00 | 7 MeV (77 % $<$ 20) | 0.3 % | 2.4 $\gamma$/reacc.; 1219 (18 %), 1763 (10 %) |
| $^{23}$Na(n,2np)$^{21}$Ne | -19.16 | 211 MeV | 0.2 % | 2.0 $\gamma$/reacc.; 440 (3 %) |
| $^{35}$Cl(n,2np)$^{33}$S | -17.79 | 247 MeV | 0.2 % | 2.1 $\gamma$/reacc.; continuo |

**Tabla A.4:** Agua + 10 % NaCl: ecuaciones, energías y porcentajes de los procesos de los neutrones (150 000 neutrones de 500 MeV).

| Reacción | $Q$ (MeV) | $E_n$ típica | % | Gammas emitidos (keV) |
|:--|:--|:--|:--|:--|
| **Dispersión elástica** (59 por neutrón incidente; % de las colisiones con blanco reconstruido, $E>1$ eV) |  |  |  |  |
| $^1$H(n,n)$^1$H | 0 | todas | 82.90 % | pierde 0–100 % de $E$; $\xi = 1.00$ |
| $^{16}$O(n,n)$^{16}$O | 0 | todas | 15.42 % | pierde 0–22 %; $\xi = 0.12$ ($<$ 1 MeV) |
| $^{23}$Na(n,n)$^{23}$Na | 0 | todas | 1.12 % | pierde 0–16 % |
| $^{35.37}$Cl(n,n)$^{35.37}$Cl | 0 | todas | 0.55 % | pierde 0–11 % |
| **Captura radiativa** (0.972 por neutrón incidente; % de las capturas en el agua) |  |  |  |  |
| $^{35}$Cl(n,$\gamma$)$^{36}$Cl | +8.580 | térmica | 67.66 % | 517, 788, 1165, 1951, 6111, 7414; $\gamma$ único 8580 |
| $^1$H(n,$\gamma$)$^2$H | +2.225 | térmica | 30.95 % | 2224 (MEIGA: 2224.4) |
| $^{23}$Na(n,$\gamma$)$^{24}$Na | +6.959 | térmica | 1.13 % | 472 (0.9 %); $\gamma$ único 6959 |
| $^{37}$Cl(n,$\gamma$)$^{38}$Cl | +6.108 | térmica | 0.23 % | 638, 755, 1692; $\gamma$ único 6108 |
| $^{16}$O(n,$\gamma$)$^{17}$O | +4.143 | térmica | 0.02 % | 871, 1088, 2184; $\gamma$ único 4143 |
| **Inelástica** (1.43 por neutrón incidente; % de las inelásticas; canales con más del 0.4 % y los del Na y el Cl) |  |  |  |  |
| $^{16}$O(n,np)$^{15}$N | -12.13 | 307 MeV | 11.6 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n')$^{16}$O | 0.00 | 20 MeV (50 % $<$ 20) | 10.5 % | 2.7 $\gamma$/reacc.; 6130 (15 %), 7117 (6 %), 2741 (4 %) |
| $^{16}$O(n,n$\alpha$)$^{12}$C | -7.16 | 37 MeV (17 % $<$ 20) | 8.7 % | 1.7 $\gamma$/reacc.; 4439 (6 %) |
| $^{16}$O(n,2np)$^{14}$N | -22.96 | 343 MeV | 8.2 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n)$^{15}$O | -15.66 | 500 MeV | 8.2 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p)$^{12}$C | -35.46 | 500 MeV | 5.8 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n2p)$^{13}$C | -30.51 | 460 MeV | 3.5 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,$\alpha$)$^{13}$C | -2.22 | 10 MeV (99 % $<$ 20) | 2.9 % | 2.4 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2p)$^{14}$C | -22.33 | 326 MeV | 2.5 % | 2.1 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p$\alpha$)$^{8}$Be | -42.82 | 500 MeV | 1.8 % | 1.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2npd)$^{12}$C | -33.23 | 377 MeV | 1.7 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n2p3$\alpha$) | -42.73 | 500 MeV | 1.3 % | 1.7 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,n')$^{23}$Na | 0.00 | 6 MeV (85 % $<$ 20) | 1.2 % | 2.6 $\gamma$/reacc.; 440 (73 %) |
| $^{16}$O(n,3np)$^{13}$N | -33.51 | 500 MeV | 1.0 % | 1.7 $\gamma$/reacc.; continuo |
| $^{16}$O(n,n2$\alpha$)$^{8}$Be | -14.53 | 65 MeV | 0.9 % | 1.5 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,p)$^{35}$S | 0.62 | térmica (96 % $<$ 20) | 0.7 % | 0.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,3n3p)$^{11}$B | -51.41 | 500 MeV | 0.7 % | 2.4 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,np)$^{22}$Ne | -8.79 | 129 MeV | 0.6 % | 2.2 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,np)$^{34}$S | -6.37 | 123 MeV | 0.6 % | 2.4 $\gamma$/reacc.; continuo |
| $^{16}$O(n,p)$^{16}$N | -9.64 | 18 MeV (56 % $<$ 20) | 0.6 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^-$)$^{15}$O | -154.45 | 500 MeV | 0.6 % | 2.3 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,n')$^{35}$Cl | 0.00 | 7 MeV (79 % $<$ 20) | 0.6 % | 2.4 $\gamma$/reacc.; 1219 (16 %), 1763 (11 %) |
| $^{16}$O(n,2npd$\alpha$)$^{8}$Be | -40.60 | 499 MeV | 0.6 % | 1.6 $\gamma$/reacc.; continuo |
| $^{1}$H(n,np$\pi^0$) | -134.98 | 500 MeV | 0.6 % | 0.0 $\gamma$/reacc.; sin $\gamma$ directos |
| $^{16}$O(n,3n)$^{14}$O | -28.89 | 500 MeV | 0.6 % | 2.2 $\gamma$/reacc.; continuo |
| $^{16}$O(n,np$\pi^0$)$^{15}$N | -147.10 | 500 MeV | 0.5 % | 2.6 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3pd2$\alpha$) | -68.80 | 500 MeV | 0.5 % | 1.9 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n3p)$^{10}$B | -62.87 | 500 MeV | 0.4 % | 1.8 $\gamma$/reacc.; continuo |
| $^{16}$O(n,4n2p)$^{11}$C | -54.18 | 500 MeV | 0.4 % | 2.3 $\gamma$/reacc.; continuo |
| $^{16}$O(n,5n4p2$\alpha$) | -71.03 | 500 MeV | 0.4 % | 1.9 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,2np)$^{21}$Ne | -19.16 | 238 MeV | 0.4 % | 2.0 $\gamma$/reacc.; continuo |
| $^{16}$O(n,2n3p$\pi^-$)$^{12}$C | -174.25 | 500 MeV | 0.4 % | 2.1 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,2np)$^{33}$S | -17.79 | 157 MeV | 0.4 % | 2.2 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,2n)$^{22}$Na | -12.42 | 494 MeV | 0.4 % | 2.1 $\gamma$/reacc.; 440 (3 %) |
| $^{23}$Na(n,3np)$^{20}$Ne | -25.92 | 349 MeV | 0.3 % | 2.1 $\gamma$/reacc.; 440 (3 %) |
| $^{35}$Cl(n,3np)$^{32}$S | -26.43 | 365 MeV | 0.3 % | 2.3 $\gamma$/reacc.; continuo |
| $^{35}$Cl(n,2n)$^{34}$Cl | -12.64 | 500 MeV | 0.3 % | 2.1 $\gamma$/reacc.; continuo |
| $^{37}$Cl(n,n')$^{37}$Cl | 0.00 | 8 MeV (82 % $<$ 20) | 0.2 % | 2.6 $\gamma$/reacc.; continuo |
| $^{23}$Na(n,3np$\alpha$)$^{16}$O | -30.65 | 455 MeV | 0.2 % | 2.1 $\gamma$/reacc.; 440 (3 %) |